# Black-Litterman research code: public edition

This is a source-code portfolio extract of a completed course research project. The original submission narrative, execution outputs, datasets and audit snapshots are not distributed here. The short navigation notes below were written for this public edition.

Read `../README.md` before execution. The full analysis requires authorized local snapshots and their original audit files; it is **not a one-command reproduction package**. Network retrieval is disabled in the public execution path. Run `python demo/smoke_demo.py` from the project directory for the separate synthetic-data check.

The historical walk-forward experiment is described as **pseudo-OOS**: the asset universe and scenario choices were formed retrospectively. It did not establish positive incremental ESG alpha. The demo verifies numerical mechanics only and is not a course result or investment performance claim.


## Environment and local input preflight


In [ ]:
# ===================== 0. Environment initialisation =====================
import numpy as np
import pandas as pd
import yfinance as yf
try:
    import cvxpy as cp
except ImportError:
    cp = None
    print("cvxpy is not available; scipy SLSQP fallback will be used for optimisation.")
from scipy.optimize import minimize
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.covariance import LedoitWolf
from IPython.display import display
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import sys
import time
import warnings
warnings.filterwarnings("default")

def format_display_value(value, style="ratio", na_label="N/A"):
    """
    Format one scalar for notebook display without changing the numeric value exported to CSV.
    Exact zero, rounded non-zero values, structural missingness and infinite estimates receive distinct labels.
    """
    if value is None or pd.isna(value):
        return na_label
    if isinstance(value, (int, float, np.integer, np.floating)):
        numeric = float(value)
        if style == "pvalue":
            if not np.isfinite(numeric) or not 0.0 <= numeric <= 1.0:
                raise ValueError(f"p-value must be finite and inside [0, 1]: {numeric}")
        if np.isposinf(numeric):
            return "Infinity"
        if np.isneginf(numeric):
            return "-Infinity"
        if style == "pvalue":
            return "<0.0001" if numeric < 0.0001 else f"{numeric:.4f}"
        if style in {"percent", "percent4"}:
            decimals = 4 if style == "percent4" else 2
            if numeric == 0.0:
                return f"{0.0:.{decimals}%} (exact)"
            display_floor = 0.5 * 10.0 ** (-decimals) / 100.0
            if abs(numeric) < display_floor:
                threshold = f"{10.0 ** (-decimals) / 100.0:.{decimals}%}"
                return f"(0, +{threshold})" if numeric > 0 else f"(-{threshold}, 0)"
            return f"{numeric:.{decimals}%}"
        if style == "basis_points":
            basis_points = numeric * 10000.0
            if basis_points == 0.0:
                return "0.00 bp (exact)"
            if abs(basis_points) < 0.005:
                return "(0, +0.01 bp)" if basis_points > 0 else "(-0.01 bp, 0)"
            return f"{basis_points:+.2f} bp"
        if style == "scientific":
            return "0 (exact)" if numeric == 0.0 else f"{numeric:.3e}"
        if style == "integer":
            rounded = round(numeric)
            if not np.isclose(numeric, rounded, atol=1e-9, rtol=0.0):
                raise ValueError(f"Non-integer value passed to integer formatter: {numeric}")
            return f"{int(rounded)}"
        if style == "currency_billions":
            return "$" + f"{numeric / 1e9:,.2f}bn" if numeric != 0.0 else "$0.00bn (exact)"
        decimals = 4 if style == "decimal4" else 3
        if numeric == 0.0:
            return f"{0.0:.{decimals}f} (exact)"
        display_floor = 0.5 * 10.0 ** (-decimals)
        if abs(numeric) < display_floor:
            threshold = f"{10.0 ** (-decimals):.{decimals}f}"
            return f"(0, +{threshold})" if numeric > 0 else f"(-{threshold}, 0)"
        return f"{numeric:.{decimals}f}"
    return value


def format_numeric_frame(frame, style="decimal4", na_label="N/A"):
    """
    Apply the scalar display convention to every value in a table while preserving the source table for export.
    """
    formatted = frame.copy().astype(object)
    for column in formatted.columns:
        formatted[column] = frame[column].map(
            lambda value: format_display_value(value, style=style, na_label=na_label)
        )
    return formatted


def find_project_root(start: Path = None) -> Path:
    """Locate this public project from its directory, notebook directory, or repository root."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        for project in (candidate, candidate / "projects" / "black-litterman"):
            if (project / "notebooks" / "analysis_clean.ipynb").is_file():
                return project
    raise FileNotFoundError("Open this notebook from the black-litterman project or repository directory.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ACTIVE_PYTHON = Path(sys.executable).resolve()
RUN_STARTED_UTC = datetime.now(timezone.utc).replace(microsecond=0)
RUN_ID = RUN_STARTED_UTC.strftime("%Y%m%dT%H%M%SZ") + "-" + os.urandom(3).hex()
RUN_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "runs" / RUN_ID
DATA_OUTPUT_DIR = RUN_OUTPUT_DIR / "data"
FIGURE_OUTPUT_DIR = RUN_OUTPUT_DIR / "figures"
FROZEN_SNAPSHOT_ROOT = PROJECT_ROOT / "inputs" / "snapshots" / "2026-07-24"
FROZEN_MAIN_DATA_DIR = FROZEN_SNAPSHOT_ROOT / "main"
FROZEN_ROBUSTNESS_DATA_DIR = FROZEN_SNAPSHOT_ROOT / "robustness_5y"
LEGACY_CACHE_DATA_DIR = PROJECT_ROOT / "outputs" / "data"
LEGACY_CACHE_ROBUSTNESS_DIR = LEGACY_CACHE_DATA_DIR / "robustness_5y"
# The public repository intentionally excludes the private data snapshot.
required_local_inputs = [
    FROZEN_SNAPSHOT_ROOT / "snapshot_manifest.json",
    FROZEN_SNAPSHOT_ROOT / "snapshot_metadata.json",
    FROZEN_SNAPSHOT_ROOT / "fred_DGS3MO_raw.csv",
    FROZEN_MAIN_DATA_DIR,
    FROZEN_ROBUSTNESS_DATA_DIR,
]
if any(not path.exists() for path in required_local_inputs):
    raise FileNotFoundError(
        "Authorized local snapshots and audit files are required; see the project README. "
        "The public synthetic demo does not require these inputs."
    )
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
CSV_ENCODING = "utf-8-sig"  # Excel/Windows-friendly UTF-8 with BOM

# Global plotting settings
sns.set_style("whitegrid")
plt.rcParams["font.family"] = ["DejaVu Sans"]
plt.rcParams["figure.dpi"] = 120

RUN_STATUS_PATH = RUN_OUTPUT_DIR / "run_status.json"
RUN_STATUS_PATH.write_text(json.dumps({
    "run_id": RUN_ID,
    "status": "running",
    "started_utc": RUN_STARTED_UTC.isoformat(),
}, indent=2), encoding="utf-8")
print(f"Run ID: {RUN_ID}")
print(f"Run output directory: {RUN_OUTPUT_DIR}")
print(f"Python version: {platform.python_version()}")


## Audited local data loading


In [ ]:
# ===================== 2.1 Data loading module  =====================
# Self-contained DataLoader implementation.
import numpy as np
import pandas as pd
import yfinance as yf
from fredapi import Fred
from datetime import datetime, timezone
import csv
import html
import io
import json
from pathlib import Path
import re
import time
import urllib.error
import urllib.parse
import urllib.request
import warnings
import zipfile
import xml.etree.ElementTree as ET
warnings.filterwarnings("default")
CSV_ENCODING = globals().get("CSV_ENCODING", "utf-8-sig")

class DataLoader:
    """
    Self-contained data loading and preprocessing class for the full CQF notebook pipeline.
    It downloads historical ETF prices, builds daily returns and excess returns, retrieves risk-free-rate data, 
        obtains official ETF AUM snapshots, and converts AUM into Black-Litterman market weights.
    Main methods: download_prices() builds price panels; 
        calc_simple_returns() converts prices to returns; 
        get_risk_free_rate() creates the daily risk-free proxy; 
        get_market_aum() retrieves or reads ETF AUM; 
        get_market_weights() normalises AUM into prior weights; 
        save_data_to_csv() writes auditable CSV outputs; 
        load_all_data() orchestrates the full data pipeline.
    Helper methods handle Yahoo/Stooq/FRED downloads, 
        official SSGA/iShares/SPDR Gold Shares AUM parsing, 
        CSV/JSON/Excel decoding, retry logic, and provider-specific fallback paths.
    """
    
    def __init__(
        self,
        start_date: str,
        end_date: str,
        fred_api_key: str = None,
        price_source: str = "yahoo_chart",
        risk_free_source: str = "fred",
        market_aum_source: str = "official",
        market_aum_file: str = None,
        cache_dir: str = None,
        max_retries: int = 3,
        retry_sleep_seconds: int = 10,
    ):
        """
        Initialise the data loader with sample dates, data-source choices and retry settings.
        It defines the investable asset universe, factor proxy tickers and official AUM provider mappings.
        When cache_dir is provided, local CSV files are used only after a live data-source failure.
        """
        self.start_date = start_date
        self.end_date = end_date
        self.fred_api_key = fred_api_key
        self.price_source = price_source.lower()
        self.risk_free_source = risk_free_source.lower()
        self.market_aum_source = market_aum_source.lower()
        self.market_aum_file = Path(market_aum_file) if market_aum_file else None
        self.cache_dir = Path(cache_dir) if cache_dir else None
        self.max_retries = max_retries
        self.retry_sleep_seconds = retry_sleep_seconds
        self.market_aum_snapshot = None
        self.rf_metadata = None
        
        # Universe: 11 US equity-sector ETFs, Treasuries, and gold (13 assets).
        self.asset_tickers = [
            "XLK", "XLF", "XLE", "XLI", "XLP", "XLY", 
            "XLV", "XLB", "XLRE", "XLC", "XLU", "TLT", "GLD"
        ]
        
        # Factor proxy ETFs: ESG and momentum proxies.
        self.factor_tickers = {
            "ESG": "ESGU",    # iShares ESG Aware MSCI USA ETF
            "Momentum": "MTUM",    # iShares MSCI USA Momentum Factor ETF
            "Market": "SPY"   # market benchmark
        }

        self.sector_spdr_tickers = {
            "XLK", "XLF", "XLE", "XLI", "XLP", "XLY",
            "XLV", "XLB", "XLRE", "XLC", "XLU"
        }
        self.sector_spdr_slugs = {
            "XLK": "the-technology-select-sector-spdr-fund-xlk",
            "XLF": "the-financial-select-sector-spdr-fund-xlf",
            "XLE": "the-energy-select-sector-spdr-fund-xle",
            "XLI": "the-industrial-select-sector-spdr-fund-xli",
            "XLP": "the-consumer-staples-select-sector-spdr-fund-xlp",
            "XLY": "the-consumer-discretionary-select-sector-spdr-fund-xly",
            "XLV": "the-health-care-select-sector-spdr-fund-xlv",
            "XLB": "the-materials-select-sector-spdr-fund-xlb",
            "XLRE": "the-real-estate-select-sector-spdr-fund-xlre",
            "XLC": "the-communication-services-select-sector-spdr-fund-xlc",
            "XLU": "the-utilities-select-sector-spdr-fund-xlu",
        }

    def _request_url(self, url: str):
        """
        Build and open a URL request with browser-like headers.
        This reduces avoidable blocking when reading public provider pages or files.
        """
        request = urllib.request.Request(
            url,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/126.0 Safari/537.36"
                ),
                "Accept": "text/csv,text/plain,application/json,*/*",
            },
        )
        return urllib.request.urlopen(request, timeout=30)

    def _read_url_bytes(self, url: str, label: str) -> bytes:
        """
        Download raw bytes from a URL with retry logic and timeout protection.
        It is used for provider webpages, CSV files, JSON files and Excel archives.
        """
        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                with self._request_url(url) as response:
                    content = response.read()
                if not content:
                    raise ValueError(f"{label}: empty response.")
                return content
            except Exception as exc:
                last_error = exc
                if attempt < self.max_retries:
                    wait = self.retry_sleep_seconds * attempt
                    print(f"{label}: download failed on attempt {attempt}/{self.max_retries}: {exc}")
                    print(f"Retrying after {wait} seconds...")
                    time.sleep(wait)

        raise RuntimeError(f"{label}: failed to download after {self.max_retries} attempts.") from last_error

    def _read_csv_url(self, url: str, label: str) -> pd.DataFrame:
        """
        Read a remote CSV file into a pandas DataFrame using the shared retry downloader.
        It keeps CSV parsing consistent across public data sources.
        """
        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                with self._request_url(url) as response:
                    content = response.read()
                data = pd.read_csv(io.BytesIO(content))
                if data.empty:
                    raise ValueError(f"{label}: CSV response is empty.")
                return data
            except urllib.error.HTTPError as exc:
                last_error = exc
                if exc.code == 404:
                    break
                if attempt < self.max_retries:
                    wait = self.retry_sleep_seconds * attempt
                    print(f"{label}: CSV download failed on attempt {attempt}/{self.max_retries}: {exc}")
                    print(f"Retrying after {wait} seconds...")
                    time.sleep(wait)
            except Exception as exc:
                last_error = exc
                if attempt < self.max_retries:
                    wait = self.retry_sleep_seconds * attempt
                    print(f"{label}: CSV download failed on attempt {attempt}/{self.max_retries}: {exc}")
                    print(f"Retrying after {wait} seconds...")
                    time.sleep(wait)

        raise RuntimeError(f"{label}: failed to download CSV after {self.max_retries} attempts.") from last_error

    def _read_json_url(self, url: str, label: str) -> dict:
        """
        Read and parse a remote JSON endpoint using the shared retry downloader.
        It returns a Python object for downstream Yahoo or provider metadata parsing.
        """
        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                with self._request_url(url) as response:
                    content = response.read().decode("utf-8")
                return json.loads(content)
            except Exception as exc:
                last_error = exc
                if attempt < self.max_retries:
                    wait = self.retry_sleep_seconds * attempt
                    time.sleep(wait)

        raise RuntimeError(f"{label}: failed to download JSON after {self.max_retries} attempts.") from last_error

    def _download_yahoo_chart_close(self, tickers: list, label: str) -> pd.DataFrame:
        """
        Download daily adjusted close prices from Yahoo Finance's chart endpoint for one ticker.
        These prices are used to compute returns, covariance matrices, factor regressions and backtests.
        """
        start_ts = int(pd.Timestamp(self.start_date).tz_localize("UTC").timestamp())
        end_ts = int((pd.Timestamp(self.end_date) + pd.Timedelta(days=1)).tz_localize("UTC").timestamp())
        prices = []

        for ticker in tickers:
            encoded_ticker = urllib.parse.quote(ticker, safe="")
            url = (
                f"https://query1.finance.yahoo.com/v8/finance/chart/{encoded_ticker}"
                f"?period1={start_ts}&period2={end_ts}&interval=1d"
                "&events=history&includeAdjustedClose=true"
            )
            raw = self._read_json_url(url, f"{label} ({ticker} from Yahoo chart)")
            chart = raw.get("chart", {})
            if chart.get("error"):
                raise RuntimeError(f"{label}: Yahoo chart error for {ticker}: {chart['error']}")

            results = chart.get("result") or []
            if not results:
                raise RuntimeError(f"{label}: Yahoo chart returned no result for {ticker}.")

            result = results[0]
            timestamps = result.get("timestamp") or []
            indicators = result.get("indicators", {})
            adjclose = indicators.get("adjclose") or []
            quote = indicators.get("quote") or []

            if adjclose and adjclose[0].get("adjclose") is not None:
                values = adjclose[0]["adjclose"]
            elif quote and quote[0].get("close") is not None:
                values = quote[0]["close"]
            else:
                raise RuntimeError(f"{label}: Yahoo chart returned no close prices for {ticker}.")

            if not timestamps or len(timestamps) != len(values):
                raise RuntimeError(f"{label}: Yahoo chart returned inconsistent data lengths for {ticker}.")

            dates = pd.to_datetime(timestamps, unit="s", utc=True).tz_convert(None).normalize()
            series = pd.Series(values, index=dates, name=ticker)
            series = pd.to_numeric(series, errors="coerce").dropna()
            if series.empty:
                raise RuntimeError(f"{label}: Yahoo chart returned no usable close prices for {ticker}.")
            prices.append(series)

        data = pd.concat(prices, axis=1).sort_index()
        return data.dropna(axis=0, how="all").ffill()

    def _download_stooq_close(self, tickers: list, label: str) -> pd.DataFrame:
        """
        Download close prices from Stooq as an independent fallback when Yahoo chart data fail.
        It helps keep the notebook runnable under temporary Yahoo rate limits or outages.
        """
        start = pd.to_datetime(self.start_date).strftime("%Y%m%d")
        end = pd.to_datetime(self.end_date).strftime("%Y%m%d")
        prices = []

        for ticker in tickers:
            symbol_candidates = list(
                dict.fromkeys(
                    [
                        f"{ticker.lower()}.us",
                        f"{ticker.upper()}.US",
                        ticker.lower(),
                        ticker.upper(),
                    ]
                )
            )
            last_error = None

            for stooq_symbol in symbol_candidates:
                url = f"https://stooq.com/q/d/l/?s={stooq_symbol}&d1={start}&d2={end}&i=d"
                try:
                    raw = self._read_csv_url(url, f"{label} ({ticker} from Stooq as {stooq_symbol})")

                    if "Date" not in raw.columns or "Close" not in raw.columns:
                        raise RuntimeError(
                            f"{label}: Stooq returned unexpected columns for {ticker} "
                            f"as {stooq_symbol}: {raw.columns.tolist()}"
                        )

                    dates = pd.to_datetime(raw["Date"], errors="coerce")
                    close = pd.to_numeric(raw["Close"], errors="coerce")
                    series = pd.Series(close.values, index=dates, name=ticker).dropna()
                    series = series[~series.index.isna()]
                    if series.empty:
                        raise RuntimeError(
                            f"{label}: Stooq returned no usable close prices for {ticker} as {stooq_symbol}."
                        )

                    prices.append(series)
                    break
                except Exception as exc:
                    last_error = exc
            else:
                raise RuntimeError(
                    f"{label}: Stooq returned no usable close prices for {ticker}. "
                    f"Tried symbols: {symbol_candidates}."
                ) from last_error

        data = pd.concat(prices, axis=1).sort_index()
        return data.dropna(axis=0, how="all").ffill()

    def _cache_file_path(self, file_name: str):
        """
        Return a local cache path when cache_dir is configured and the file exists.
        The method is intentionally small because cached data should be a fallback, not the primary source.
        """
        if self.cache_dir is None:
            return None
        cache_path = self.cache_dir / file_name
        return cache_path if cache_path.exists() else None

    def _validate_cached_window(self, data: pd.DataFrame, label: str) -> None:
        """
        Check that a cached time series broadly covers the requested sample window.
        A small tolerance is allowed for weekends, holidays and same-day execution before the US market close.
        """
        if data.empty:
            raise RuntimeError(f"{label}: cached data is empty.")
        start = pd.Timestamp(self.start_date).normalize()
        end = pd.Timestamp(self.end_date).normalize()
        cache_start = pd.Timestamp(data.index.min()).normalize()
        cache_end = pd.Timestamp(data.index.max()).normalize()
        tolerance = pd.Timedelta(days=7)
        if cache_start > start + tolerance:
            raise RuntimeError(
                f"{label}: cached data starts at {cache_start.date()}, "
                f"which does not cover requested start date {start.date()}."
            )
        if cache_end < end - tolerance:
            raise RuntimeError(
                f"{label}: cached data ends at {cache_end.date()}, "
                f"which does not cover requested end date {end.date()}."
            )

    def _read_cached_timeseries_csv(self, file_name: str, label: str):
        """
        Read a date-indexed CSV previously exported by save_data_to_csv().
        It standardises the date index and restricts rows to the current requested sample window.
        """
        cache_path = self._cache_file_path(file_name)
        if cache_path is None:
            raise FileNotFoundError(f"{label}: cached file {file_name} was not found.")

        cached = pd.read_csv(cache_path, encoding=CSV_ENCODING)
        date_col = next(
            (col for col in cached.columns if str(col).lower() in {"date", "index"}),
            cached.columns[0],
        )
        cached[date_col] = pd.to_datetime(cached[date_col], errors="coerce")
        cached = cached.dropna(subset=[date_col]).set_index(date_col).sort_index()
        cached.index = pd.to_datetime(cached.index, errors="coerce")
        cached = cached.loc[~cached.index.isna()]
        if getattr(cached.index, "tz", None) is not None:
            cached.index = cached.index.tz_convert(None)
        cached.index = cached.index.normalize()
        cached = cached.loc[~cached.index.duplicated(keep="last")]
        cached = cached.loc[
            (cached.index >= pd.Timestamp(self.start_date).normalize())
            & (cached.index <= pd.Timestamp(self.end_date).normalize())
        ]
        self._validate_cached_window(cached, label)
        return cached, cache_path

    def _load_cached_price_panel(self, tickers: list, label: str, live_error: Exception) -> pd.DataFrame:
        """
        Load cached asset or factor price panels after a live price-source failure.
        The function validates ticker coverage and maps cached factor names back to ETF tickers when needed.
        It also rebuilds one leading price anchor from cached returns so pct_change() preserves the first return date.
        """
        requested = list(tickers)
        asset_tickers = list(self.asset_tickers)
        factor_symbols = list(self.factor_tickers.values())
        factor_name_to_symbol = {name: symbol for name, symbol in self.factor_tickers.items()}

        if requested == asset_tickers:
            file_name = "asset_prices.csv"
            returns_file_name = "asset_returns.csv"
            rename_map = {}
        elif requested == factor_symbols:
            file_name = "factor_prices.csv"
            returns_file_name = "factor_returns.csv"
            rename_map = factor_name_to_symbol
        else:
            raise RuntimeError(f"{label}: no cache mapping is defined for tickers {requested}.")

        cached, cache_path = self._read_cached_timeseries_csv(file_name, label)
        cached = cached.rename(columns=rename_map)
        missing = [ticker for ticker in requested if ticker not in cached.columns]
        if missing:
            raise RuntimeError(f"{label}: cached file {cache_path} is missing columns {missing}.")

        data = cached.loc[:, requested].apply(pd.to_numeric, errors="coerce")
        data = data.dropna(axis=0, how="all").ffill()
        missing_after_parse = data.columns[data.isna().all()].tolist()
        if missing_after_parse:
            raise RuntimeError(f"{label}: cached file {cache_path} has no usable numeric data for {missing_after_parse}.")

        returns_cache_path = self._cache_file_path(returns_file_name)
        if returns_cache_path is not None and not data.empty:
            try:
                cached_returns, _ = self._read_cached_timeseries_csv(returns_file_name, f"{label} return cache")
                cached_returns = cached_returns.rename(columns=rename_map)
                if all(ticker in cached_returns.columns for ticker in requested):
                    first_date = data.index.min()
                    if first_date in cached_returns.index:
                        first_returns = pd.to_numeric(cached_returns.loc[first_date, requested], errors="coerce")
                        first_prices = pd.to_numeric(data.loc[first_date, requested], errors="coerce")
                        if first_returns.notna().all() and first_prices.notna().all() and (first_returns > -0.999999).all():
                            anchor_prices = first_prices / (1 + first_returns)
                            anchor_date = first_date - pd.tseries.offsets.BDay(1)
                            if anchor_date not in data.index:
                                anchor = pd.DataFrame([anchor_prices], index=[anchor_date])
                                anchor.index.name = data.index.name
                                data = pd.concat([anchor, data]).sort_index()
            except Exception as anchor_error:
                print(f"{label}: cached return anchor was not applied ({type(anchor_error).__name__}).")

        print(
            f"{label}: live {self.price_source} download failed ({type(live_error).__name__}); "
            f"using cached CSV fallback: {cache_path}"
        )
        return data

    def _load_cached_risk_free_rate(self, live_error: Exception) -> pd.Series:
        """
        Load a cached daily FRED DGS3MO series only after its source metadata passes validation.
        """
        cached, cache_path = self._read_cached_timeseries_csv(
            "rf_dgs3mo_daily.csv", "FRED DGS3MO risk-free rate"
        )
        metadata_path = self._cache_file_path("rf_metadata.json")
        if metadata_path is None:
            raise FileNotFoundError("Missing source-specific risk-free metadata: rf_metadata.json")
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata.get("source") != "FRED" or metadata.get("series_id") != "DGS3MO":
            raise RuntimeError("Cached risk-free metadata is not FRED DGS3MO; refusing fallback.")
        required_contract = {
            "fred_observed_end": "2026-07-22",
            "aligned_end": "2026-07-23",
            "alignment_method": "asof_forward_fill_no_backfill",
            "terminal_forward_fill_dates": ["2026-07-23"],
        }
        for field, expected in required_contract.items():
            if metadata.get(field) != expected:
                raise RuntimeError(f"Cached risk-free metadata mismatch for {field}.")
        numeric = cached.apply(pd.to_numeric, errors="coerce")
        if numeric.empty:
            raise RuntimeError(f"FRED DGS3MO cache {cache_path} has no numeric columns.")
        series = numeric.iloc[:, 0].dropna()
        if series.empty:
            raise RuntimeError(f"FRED DGS3MO cache {cache_path} has no usable values.")
        series.name = "DGS3MO"
        self.rf_metadata = metadata
        print(
            f"FRED DGS3MO live download failed ({type(live_error).__name__}); "
            f"using source-validated cached snapshot: {cache_path}"
        )
        return series

    def _download_fred_series(self, series_id: str) -> pd.Series:
        """
        Download a FRED time series through the public CSV endpoint without an API key.
        It is mainly used as a backup path for risk-free-rate data.
        """
        query = urllib.parse.urlencode({
            "id": series_id, "cosd": self.start_date, "coed": self.end_date,
        })
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?{query}"
        raw = self._read_csv_url(url, f"FRED {series_id}")

        if "observation_date" not in raw.columns or series_id not in raw.columns:
            raise RuntimeError(f"FRED {series_id}: unexpected columns: {raw.columns.tolist()}")

        dates = pd.to_datetime(raw["observation_date"], errors="coerce")
        values = pd.to_numeric(raw[series_id].replace(".", np.nan), errors="coerce")
        series = pd.Series(values.values, index=dates, name=series_id).dropna()
        series = series.loc[
            (series.index >= pd.to_datetime(self.start_date))
            & (series.index <= pd.to_datetime(self.end_date))
        ]
        if series.empty:
            raise RuntimeError(f"FRED {series_id}: no observations in requested date range.")
        return series

    def _download_adj_close(self, tickers: list, label: str) -> pd.DataFrame:
        """
        Download adjusted close prices for one ticker using the configured price source and fallbacks.
        It standardises output as a named Series aligned to the requested date range.
        """
        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                raw = yf.download(
                    tickers=tickers,
                    start=self.start_date,
                    end=self.end_date,
                    auto_adjust=False,
                    progress=False,
                    threads=False,
                    timeout=30,
                )

                if raw.empty:
                    raise ValueError(f"{label}: yfinance returned an empty table.")

                try:
                    data = raw["Adj Close"]
                except KeyError as exc:
                    raise ValueError(f"{label}: yfinance response has no 'Adj Close' column.") from exc
                if isinstance(data, pd.Series):
                    col_name = tickers[0] if isinstance(tickers, list) else tickers
                    data = data.to_frame(name=col_name)

                data = data.dropna(axis=0, how="all").ffill()
                if data.empty or data.dropna(axis=0, how="all").empty:
                    raise ValueError(f"{label}: adjusted close prices are all missing.")

                return data
            except Exception as exc:
                last_error = exc
                if attempt < self.max_retries:
                    wait = self.retry_sleep_seconds * attempt
                    print(f"{label}: download failed on attempt {attempt}/{self.max_retries}: {exc}")
                    print(f"Retrying after {wait} seconds...")
                    time.sleep(wait)

        root_cause = (
            "unknown error"
            if last_error is None
            else f"{last_error.__class__.__name__}: {last_error}"
        )
        raise RuntimeError(
            f"{label}: failed to download from yfinance after {self.max_retries} attempts. "
            f"Last error: {root_cause}. "
            "Possible causes include Yahoo rate limiting/blocking, a network or proxy problem, "
            "an unavailable ticker/date range, or a changed Yahoo response format. "
            "Try again later, reduce the ticker batch size, or switch to "
            "price_source='yahoo_chart' and risk_free_source='yahoo_chart'."
        ) from last_error

    def download_prices(self, tickers: list) -> pd.DataFrame:
        """
        Download a panel of adjusted close prices for a list of tickers.
        It combines individual price Series into a clean DataFrame for assets or factor proxies.
        """
        if self.price_source not in {"yahoo_chart", "stooq", "yfinance"}:
            raise ValueError("price_source must be 'yahoo_chart', 'stooq', or 'yfinance'.")

        try:
            if self.price_source == "yahoo_chart":
                data = self._download_yahoo_chart_close(tickers, "price data")
            elif self.price_source == "stooq":
                data = self._download_stooq_close(tickers, "price data")
            else:
                data = self._download_adj_close(tickers, "price data")
        except Exception as live_error:
            try:
                data = self._load_cached_price_panel(tickers, "price data", live_error)
            except Exception as cache_error:
                raise RuntimeError(
                    f"price data: live {self.price_source} download failed and cached CSV fallback was unavailable."
                ) from cache_error

        data = data.reindex(columns=tickers)
        missing = data.columns[data.isna().all()].tolist()
        if missing:
            raise RuntimeError(f"price data: missing adjusted close prices for {missing}")
        return data

    def calc_simple_returns(self, prices: pd.DataFrame) -> pd.DataFrame:
        """
        Convert price levels into simple daily arithmetic returns.
        Missing rows introduced by percentage changes are removed before modelling.
        """
        returns = prices.pct_change().dropna()
        return returns

    def get_risk_free_rate(self) -> pd.Series:
        """
        Obtain the daily risk-free-rate proxy and align it to the project sample window.
        The series is used to convert raw asset and factor returns into excess returns.
        """
        if self.risk_free_source != "fred":
            raise ValueError("The report requires risk_free_source='fred' with series DGS3MO.")

        try:
            if self.fred_api_key:
                fred = Fred(api_key=self.fred_api_key)
                rf_percent = fred.get_series("DGS3MO", self.start_date, self.end_date)
            else:
                rf_percent = self._download_fred_series("DGS3MO")
        except Exception as live_error:
            try:
                return self._load_cached_risk_free_rate(live_error)
            except Exception as cache_error:
                raise RuntimeError(
                    "FRED DGS3MO download failed and no source-validated DGS3MO cache was available."
                ) from cache_error

        rf_percent = pd.Series(rf_percent, dtype=float).sort_index().dropna()
        rf_daily = (rf_percent / 100.0 / 252.0).reindex(
            pd.date_range(self.start_date, self.end_date, freq="B")
        ).ffill().dropna()
        rf_daily.name = "DGS3MO"
        self.rf_metadata = {
            "source": "FRED",
            "series_id": "DGS3MO",
            "title": "Market Yield on U.S. Treasury Securities at 3-Month Constant Maturity",
            "units": "Percent per annum, quoted on an investment basis",
            "daily_conversion": "DGS3MO / 100 / 252",
            "source_url": "https://fred.stlouisfed.org/series/DGS3MO",
            "requested_start": self.start_date,
            "requested_end": self.end_date,
            "fred_observed_start": str(rf_percent.index.min().date()),
            "fred_observed_end": str(rf_percent.index.max().date()),
            "alignment_method": "asof_forward_fill_no_backfill",
            "retrieved_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        }
        return rf_daily

    @staticmethod
    def _decode_text(content: bytes) -> str:
        """
        Decode downloaded provider files that may use UTF-8, UTF-16 or Windows encodings.
        This makes downstream HTML, CSV and JSON parsing more robust.
        """
        for encoding in ("utf-8-sig", "utf-16", "cp1252", "latin-1"):
            try:
                return content.decode(encoding)
            except UnicodeDecodeError:
                continue
        return content.decode("utf-8", errors="replace")

    @staticmethod
    def _html_to_text(content: bytes) -> str:
        """
        Convert HTML bytes into readable plain text for regex-based extraction.
        Tags are stripped and HTML entities are unescaped before searching for AUM fields.
        """
        raw = DataLoader._decode_text(content)
        raw = re.sub(r"<script\b[^>]*>.*?</script>", " ", raw, flags=re.IGNORECASE | re.DOTALL)
        raw = re.sub(r"<style\b[^>]*>.*?</style>", " ", raw, flags=re.IGNORECASE | re.DOTALL)
        raw = re.sub(r"<[^>]+>", " ", raw)
        return re.sub(r"\s+", " ", html.unescape(raw)).strip()

    @staticmethod
    def _normalize_label(value) -> str:
        """
        Normalise labels by lowercasing and removing non-alphanumeric characters.
        This allows provider field names with different punctuation to be matched reliably.
        """
        return re.sub(r"[^a-z0-9]+", "", str(value).lower())

    @staticmethod
    def _parse_usd_value(value) -> float:
        """
        Parse USD amounts from strings such as dollar values, millions, billions or trillions.
        The output is a numeric value in US dollars used for ETF AUM calculations.
        """
        if value is None:
            return np.nan
        if isinstance(value, (int, float, np.integer, np.floating)) and not pd.isna(value):
            parsed = float(value)
            return parsed if parsed > 0 else np.nan

        text_value = html.unescape(str(value)).strip()
        if not text_value or text_value.lower() in {"nan", "none", "n/a", "--", "-"}:
            return np.nan
        if "%" in text_value:
            return np.nan

        lower = text_value.lower()
        multiplier = 1.0
        if re.search(r"\b(trillion|tn)\b", lower) or re.search(r"\d\s*t$", lower):
            multiplier = 1_000_000_000_000.0
        elif re.search(r"\b(billion|bn)\b", lower) or re.search(r"\d\s*b$", lower):
            multiplier = 1_000_000_000.0
        elif re.search(r"\b(million|mn|mm)\b", lower) or re.search(r"\d\s*m$", lower):
            multiplier = 1_000_000.0
        elif re.search(r"\b(thousand|k)\b", lower) or re.search(r"\d\s*k$", lower):
            multiplier = 1_000.0

        cleaned = (
            lower.replace("$", "")
            .replace(",", "")
            .replace("usd", "")
            .replace("us$", "")
        )
        match = re.search(r"[-+]?\d*\.?\d+", cleaned)
        if not match:
            return np.nan
        parsed = float(match.group(0)) * multiplier
        return parsed if parsed > 0 else np.nan

    @staticmethod
    def _coerce_date_string(value):
        """
        Convert provider date-like values into a standard YYYY-MM-DD string when possible.
        It supports pandas timestamps and common text formats used in public files.
        """
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return None
        parsed = pd.to_datetime(value, errors="coerce")
        if pd.isna(parsed):
            return None
        return parsed.date().isoformat()

    @staticmethod
    def _find_as_of_date(text: str):
        """
        Search text for an official as-of date near AUM or holdings information.
        The result is stored with the AUM record for auditability.
        """
        date_patterns = [
            r"(?:as of|as at|holdings as of|fund holdings as of)\s*[:,-]?\s*"
            r"([A-Za-z]{3,9}\s+\d{1,2},\s+\d{4})",
            r"(?:as of|as at|holdings as of|fund holdings as of)\s*[:,-]?\s*"
            r"([A-Za-z]{3,9}\s+\d{1,2}\s+\d{4})",
            r"(?:as of|as at|holdings as of|fund holdings as of)\s*[:,-]?\s*"
            r"(\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4})",
            r"(?:as of|as at|holdings as of|fund holdings as of)\s*[:,-]?\s*"
            r"(\d{1,2}/\d{1,2}/\d{2,4})",
            r"(?:as of|as at|holdings as of|fund holdings as of)\s*[:,-]?\s*"
            r"(\d{4}-\d{1,2}-\d{1,2})",
            r"(?:as of|as at|holdings as of|fund holdings as of)\s*[:,-]?\s*"
            r"(\d{1,2}-[A-Za-z]{3,9}-\d{2,4})",
        ]
        for pattern in date_patterns:
            match = re.search(pattern, text, flags=re.IGNORECASE)
            if match:
                date_value = DataLoader._coerce_date_string(match.group(1))
                if date_value:
                    return date_value
        return None

    def _market_aum_row(
        self,
        ticker: str,
        asset_value_usd: float,
        source_field: str,
        source: str,
        source_url: str,
        as_of_date=None,
    ) -> dict:
        """
        Build one standardised ETF-AUM record for a ticker.
        The record includes ticker, AUM value, source field, source URL, provider date and retrieval time.
        """
        asset_value_usd = pd.to_numeric(asset_value_usd, errors="coerce")
        if pd.isna(asset_value_usd) or asset_value_usd <= 0:
            raise ValueError(f"market AUM ({ticker}): parsed AUM is missing or non-positive.")
        return {
            "ticker": ticker,
            "asset_value_usd": float(asset_value_usd),
            "source_field": source_field,
            "source": source,
            "source_url": source_url,
            "as_of_date": as_of_date,
            "as_of_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        }

    def _extract_labeled_net_assets(self, text: str, labels: list):
        """
        Extract net-assets or AUM values from text using labelled field patterns.
        It is designed for provider pages that present AUM as visible text.
        """
        compact = re.sub(r"\s+", " ", text)
        date_pattern = (
            r"(?:[A-Za-z]{3,9}\s+\d{1,2},\s+\d{4}|"
            r"[A-Za-z]{3,9}\s+\d{1,2}\s+\d{4}|"
            r"\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4}|"
            r"\d{1,2}/\d{1,2}/\d{2,4}|"
            r"\d{4}-\d{1,2}-\d{1,2}|"
            r"\d{1,2}-[A-Za-z]{3,9}-\d{2,4})"
        )
        money_pattern = r"\$\s*\d[\d,]*(?:\.\d+)?(?:\s*(?:trillion|billion|million|tn|bn|mn|mm|b|m))?"

        for label in labels:
            escaped = re.escape(label)
            patterns = [
                rf"{escaped}\s+as of\s+(?P<date>{date_pattern})\s+(?P<value>{money_pattern})",
                rf"{escaped}.{{0,140}}?(?P<value>{money_pattern})",
            ]
            for pattern in patterns:
                match = re.search(pattern, compact, flags=re.IGNORECASE)
                if not match:
                    continue
                value = self._parse_usd_value(match.group("value"))
                if pd.notna(value) and value > 1_000_000:
                    as_of_date = match.groupdict().get("date")
                    return value, label, self._coerce_date_string(as_of_date) or self._find_as_of_date(compact)
        return np.nan, None, None

    def _extract_metadata_net_assets(self, text: str):
        """
        Extract net-assets values from embedded metadata or structured text blocks.
        It is a fallback when the value is not available in normal visible text.
        """
        label_pattern = re.compile(
            r"\b(net assets|total net assets|net asset value|total net asset value|assets under management|aum)\b",
            flags=re.IGNORECASE,
        )
        for delimiter in [",", "\t", ";"]:
            reader = csv.reader(io.StringIO(text), delimiter=delimiter)
            for row in reader:
                if not row:
                    continue
                joined = " ".join(str(cell).strip() for cell in row if str(cell).strip())
                if not label_pattern.search(joined):
                    continue
                values = [self._parse_usd_value(cell) for cell in row]
                values = [value for value in values if pd.notna(value) and value > 1_000_000]
                if values:
                    return max(values), "reported_net_assets", self._find_as_of_date(joined) or self._find_as_of_date(text)
        return np.nan, None, None

    def _find_delimited_table(self, text: str, required_column_terms: list) -> pd.DataFrame:
        """
        Detect and parse a delimited table from plain text content.
        This helps read holdings files whose delimiter is comma, tab or another separator.
        """
        lines = [line for line in text.splitlines() if line.strip()]
        for delimiter in [",", "\t", ";"]:
            for header_idx, line in enumerate(lines[:120]):
                try:
                    cells = next(csv.reader([line], delimiter=delimiter))
                except csv.Error:
                    continue
                normalized = [self._normalize_label(cell) for cell in cells]
                if not any(all(term in column for term in required_column_terms) for column in normalized):
                    continue
                table_text = "\n".join(lines[header_idx:])
                try:
                    table = pd.read_csv(
                        io.StringIO(table_text),
                        sep=delimiter,
                        engine="python",
                        on_bad_lines="skip",
                    )
                except Exception:
                    continue
                if not table.empty:
                    return table
        return pd.DataFrame()

    def _sum_market_value_column(self, table: pd.DataFrame):
        """
        Sum a market-value column in a holdings table to estimate fund AUM.
        It is used when provider files disclose holdings values instead of a single AUM number.
        """
        for column in table.columns:
            normalized = self._normalize_label(column)
            if "marketvalue" not in normalized:
                continue
            values = table[column].map(self._parse_usd_value).dropna()
            total = values.sum()
            if total > 1_000_000:
                return float(total), f"sum({column})"
        return np.nan, None

    @staticmethod
    def _xlsx_column_index(cell_reference: str) -> int:
        """
        Convert an Excel cell reference such as C12 into a zero-based column index.
        It supports the lightweight XLSX parser used when openpyxl is unavailable.
        """
        letters = re.sub(r"[^A-Z]", "", str(cell_reference).upper())
        index = 0
        for letter in letters:
            index = index * 26 + (ord(letter) - ord("A") + 1)
        return max(index - 1, 0)

    @staticmethod
    def _xlsx_text(element) -> str:
        """
        Extract concatenated text from an XLSX XML element.
        This helper reads inline strings and shared-string references in raw XLSX files.
        """
        if element is None:
            return ""
        return "".join(node.text or "" for node in element.iter() if node.tag.endswith("}t"))

    def _read_xlsx_without_openpyxl(self, content: bytes) -> dict:
        """
        Read basic XLSX worksheets using only the Python standard library.
        This fallback keeps GLD or holdings-file parsing available when openpyxl is not installed.
        """
        namespace = {
            "a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
            "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
            "rel": "http://schemas.openxmlformats.org/package/2006/relationships",
        }

        with zipfile.ZipFile(io.BytesIO(content)) as workbook_zip:
            shared_strings = []
            if "xl/sharedStrings.xml" in workbook_zip.namelist():
                shared_root = ET.fromstring(workbook_zip.read("xl/sharedStrings.xml"))
                for item in shared_root.findall("a:si", namespace):
                    shared_strings.append(self._xlsx_text(item))

            workbook_root = ET.fromstring(workbook_zip.read("xl/workbook.xml"))
            rels_root = ET.fromstring(workbook_zip.read("xl/_rels/workbook.xml.rels"))
            relationships = {}
            for relationship in rels_root.findall("rel:Relationship", namespace):
                target = relationship.attrib.get("Target", "")
                if target.startswith("/"):
                    target = target.lstrip("/")
                elif not target.startswith("xl/"):
                    target = f"xl/{target}"
                relationships[relationship.attrib["Id"]] = target

            sheets = {}
            for sheet in workbook_root.findall("a:sheets/a:sheet", namespace):
                sheet_name = sheet.attrib.get("name", "Sheet")
                rel_id = sheet.attrib.get(f"{{{namespace['r']}}}id")
                sheet_path = relationships.get(rel_id)
                if not sheet_path or sheet_path not in workbook_zip.namelist():
                    continue

                sheet_root = ET.fromstring(workbook_zip.read(sheet_path))
                rows = []
                max_width = 0
                for row in sheet_root.findall(".//a:sheetData/a:row", namespace):
                    values = []
                    for cell in row.findall("a:c", namespace):
                        cell_ref = cell.attrib.get("r", "")
                        col_idx = self._xlsx_column_index(cell_ref)
                        while len(values) <= col_idx:
                            values.append("")

                        cell_type = cell.attrib.get("t")
                        value_element = cell.find("a:v", namespace)
                        if cell_type == "s" and value_element is not None:
                            value_index = int(value_element.text)
                            value = shared_strings[value_index] if value_index < len(shared_strings) else ""
                        elif cell_type == "inlineStr":
                            value = self._xlsx_text(cell.find("a:is", namespace))
                        elif value_element is not None:
                            value = value_element.text or ""
                        else:
                            value = ""
                        values[col_idx] = value
                    max_width = max(max_width, len(values))
                    rows.append(values)

                normalized_rows = [row + [""] * (max_width - len(row)) for row in rows]
                sheets[sheet_name] = pd.DataFrame(normalized_rows)

        if not sheets:
            raise RuntimeError("No readable sheets found in provider XLSX file.")
        return sheets

    def _read_excel_sheets(self, content: bytes) -> dict:
        """
        Read all sheets from an Excel file using openpyxl when available, otherwise the standard-library fallback.
        It returns a dictionary of sheet names to DataFrames for downstream AUM extraction.
        """
        try:
            return pd.read_excel(io.BytesIO(content), sheet_name=None, header=None)
        except Exception:
            return self._read_xlsx_without_openpyxl(content)

    def _extract_holdings_aum_from_text(self, content: bytes):
        """
        Estimate AUM from a text holdings file by locating and summing market-value fields.
        It handles provider downloads that are text or CSV rather than Excel.
        """
        text = self._decode_text(content)
        metadata_value, metadata_field, metadata_date = self._extract_metadata_net_assets(text)
        if pd.notna(metadata_value):
            return metadata_value, metadata_field, metadata_date

        table = self._find_delimited_table(text, ["market", "value"])
        if not table.empty:
            value, field = self._sum_market_value_column(table)
            if pd.notna(value):
                return value, field, self._find_as_of_date(text)

        raise RuntimeError("No net-assets or market-value field found in provider text file.")

    def _extract_holdings_aum_from_excel(self, content: bytes):
        """
        Estimate AUM from Excel holdings sheets by summing usable market-value columns.
        It is used for provider archives where fund assets are implied by holdings values.
        """
        try:
            sheets = self._read_excel_sheets(content)
        except Exception as exc:
            raise RuntimeError(
                "Could not parse provider Excel file with pandas or the built-in XLSX fallback."
            ) from exc

        for sheet_name, frame in sheets.items():
            if frame.empty:
                continue
            for row_idx in range(min(len(frame), 120)):
                cells = ["" if pd.isna(value) else str(value).strip() for value in frame.iloc[row_idx].tolist()]
                normalized = [self._normalize_label(cell) for cell in cells]
                if not any("marketvalue" in column for column in normalized):
                    continue

                table = frame.iloc[row_idx + 1:].copy()
                table.columns = cells
                value, field = self._sum_market_value_column(table)
                if pd.notna(value):
                    sheet_text = " ".join(cells)
                    return value, f"{field} from {sheet_name}", self._find_as_of_date(sheet_text)

        raise RuntimeError("No market-value column found in provider Excel file.")

    def _extract_latest_value_from_excel(self, content: bytes, value_terms: list):
        """
        Extract the latest dated numeric value from Excel sheets.
        It supports historical AUM archives where each row is a dated snapshot.
        """
        try:
            sheets = self._read_excel_sheets(content)
        except Exception as exc:
            raise RuntimeError(
                "Could not parse provider Excel file with pandas or the built-in XLSX fallback."
            ) from exc

        for sheet_name, frame in sheets.items():
            if frame.empty:
                continue
            for row_idx in range(min(len(frame), 120)):
                cells = ["" if pd.isna(value) else str(value).strip() for value in frame.iloc[row_idx].tolist()]
                normalized = [self._normalize_label(cell) for cell in cells]

                value_col = None
                date_col = None
                for idx, column in enumerate(normalized):
                    if all(term in column for term in value_terms):
                        value_col = cells[idx]
                    if column in {"date", "asofdate"} or column.endswith("date"):
                        date_col = cells[idx]
                if value_col is None:
                    continue

                table = frame.iloc[row_idx + 1:].copy()
                table.columns = cells
                parsed = table[value_col].map(self._parse_usd_value)
                valid = table.loc[parsed.notna()].copy()
                if valid.empty:
                    continue
                valid["_parsed_value"] = parsed.loc[parsed.notna()].values

                if date_col and date_col in valid.columns:
                    valid["_parsed_date"] = pd.to_datetime(valid[date_col], errors="coerce")
                    valid_with_dates = valid.loc[valid["_parsed_date"].notna()]
                    if not valid_with_dates.empty:
                        row = valid_with_dates.sort_values("_parsed_date").iloc[-1]
                        return (
                            float(row["_parsed_value"]),
                            value_col,
                            self._coerce_date_string(row["_parsed_date"]),
                        )

                row = valid.iloc[-1]
                return float(row["_parsed_value"]), value_col, None

        raise RuntimeError("No matching net-asset-value column found in provider Excel file.")

    def _extract_latest_value_from_json(self, content: bytes, value_terms: list):
        """
        Recursively search JSON-like content for the latest usable numeric AUM value.
        It is a defensive parser for provider or Yahoo metadata structures.
        """
        raw = json.loads(self._decode_text(content))
        records = []

        def collect_records(value):
            """
            Recursively collect candidate records containing dates and numeric values from nested JSON objects.
            The parent function ranks these records to select the most recent valid AUM observation.
            """
            if isinstance(value, dict):
                if any(all(term in self._normalize_label(key) for term in value_terms) for key in value.keys()):
                    records.append(value)
                for child in value.values():
                    collect_records(child)
            elif isinstance(value, list):
                for child in value:
                    collect_records(child)

        collect_records(raw)
        candidates = []
        for record in records:
            value_key = None
            date_key = None
            for key in record.keys():
                normalized = self._normalize_label(key)
                if all(term in normalized for term in value_terms):
                    value_key = key
                if normalized in {"date", "asofdate"} or normalized.endswith("date"):
                    date_key = key
            if not value_key:
                continue
            parsed_value = self._parse_usd_value(record.get(value_key))
            if pd.isna(parsed_value):
                continue
            parsed_date = self._coerce_date_string(record.get(date_key)) if date_key else None
            candidates.append((parsed_date, parsed_value, value_key))

        if not candidates:
            raise RuntimeError("No matching net-asset-value field found in provider JSON.")

        candidates.sort(key=lambda item: item[0] or "")
        as_of_date, value, field = candidates[-1]
        return value, field, as_of_date

    def _download_sector_spdr_aum(self, ticker: str) -> dict:
        """
        Download and parse official AUM for a Select Sector SPDR ETF from State Street / SSGA sources.
        It tries the product page and related public data files before returning a standard AUM row.
        """
        slug = self.sector_spdr_slugs.get(ticker)
        errors = []
        if slug:
            page_url = f"https://www.ssga.com/us/en/intermediary/etfs/funds/{slug}"
            try:
                content = self._read_url_bytes(page_url, f"market AUM ({ticker} State Street product page)")
                text = self._html_to_text(content)
                value, field, as_of_date = self._extract_labeled_net_assets(
                    text,
                    ["Total Net Assets", "Net Assets", "Assets Under Management"],
                )
                if pd.notna(value):
                    return self._market_aum_row(
                        ticker,
                        value,
                        field,
                        "State Street official product page",
                        page_url,
                        as_of_date,
                    )
            except Exception as exc:
                errors.append(f"State Street product page: {exc}")

        candidates = [
            (
                "Select Sector SPDR official holdings CSV",
                f"https://www.sectorspdrs.com/sectorspdr/IDCO.Client.Spdrs.Holdings/Export/ExportCsv?symbol={ticker}",
            ),
            (
                "State Street global official holdings XLSX",
                "https://www.ssga.com/library-content/"
                f"products/fund-data/etfs/us/holdings-daily-us-en-{ticker.lower()}.xlsx",
            ),
            (
                "State Street official holdings XLSX",
                "https://www.ssga.com/us/en/intermediary/etfs/library-content/"
                f"products/fund-data/etfs/us/holdings-daily-us-en-{ticker.lower()}.xlsx",
            ),
            (
                "State Street institutional official holdings XLSX",
                "https://www.ssga.com/us/en/institutional/etfs/library-content/"
                f"products/fund-data/etfs/us/holdings-daily-us-en-{ticker.lower()}.xlsx",
            ),
        ]
        for source, url in candidates:
            try:
                content = self._read_url_bytes(url, f"market AUM ({ticker} from {source})")
                if content[:2] == b"PK" or url.lower().endswith(".xlsx"):
                    value, field, as_of_date = self._extract_holdings_aum_from_excel(content)
                else:
                    value, field, as_of_date = self._extract_holdings_aum_from_text(content)
                return self._market_aum_row(ticker, value, field, source, url, as_of_date)
            except Exception as exc:
                errors.append(f"{source}: {exc}")

        raise RuntimeError(f"market AUM ({ticker}): official provider sources failed: {' | '.join(errors)}")

    def _download_ishares_aum(self, ticker: str) -> dict:
        """
        Download and parse official AUM for an iShares ETF such as TLT.
        It uses iShares public product and holdings resources and records the source used.
        """
        product_id = {"TLT": "239454"}[ticker]
        page_url = f"https://www.ishares.com/us/products/{product_id}/ishares-20-year-treasury-bond-etf"
        errors = []

        try:
            content = self._read_url_bytes(page_url, f"market AUM ({ticker} iShares product page)")
            text = self._html_to_text(content)
            value, field, as_of_date = self._extract_labeled_net_assets(text, ["Net Assets of Fund"])
            if pd.notna(value):
                return self._market_aum_row(ticker, value, field, "iShares official product page", page_url, as_of_date)
        except Exception as exc:
            errors.append(f"iShares product page: {exc}")

        holdings_url = (
            f"{page_url}/1467271812596.ajax?"
            f"fileType=csv&fileName={ticker}_holdings&dataType=fund"
        )
        try:
            content = self._read_url_bytes(holdings_url, f"market AUM ({ticker} iShares holdings CSV)")
            value, field, as_of_date = self._extract_holdings_aum_from_text(content)
            return self._market_aum_row(ticker, value, field, "iShares official holdings CSV", holdings_url, as_of_date)
        except Exception as exc:
            errors.append(f"iShares holdings CSV: {exc}")

        raise RuntimeError(f"market AUM ({ticker}): official provider sources failed: {' | '.join(errors)}")

    def _download_gld_aum(self, ticker: str) -> dict:
        """
        Download and parse official AUM for GLD from SPDR Gold Shares public resources.
        It handles both the product page and historical archive files.
        """
        page_url = "https://www.spdrgoldshares.com/usa/gld/"
        archive_url = "https://api.spdrgoldshares.com/api/v1/historical-archive?exchange=NYSE&lang=en&product=gld"
        errors = []

        try:
            content = self._read_url_bytes(page_url, "market AUM (GLD SPDR Gold Shares product page)")
            text = self._html_to_text(content)
            value, field, as_of_date = self._extract_labeled_net_assets(
                text,
                ["Total Net Asset Value in the Trust"],
            )
            if pd.notna(value):
                return self._market_aum_row(ticker, value, field, "SPDR Gold Shares official product page", page_url, as_of_date)
        except Exception as exc:
            errors.append(f"SPDR Gold Shares product page: {exc}")

        try:
            content = self._read_url_bytes(archive_url, "market AUM (GLD SPDR Gold Shares historical archive)")
            if content[:2] == b"PK":
                value, field, as_of_date = self._extract_latest_value_from_excel(
                    content,
                    ["total", "net", "asset", "value"],
                )
            else:
                stripped = content.lstrip()
                if stripped.startswith(b"{") or stripped.startswith(b"["):
                    value, field, as_of_date = self._extract_latest_value_from_json(
                        content,
                        ["total", "net", "asset", "value"],
                    )
                else:
                    value, field, as_of_date = self._extract_holdings_aum_from_text(content)
            return self._market_aum_row(ticker, value, field, "SPDR Gold Shares official historical archive", archive_url, as_of_date)
        except Exception as exc:
            errors.append(f"SPDR Gold Shares historical archive: {exc}")

        raise RuntimeError(f"market AUM ({ticker}): official provider sources failed: {' | '.join(errors)}")

    def _download_official_market_aum_asset(self, ticker: str) -> dict:
        """
        Route one ticker to the correct official ETF provider AUM downloader.
        It centralises provider-specific logic for sector SPDR, iShares and GLD assets.
        """
        if ticker in self.sector_spdr_tickers:
            return self._download_sector_spdr_aum(ticker)
        if ticker == "TLT":
            return self._download_ishares_aum(ticker)
        if ticker == "GLD":
            return self._download_gld_aum(ticker)
        raise ValueError(f"market AUM ({ticker}): no official provider mapping configured.")

    def _download_official_market_aum(self, tickers: list) -> pd.DataFrame:
        """
        Download official AUM records for the full asset universe.
        It validates completeness and returns a standard DataFrame used to build market weights.
        """
        rows = []
        errors = {}
        for ticker in tickers:
            try:
                rows.append(self._download_official_market_aum_asset(ticker))
            except Exception as exc:
                errors[ticker] = str(exc)

        if errors:
            details = "\n".join(f"- {ticker}: {message}" for ticker, message in errors.items())
            raise RuntimeError(
                "market AUM: failed to obtain official ETF AUM for the following tickers.\n"
                f"{details}\n"
                "For an academic final project, do not replace these with static assumptions. "
                "Either rerun with network access to the official provider sites, or set "
                "market_aum_source='file' and provide an audited CSV with columns "
                "ticker, asset_value_usd, as_of_date, source, source_url."
            )

        market_aum = pd.DataFrame(rows).set_index("ticker").reindex(tickers)
        missing = market_aum.index[market_aum["asset_value_usd"].isna()].tolist()
        if missing:
            raise RuntimeError(f"market AUM: official provider data is missing AUM for {missing}.")
        return market_aum

    def _read_market_aum_file(self, file_path: Path) -> pd.DataFrame:
        """
        Read a user-supplied audited AUM CSV file as an alternative to live official webpages.
        This supports reproducibility if public provider pages become unavailable or change structure.
        """
        if file_path is None:
            raise ValueError(
                "market_aum_source='file' requires market_aum_file='path/to/market_aum.csv'."
            )
        if not file_path.exists():
            raise FileNotFoundError(f"market AUM file not found: {file_path}")

        market_aum = pd.read_csv(file_path)
        if "ticker" not in market_aum.columns:
            if market_aum.columns[0].lower() in {"index", "unnamed: 0"}:
                market_aum = market_aum.rename(columns={market_aum.columns[0]: "ticker"})
            else:
                raise ValueError("market AUM file must contain a 'ticker' column.")

        value_column = None
        for candidate in ["asset_value_usd", "net_assets_usd", "aum_usd", "total_net_assets_usd"]:
            if candidate in market_aum.columns:
                value_column = candidate
                break
        if value_column is None:
            raise ValueError(
                "market AUM file must contain one of: asset_value_usd, net_assets_usd, "
                "aum_usd, total_net_assets_usd."
            )

        market_aum["asset_value_usd"] = market_aum[value_column].map(self._parse_usd_value)
        market_aum = market_aum.set_index("ticker").reindex(self.asset_tickers)
        missing = market_aum.index[market_aum["asset_value_usd"].isna()].tolist()
        if missing:
            raise ValueError(f"market AUM file is missing usable AUM values for {missing}.")

        if "source_field" not in market_aum.columns:
            market_aum["source_field"] = value_column
        if "source" not in market_aum.columns:
            market_aum["source"] = "User-provided official AUM CSV"
        if "source_url" not in market_aum.columns:
            market_aum["source_url"] = str(file_path)
        if "as_of_date" not in market_aum.columns:
            market_aum["as_of_date"] = None
        if "as_of_utc" not in market_aum.columns:
            market_aum["as_of_utc"] = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
        else:
            fallback_time = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
            market_aum["as_of_utc"] = market_aum["as_of_utc"].fillna(fallback_time)
        return market_aum[
            ["asset_value_usd", "source_field", "source", "source_url", "as_of_date", "as_of_utc"]
        ]

    @staticmethod
    def _numeric_field(data: dict, field_names: list):
        """
        Return the first usable numeric value from a list of possible Yahoo Finance response fields.
        It handles nested dictionaries that store both raw and formatted values.
        """
        for field in field_names:
            value = data.get(field)
            if isinstance(value, dict):
                value = value.get("raw")
            value = pd.to_numeric(value, errors="coerce")
            if pd.notna(value) and value > 0:
                return float(value), field
        return None, None

    def _download_yfinance_asset(self, ticker: str) -> dict:
        """
        Download one ETF's net assets through yfinance as a non-official fallback.
        It is not the preferred academic source but can help diagnose provider-data failures.
        """
        last_error = None

        for attempt in range(1, self.max_retries + 1):
            try:
                yf_ticker = yf.Ticker(ticker)
                info = yf_ticker.get_info() if hasattr(yf_ticker, "get_info") else yf_ticker.info
                asset_value, source_field = self._numeric_field(
                    info or {},
                    ["totalAssets", "netAssets"],
                )
                return {
                    "asset_value_usd": asset_value if asset_value is not None else np.nan,
                    "source_field": source_field,
                    "source": "Yahoo Finance via yfinance",
                    "source_url": f"https://finance.yahoo.com/quote/{urllib.parse.quote(ticker, safe='')}",
                    "as_of_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
                }
            except Exception as exc:
                last_error = exc
                if attempt < self.max_retries:
                    wait = self.retry_sleep_seconds * attempt
                    print(f"market AUM ({ticker} via yfinance): failed on attempt {attempt}/{self.max_retries}: {exc}")
                    print(f"Retrying after {wait} seconds...")
                    time.sleep(wait)

        raise RuntimeError(
            f"market AUM ({ticker} via yfinance): failed after {self.max_retries} attempts."
        ) from last_error

    def _download_yfinance_assets(self, tickers: list) -> pd.DataFrame:
        """
        Download ETF net assets for all assets through yfinance one ticker at a time.
        This function is retained as a fallback mode rather than the default AUM source.
        """
        rows = []
        for ticker in tickers:
            row = self._download_yfinance_asset(ticker)
            row["ticker"] = ticker
            rows.append(row)
        return pd.DataFrame(rows).set_index("ticker").reindex(tickers)

    def _download_yahoo_quote_summary_asset(self, ticker: str) -> dict:
        """
        Download one ETF's net assets from Yahoo Finance quoteSummary as another fallback.
        It parses Yahoo JSON fields when official or yfinance sources are unavailable.
        """
        encoded_ticker = urllib.parse.quote(ticker, safe="")
        url = (
            f"https://query1.finance.yahoo.com/v10/finance/quoteSummary/{encoded_ticker}"
            "?modules=summaryDetail"
        )
        raw = self._read_json_url(url, f"market AUM ({ticker} Yahoo quoteSummary)")
        results = raw.get("quoteSummary", {}).get("result") or []
        asset_value = None
        source_field = None
        if results:
            result = results[0]
            asset_value, source_field = self._numeric_field(
                result.get("summaryDetail", {}),
                ["totalAssets", "netAssets"],
            )
        return {
            "asset_value_usd": asset_value if asset_value is not None else np.nan,
            "source_field": source_field,
            "source": "Yahoo Finance quoteSummary",
            "source_url": url,
        }

    def get_market_aum(self) -> pd.DataFrame:
        """
        Select the configured ETF-AUM source and return a complete AUM DataFrame.
        The default is official provider data; file and Yahoo-style fallbacks are available for robustness.
        """
        if self.market_aum_source in {"official", "provider", "official_provider"}:
            try:
                return self._download_official_market_aum(self.asset_tickers)
            except Exception as live_error:
                cached_aum_path = self._cache_file_path("market_aum.csv")
                if cached_aum_path is not None:
                    print(
                        "market AUM: live official provider download failed "
                        f"({type(live_error).__name__}); using cached CSV fallback: {cached_aum_path}"
                    )
                    return self._read_market_aum_file(cached_aum_path)
                raise

        if self.market_aum_source in {"file", "csv"}:
            return self._read_market_aum_file(self.market_aum_file)

        if self.market_aum_source in {"yfinance", "yahoo"}:
            market_aum = self._download_yfinance_assets(self.asset_tickers)
            missing = market_aum.index[market_aum["asset_value_usd"].isna()].tolist()

            for ticker in missing:
                try:
                    fallback = self._download_yahoo_quote_summary_asset(ticker)
                except Exception as exc:
                    print(f"market AUM ({ticker} Yahoo quoteSummary fallback) failed: {exc}")
                    continue
                if pd.notna(fallback["asset_value_usd"]):
                    for key, value in fallback.items():
                        market_aum.loc[ticker, key] = value
                    market_aum.loc[ticker, "as_of_utc"] = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

            missing = market_aum.index[market_aum["asset_value_usd"].isna()].tolist()
            if missing:
                raise RuntimeError(
                    "market AUM: failed to obtain ETF assets for "
                    f"{missing}. Do not use static market weights."
                )
            return market_aum

        raise ValueError("market_aum_source must be 'official', 'file', or 'yfinance'.")

    def get_market_weights(self) -> pd.Series:
        """
        Convert ETF AUM values into normalised Black-Litterman market-equilibrium weights.
        These weights are the market portfolio proxy used in reverse optimisation.
        """
        market_aum = self.get_market_aum()
        weights = market_aum["asset_value_usd"] / market_aum["asset_value_usd"].sum()
        weights = weights.reindex(self.asset_tickers)
        weights.name = "market_weight"

        self.market_aum_snapshot = market_aum.copy()
        self.market_aum_snapshot["market_weight"] = weights
        return weights

    @staticmethod
    def save_data_to_csv(data: dict, output_dir="outputs/data") -> dict:
        """
        Save pandas DataFrame and Series entries from the loaded data dictionary to CSV files.
        This creates an auditable disk copy of prices, returns, excess returns, AUM and weights.
        """
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        saved_files = {}
        for name, value in data.items():
            csv_path = output_path / f"{name}.csv"
            if isinstance(value, pd.DataFrame):
                index_label = value.index.name or ("date" if isinstance(value.index, pd.DatetimeIndex) else "index")
                value.to_csv(csv_path, index_label=index_label, encoding=CSV_ENCODING)
            elif isinstance(value, pd.Series):
                index_label = value.index.name or ("date" if isinstance(value.index, pd.DatetimeIndex) else "index")
                if name == "rf_daily" and value.name == "DGS3MO":
                    csv_path = output_path / "rf_dgs3mo_daily.csv"
                value.to_frame(name=value.name or name).to_csv(csv_path, index_label=index_label, encoding=CSV_ENCODING)
            elif isinstance(value, dict):
                json_path = output_path / f"{name}.json"
                json_path.write_text(json.dumps(value, indent=2), encoding="utf-8")
                saved_files[name] = json_path
                continue
            else:
                continue
            saved_files[name] = csv_path

        return saved_files

    def load_all_data(self) -> dict:
        """
        Run the complete data pipeline for prices, returns, risk-free rates, AUM and market weights.
        It aligns all series to common dates and returns the objects required by the model and backtest.
        """
        # Asset data
        asset_prices = self.download_prices(self.asset_tickers)
        asset_returns = self.calc_simple_returns(asset_prices)
        
        # Factor data
        factor_symbols = list(self.factor_tickers.values())
        factor_prices = self.download_prices(factor_symbols)
        factor_prices.columns = list(self.factor_tickers.keys())
        factor_returns = self.calc_simple_returns(factor_prices)
        
        # risk-free interest rate
        rf_daily = self.get_risk_free_rate()
        
        # Align the dates
        common_dates = asset_returns.index.intersection(factor_returns.index).intersection(rf_daily.index)
        if common_dates.empty:
            raise RuntimeError(
                "No overlapping trading dates after data loading. Check whether price/risk-free downloads succeeded "
                "and whether the requested date range contains market data."
            )

        asset_returns = asset_returns.loc[common_dates]
        factor_returns = factor_returns.loc[common_dates]
        rf_daily = rf_daily.loc[common_dates]
        self.rf_metadata = dict(self.rf_metadata)
        self.rf_metadata["aligned_start"] = common_dates.min().date().isoformat()
        self.rf_metadata["aligned_end"] = common_dates.max().date().isoformat()
        fred_observed_end = pd.Timestamp(self.rf_metadata["fred_observed_end"])
        self.rf_metadata["terminal_forward_fill_dates"] = [
            date.date().isoformat() for date in common_dates if date > fred_observed_end
        ]
        
        # Calculate the excess return rate
        asset_excess = asset_returns.sub(rf_daily, axis=0)
        factor_excess = factor_returns.sub(rf_daily, axis=0)
        market_weights = self.get_market_weights()
        
        return {
            "asset_prices": asset_prices.loc[common_dates],
            "factor_prices": factor_prices.loc[common_dates],
            "asset_returns": asset_returns,
            "asset_excess": asset_excess,
            "factor_returns": factor_returns,
            "factor_excess": factor_excess,
            "rf_daily": rf_daily,
            "rf_metadata": self.rf_metadata,
            "market_weights": market_weights,
            "market_aum": self.market_aum_snapshot
        }

ProjectDataLoader = DataLoader


def load_audited_snapshot(snapshot_dir: Path) -> dict:
    """
    Validate the root SHA-256 manifest and then load the frozen snapshot inputs for the requested analysis window.
    """
    snapshot_dir = Path(snapshot_dir)
    snapshot_root = snapshot_dir.parent
    manifest_path = snapshot_root / "snapshot_manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing audited snapshot manifest: {manifest_path}")
    manifest_bytes = manifest_path.read_bytes()
    manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
    manifest = json.loads(manifest_bytes.decode("utf-8"))
    if manifest.get("status") != "complete":
        raise AssertionError("Snapshot manifest status must be complete.")
    if manifest.get("analysis_cutoff") != "2026-07-24":
        raise AssertionError("Snapshot manifest analysis cut-off does not match the report.")
    if manifest.get("expected_market_end") != "2026-07-23":
        raise AssertionError("Snapshot manifest does not require the 2026-07-23 complete market close.")
    if manifest.get("fred_observed_end") != "2026-07-22":
        raise AssertionError("Snapshot manifest does not record the 2026-07-22 FRED observation end.")
    if manifest.get("risk_free_source") != "FRED DGS3MO":
        raise AssertionError("Snapshot manifest must identify FRED DGS3MO.")
    manifest_files = {entry["path"]: entry for entry in manifest.get("files", [])}

    def verify_manifest_file(path: Path) -> None:
        """
        Verify that one snapshot file matches the byte size and SHA-256 digest declared in the manifest.
        """
        relative_path = path.relative_to(snapshot_root).as_posix()
        entry = manifest_files.get(relative_path)
        if entry is None:
            raise AssertionError(f"Snapshot file is absent from manifest: {relative_path}")
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest != entry.get("sha256") or path.stat().st_size != entry.get("bytes"):
            raise AssertionError(f"Snapshot file failed size/SHA-256 validation: {relative_path}")

    frame_names = [
        "asset_prices", "factor_prices", "asset_returns", "asset_excess",
        "factor_returns", "factor_excess",
    ]
    bundle = {}
    for name in frame_names:
        path = snapshot_dir / f"{name}.csv"
        if not path.exists():
            raise FileNotFoundError(f"Missing audited snapshot input: {path}")
        verify_manifest_file(path)
        frame = pd.read_csv(path, index_col=0, parse_dates=True, encoding=CSV_ENCODING)
        frame.index = pd.DatetimeIndex(frame.index).tz_localize(None).normalize()
        bundle[name] = frame.apply(pd.to_numeric, errors="raise").sort_index()

    rf_path = snapshot_dir / "rf_dgs3mo_daily.csv"
    rf_annual_path = snapshot_dir / "rf_dgs3mo_annual_percent.csv"
    rf_metadata_path = snapshot_dir / "rf_metadata.json"
    weights_path = snapshot_dir / "market_weights.csv"
    aum_path = snapshot_dir / "market_aum.csv"
    for path in [rf_path, rf_annual_path, rf_metadata_path, weights_path, aum_path]:
        if not path.exists():
            raise FileNotFoundError(f"Missing audited snapshot input: {path}")
        verify_manifest_file(path)

    rf_frame = pd.read_csv(rf_path, index_col=0, parse_dates=True, encoding=CSV_ENCODING)
    rf_frame.index = pd.DatetimeIndex(rf_frame.index).tz_localize(None).normalize()
    bundle["rf_daily"] = pd.to_numeric(rf_frame.iloc[:, 0], errors="raise").sort_index()
    bundle["rf_daily"].name = rf_frame.columns[0]
    rf_annual_frame = pd.read_csv(rf_annual_path, index_col=0, parse_dates=True, encoding=CSV_ENCODING)
    rf_annual_frame.index = pd.DatetimeIndex(rf_annual_frame.index).tz_localize(None).normalize()
    rf_annual_percent = pd.to_numeric(rf_annual_frame.iloc[:, 0], errors="raise").sort_index()
    bundle["rf_metadata"] = json.loads(rf_metadata_path.read_text(encoding="utf-8"))
    if bundle["rf_metadata"].get("source") != "FRED" or bundle["rf_metadata"].get("series_id") != "DGS3MO":
        raise AssertionError("Archived risk-free metadata must identify FRED DGS3MO.")
    required_rf_metadata = {
        "analysis_cutoff": "2026-07-24",
        "expected_market_end": "2026-07-23",
        "aligned_end": "2026-07-23",
        "fred_observed_end": "2026-07-22",
        "alignment_method": "asof_forward_fill_no_backfill",
        "terminal_forward_fill_dates": ["2026-07-23"],
        "daily_conversion": "DGS3MO / 100 / 252",
    }
    for metadata_key, expected_value in required_rf_metadata.items():
        if bundle["rf_metadata"].get(metadata_key) != expected_value:
            raise AssertionError(f"Risk-free metadata mismatch for {metadata_key}.")
    raw_fred_path = snapshot_root / "fred_DGS3MO_raw.csv"
    snapshot_metadata_path = snapshot_root / "snapshot_metadata.json"
    verify_manifest_file(raw_fred_path)
    verify_manifest_file(snapshot_metadata_path)
    bundle["snapshot_metadata"] = json.loads(snapshot_metadata_path.read_text(encoding="utf-8"))
    if bundle["snapshot_metadata"].get("expected_market_end") != "2026-07-23":
        raise AssertionError("Snapshot metadata market end does not match the report.")
    if bundle["snapshot_metadata"].get("fred_observed_end") != "2026-07-22":
        raise AssertionError("Snapshot metadata FRED observation end does not match the report.")
    if bundle["snapshot_metadata"].get("risk_free_alignment_method") != "asof_forward_fill_no_backfill":
        raise AssertionError("Snapshot metadata risk-free alignment method is incorrect.")
    raw_fred_bytes = raw_fred_path.read_bytes()
    raw_fred_sha256 = hashlib.sha256(raw_fred_bytes).hexdigest()
    if bundle["rf_metadata"].get("raw_download_sha256") != raw_fred_sha256:
        raise AssertionError("Risk-free metadata does not match the archived raw FRED payload.")
    if manifest.get("raw_fred_sha256") != raw_fred_sha256:
        raise AssertionError("Snapshot manifest does not match the archived raw FRED payload.")
    raw_fred = pd.read_csv(io.BytesIO(raw_fred_bytes))
    raw_date_column = "observation_date" if "observation_date" in raw_fred.columns else "DATE"
    if raw_date_column not in raw_fred.columns or "DGS3MO" not in raw_fred.columns:
        raise AssertionError("Archived raw FRED payload has unexpected columns.")
    raw_fred_series = pd.Series(
        pd.to_numeric(raw_fred["DGS3MO"].replace(".", np.nan), errors="coerce").to_numpy(),
        index=pd.to_datetime(raw_fred[raw_date_column], errors="coerce"),
        name="DGS3MO",
    ).dropna().sort_index()
    if (raw_fred_series.index > pd.Timestamp("2026-07-23")).any():
        raise AssertionError("Archived raw FRED payload contains a post-cut-off observation.")
    expected_fred_observed_end = pd.Timestamp(bundle["rf_metadata"]["fred_observed_end"])
    if raw_fred_series.empty or raw_fred_series.index.max() != expected_fred_observed_end:
        raise AssertionError("Archived raw FRED end does not match audited metadata.")
    if expected_fred_observed_end != pd.Timestamp("2026-07-22"):
        raise AssertionError("Archived raw FRED observations must end on 2026-07-22.")
    if bundle["rf_daily"].name != "DGS3MO":
        raise AssertionError("Archived risk-free column must be named DGS3MO.")

    weight_frame = pd.read_csv(weights_path, index_col=0, encoding=CSV_ENCODING)
    bundle["market_weights"] = pd.to_numeric(weight_frame.iloc[:, 0], errors="raise")
    bundle["market_weights"].name = "market_weight"
    bundle["market_aum"] = pd.read_csv(aum_path, index_col=0, encoding=CSV_ENCODING)

    common_index = bundle["asset_returns"].index
    if common_index.max() != pd.Timestamp("2026-07-23"):
        raise AssertionError("Snapshot returns must end exactly on 2026-07-23.")
    for name in ["asset_excess", "factor_returns", "factor_excess"]:
        if not bundle[name].index.equals(common_index):
            raise AssertionError(f"Snapshot date index mismatch: {name}")
    for price_name, return_name in [("asset_prices", "asset_returns"), ("factor_prices", "factor_returns")]:
        if not bundle[price_name].index.equals(common_index):
            raise AssertionError(f"Snapshot date index mismatch: {price_name}")
        reconstructed_returns = bundle[price_name].pct_change(fill_method=None)
        if not np.allclose(
            reconstructed_returns.iloc[1:].to_numpy(), bundle[return_name].iloc[1:].to_numpy(),
            atol=1e-12, rtol=1e-9, equal_nan=False,
        ):
            raise AssertionError(f"{return_name} do not reconcile to aligned prices from the second row.")
    if not bundle["rf_daily"].index.equals(common_index):
        raise AssertionError("Snapshot date index mismatch: rf_daily")
    if not rf_annual_percent.index.equals(common_index):
        raise AssertionError("Snapshot date index mismatch: rf_dgs3mo_annual_percent")
    expected_annual_percent = raw_fred_series.reindex(common_index, method="ffill")
    expected_terminal_fill_dates = [
        date.date().isoformat() for date in common_index if date > expected_fred_observed_end
    ]
    if bundle["rf_metadata"]["terminal_forward_fill_dates"] != expected_terminal_fill_dates:
        raise AssertionError("Archived terminal risk-free forward-fill dates are incorrect.")
    if expected_annual_percent.isna().any() or not np.allclose(
        rf_annual_percent.to_numpy(), expected_annual_percent.to_numpy(), atol=1e-14, rtol=0.0,
    ):
        raise AssertionError("Archived annual DGS3MO values do not reconcile to raw FRED data.")
    expected_rf_daily = expected_annual_percent / 100.0 / 252.0
    if not np.allclose(
        bundle["rf_daily"].to_numpy(), expected_rf_daily.to_numpy(), atol=1e-14, rtol=0.0,
    ):
        raise AssertionError("Archived daily risk-free rates are not DGS3MO / 100 / 252.")
    if not np.allclose(
        bundle["asset_returns"].sub(bundle["rf_daily"], axis=0),
        bundle["asset_excess"], atol=1e-12, rtol=1e-10,
    ):
        raise AssertionError("Archived asset excess returns do not reconcile to returns minus risk-free.")
    if not np.allclose(
        bundle["factor_returns"].sub(bundle["rf_daily"], axis=0),
        bundle["factor_excess"], atol=1e-12, rtol=1e-10,
    ):
        raise AssertionError("Archived factor excess returns do not reconcile to returns minus risk-free.")
    if not np.isclose(bundle["market_weights"].sum(), 1.0, atol=1e-10):
        raise AssertionError("Archived market weights do not sum to one.")
    bundle["snapshot_manifest_sha256"] = manifest_sha256
    bundle["snapshot_manifest_path"] = manifest_path.relative_to(PROJECT_ROOT).as_posix()
    return bundle


MAIN_LOOKBACK_YEARS = 3
ROBUSTNESS_LOOKBACK_YEARS = 5
ANALYSIS_AS_OF_DATE = pd.Timestamp("2026-07-24")
EXPECTED_COMPLETE_MARKET_DATE = pd.Timestamp("2026-07-23")
END_DATE = ANALYSIS_AS_OF_DATE
START_DATE = END_DATE - pd.DateOffset(years=MAIN_LOOKBACK_YEARS)
ROBUSTNESS_START_DATE = END_DATE - pd.DateOffset(years=ROBUSTNESS_LOOKBACK_YEARS)
START_DATE_STR = START_DATE.strftime("%Y-%m-%d")
ROBUSTNESS_START_DATE_STR = ROBUSTNESS_START_DATE.strftime("%Y-%m-%d")
END_DATE_STR = END_DATE.strftime("%Y-%m-%d")
ROBUSTNESS_DATA_OUTPUT_DIR = DATA_OUTPUT_DIR / "robustness_5y"
ROBUSTNESS_DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
USE_FROZEN_SNAPSHOT = True

print("=== Analysis provenance ===")
print(f"Run ID: {RUN_ID}")
print(f"Analysis cut-off: {END_DATE_STR}")
print(f"Requested main price window: [{START_DATE_STR}, {END_DATE_STR}]")
print(f"Requested robustness price window: [{ROBUSTNESS_START_DATE_STR}, {END_DATE_STR}]")

# Public execution uses local audited inputs only. Historical provider helpers above
# are retained as research source, but are not invoked by this execution path.
if not USE_FROZEN_SNAPSHOT:
    raise RuntimeError("This public notebook requires local audited snapshots; live downloads are disabled.")
print("Input mode: authorized local audited snapshot; see README for prerequisites")
data = load_audited_snapshot(FROZEN_MAIN_DATA_DIR)
robustness_data = load_audited_snapshot(FROZEN_ROBUSTNESS_DATA_DIR)

saved_data_files = ProjectDataLoader.save_data_to_csv(data, DATA_OUTPUT_DIR)
robustness_saved_data_files = ProjectDataLoader.save_data_to_csv(robustness_data, ROBUSTNESS_DATA_OUTPUT_DIR)
market_aum_dates = sorted(data["market_aum"]["as_of_date"].dropna().astype(str).unique())

main_start = data["asset_returns"].index.min()
main_end = data["asset_returns"].index.max()
robustness_start = robustness_data["asset_returns"].index.min()
robustness_end = robustness_data["asset_returns"].index.max()
if main_end != EXPECTED_COMPLETE_MARKET_DATE or robustness_end != EXPECTED_COMPLETE_MARKET_DATE:
    raise AssertionError("Both return panels must end on the latest complete market date, 2026-07-23.")
if data["snapshot_manifest_sha256"] != robustness_data["snapshot_manifest_sha256"]:
    raise AssertionError("Main and robustness windows were not loaded from the same snapshot manifest.")

provenance_table = pd.DataFrame([
    ["Run started UTC", RUN_STARTED_UTC.isoformat()],
    ["Fixed analysis cut-off", END_DATE_STR],
    ["Requested main window", f"{START_DATE_STR} to {END_DATE_STR}"],
    ["Actual aligned main returns", f"{main_start.date()} to {main_end.date()} ({len(data['asset_returns'])} observations)"],
    ["Requested robustness window", f"{ROBUSTNESS_START_DATE_STR} to {END_DATE_STR}"],
    ["Actual aligned robustness returns", f"{robustness_start.date()} to {robustness_end.date()} ({len(robustness_data['asset_returns'])} observations)"],
    ["AUM provider as-of dates", ", ".join(market_aum_dates)],
    ["AUM retrieval UTC", str(data["market_aum"]["as_of_utc"].max())],
    ["Archived price source", data.get("snapshot_metadata", {}).get("price_source", "Live loader source")],
    ["Price/return reconciliation", data.get("snapshot_metadata", {}).get("price_return_reconciliation", "Live calculation")],
    ["Risk-free source and series", f"{data['rf_metadata']['source']} {data['rf_metadata']['series_id']}"],
    ["Raw FRED observed dates", f"{data['rf_metadata']['fred_observed_start']} to {data['rf_metadata']['fred_observed_end']}"],
    ["Risk-free aligned market dates", f"{data['rf_metadata']['aligned_start']} to {data['rf_metadata']['aligned_end']}"],
    ["Risk-free alignment method", data['rf_metadata']['alignment_method']],
    ["Terminal risk-free carry-forward dates", ", ".join(data['rf_metadata']['terminal_forward_fill_dates'])],
    ["Risk-free daily conversion", data["rf_metadata"]["daily_conversion"]],
    ["Input mode", "Audited archived snapshot" if USE_FROZEN_SNAPSHOT else "Live with archived fallback"],
], columns=["Provenance field", "Value"])
provenance_table.to_csv(DATA_OUTPUT_DIR / "run_provenance.csv", index=False, encoding=CSV_ENCODING)

print("=== Data loading completed ===")
print(f"Fresh run data directory: {DATA_OUTPUT_DIR}")
display(provenance_table)
market_aum_display = data["market_aum"][["asset_value_usd", "market_weight", "as_of_date", "source"]].copy().astype(object)
market_aum_display["asset_value_usd"] = data["market_aum"]["asset_value_usd"].map(
    lambda value: format_display_value(value, style="currency_billions", na_label="Missing")
)
market_aum_display["market_weight"] = data["market_aum"]["market_weight"].map(
    lambda value: format_display_value(value, style="percent", na_label="Missing")
)
market_aum_display = market_aum_display.rename(columns={
    "asset_value_usd": "Official AUM",
    "market_weight": "AUM Weight",
    "as_of_date": "Provider As-of Date",
    "source": "Official Source",
})
display(market_aum_display)
print("Number of Assets: {}".format(len(data["asset_returns"].columns)))
mean_rf_daily = float(data["rf_daily"].mean())
print(
    "Mean daily risk-free rate: {}; simple annual equivalent: {}".format(
        format_display_value(mean_rf_daily, style="percent4"),
        format_display_value(mean_rf_daily * 252.0, style="percent"),
    )
)
print("Asset Pool: {}".format(", ".join(data["asset_returns"].columns)))


## Black-Litterman posterior


In [ ]:
# ===================== 2.2 Core implementation of the Black-Litterman model =====================
import numpy as np
import pandas as pd


class BlackLittermanModel:
    """
    Black-Litterman model class for combining market-equilibrium prior returns with investor views.
    The class uses excess returns, AUM-based market weights and an optional covariance override to compute the reverse-optimised prior, constructs absolute and relative view matrices, applies confidence-scaled view uncertainty, and returns posterior expected excess returns plus predictive covariance.
    Main methods: _build_view_matrices() converts view dictionaries into P, Q and confidence inputs; _validate_assets() checks ticker validity; _calc_omega_confidence() builds the view uncertainty matrix Omega; posterior() performs the Bayesian update and returns all reporting objects.
    The output is used by the optimiser to construct portfolios and by the report to compare prior versus posterior return assumptions.
    """

    def __init__(
        self,
        excess_returns: pd.DataFrame,
        market_weights: pd.Series,
        tau: float = 0.025,
        risk_aversion_market: float = 3.0,
        covariance_matrix: pd.DataFrame = None,
    ):
        """
        Initialise the Black-Litterman model using excess returns, market weights, tau and market risk aversion.
        It computes the historical covariance matrix, or uses a supplied covariance estimate, and then builds the reverse-optimised prior return vector.
        """
        self.assets = excess_returns.columns.tolist()
        self.n_assets = len(self.assets)

        if covariance_matrix is None:
            self.Sigma_df = excess_returns.cov().reindex(index=self.assets, columns=self.assets)
        else:
            self.Sigma_df = covariance_matrix.reindex(index=self.assets, columns=self.assets).astype(float)
            if self.Sigma_df.isna().any().any():
                raise ValueError("Covariance matrix is missing one or more model assets.")
        self.Sigma = self.Sigma_df.values.astype(float)
        self.Sigma = 0.5 * (self.Sigma + self.Sigma.T)

        weights = market_weights.reindex(self.assets).astype(float)
        if weights.isna().any():
            missing = weights[weights.isna()].index.tolist()
            raise ValueError(f"Missing market weights for assets: {missing}")
        if weights.sum() <= 0:
            raise ValueError("Market weights must have a positive sum.")
        weights = weights / weights.sum()

        self.w_mkt = weights.values.reshape(-1, 1)
        self.tau = float(tau)
        self.lambda_mkt = float(risk_aversion_market)
        self.pi = self.lambda_mkt * self.Sigma @ self.w_mkt

    def _build_view_matrices(self, views: list, weight_scheme: str = "aum_proxy") -> tuple:
        """
        Convert absolute and relative view definitions into the P matrix, Q vector and confidence list.
        Relative baskets can be weighted equally or by the ETF-AUM proxy.
        """
        n_views = len(views)
        P = np.zeros((n_views, self.n_assets))
        Q = np.zeros((n_views, 1))
        confidences = []
        market_weights = pd.Series(self.w_mkt.flatten(), index=self.assets)

        for i, view in enumerate(views):
            Q[i] = float(view["return"]) / 252.0
            confidences.append(float(view.get("confidence", 0.5)))

            if view["type"] == "absolute":
                asset = view["asset"]
                self._validate_assets([asset])
                P[i, self.assets.index(asset)] = 1.0

            elif view["type"] == "relative":
                assets_long = list(view["assets_long"])
                assets_short = list(view["assets_short"])
                self._validate_assets(assets_long + assets_short)

                if weight_scheme == "equal":
                    long_weights = pd.Series(1.0 / len(assets_long), index=assets_long)
                    short_weights = pd.Series(1.0 / len(assets_short), index=assets_short)
                elif weight_scheme == "aum_proxy":
                    long_weights = market_weights[assets_long] / market_weights[assets_long].sum()
                    short_weights = market_weights[assets_short] / market_weights[assets_short].sum()
                else:
                    raise ValueError("weight_scheme must be 'equal' or 'aum_proxy'.")

                for asset, weight in long_weights.items():
                    P[i, self.assets.index(asset)] = weight
                for asset, weight in short_weights.items():
                    P[i, self.assets.index(asset)] = -weight
            else:
                raise ValueError(f"Unsupported view type: {view['type']}")

        return P, Q, confidences

    def _validate_assets(self, assets: list) -> None:
        """
        Check that every asset referenced in a view exists in the model universe.
        It raises a clear error before matrix construction if a ticker is invalid.
        """
        missing = [asset for asset in assets if asset not in self.assets]
        if missing:
            raise ValueError(f"Unknown assets in view definition: {missing}")

    def _calc_omega_confidence(self, P: np.ndarray, confidences: list) -> np.ndarray:
        """
        Build the diagonal view-uncertainty matrix Omega using confidence-scaled view variance.
        Higher confidence produces lower view uncertainty and gives that view more influence.
        """
        omega_values = []
        for i, confidence in enumerate(confidences):
            confidence = float(np.clip(confidence, 1e-4, 0.9999))
            p = P[i].reshape(1, -1)
            view_variance = (p @ (self.tau * self.Sigma) @ p.T)[0, 0]
            view_variance = max(float(view_variance), 1e-12)
            omega_values.append(view_variance * (1.0 - confidence) / confidence)
        return np.diag(omega_values)

    def posterior(self, views: list, weight_scheme: str = "aum_proxy") -> dict:
        """
        Compute posterior expected excess returns and predictive covariance under Black-Litterman.
        It returns prior, posterior, view matrices and covariance components for reporting.
        """
        if len(views) == 0:
            mean_uncertainty = self.tau * self.Sigma
            predictive_cov = self.Sigma + mean_uncertainty
            predictive_cov = 0.5 * (predictive_cov + predictive_cov.T)
            return {
                "mu_post": pd.Series(self.pi.flatten(), index=self.assets),
                "Sigma_post": pd.DataFrame(predictive_cov, index=self.assets, columns=self.assets),
                "Sigma_mean_uncertainty": pd.DataFrame(mean_uncertainty, index=self.assets, columns=self.assets),
                "Sigma_historical": self.Sigma_df,
                "pi_prior": pd.Series(self.pi.flatten(), index=self.assets),
                "P_matrix": pd.DataFrame(columns=self.assets, dtype=float),
                "Q_vector": pd.Series(dtype=float),
                "Omega_matrix": pd.DataFrame(dtype=float),
                "view_confidences": pd.Series(dtype=float),
                "linear_solve_relative_residual": 0.0,
            }

        P, Q, confidences = self._build_view_matrices(views, weight_scheme=weight_scheme)
        Omega = self._calc_omega_confidence(P, confidences)

        tau_sigma = self.tau * self.Sigma
        middle = P @ tau_sigma @ P.T + Omega
        innovation = Q - P @ self.pi
        solved_innovation = np.linalg.solve(middle, innovation)
        post_mu = self.pi + tau_sigma @ P.T @ solved_innovation
        solved_covariance = np.linalg.solve(middle, P @ tau_sigma)
        mean_uncertainty = tau_sigma - tau_sigma @ P.T @ solved_covariance
        mean_uncertainty = 0.5 * (mean_uncertainty + mean_uncertainty.T)
        solve_residual = np.linalg.norm(middle @ solved_innovation - innovation)
        solve_scale = max(np.linalg.norm(innovation), np.finfo(float).eps)
        linear_solve_relative_residual = float(solve_residual / solve_scale)
        predictive_cov = self.Sigma + mean_uncertainty
        predictive_cov = 0.5 * (predictive_cov + predictive_cov.T)

        return {
            "mu_post": pd.Series(post_mu.flatten(), index=self.assets),
            "Sigma_post": pd.DataFrame(predictive_cov, index=self.assets, columns=self.assets),
            "Sigma_mean_uncertainty": pd.DataFrame(mean_uncertainty, index=self.assets, columns=self.assets),
            "Sigma_historical": self.Sigma_df,
            "pi_prior": pd.Series(self.pi.flatten(), index=self.assets),
            "P_matrix": pd.DataFrame(P, columns=self.assets),
            "Q_vector": pd.Series(Q.flatten()),
            "Omega_matrix": pd.DataFrame(Omega),
            "view_confidences": pd.Series(confidences),
            "linear_solve_relative_residual": linear_solve_relative_residual,
        }


## Constrained portfolio allocation


In [ ]:
# ===================== 2.3 Portfolio optimisation module =====================
import numpy as np
import pandas as pd
from scipy.optimize import minimize

try:
    import cvxpy as cp
except ImportError:
    cp = None


class PortfolioOptimizer:
    """
    Portfolio optimisation class for constrained allocation using Black-Litterman posterior inputs.
    It works with expected excess returns and predictive covariance, enforcing long-only, fully invested and single-asset cap constraints.
    Main methods: mean_variance() solves utility maximisation for different risk-aversion levels; max_sharpe() solves the constrained maximum-Sharpe allocation; equal_risk_contribution() builds a risk-budgeting benchmark; calc_portfolio_stats() reports model-implied return, volatility, Sharpe and risk contributions.
    Helper methods define common constraints, project near-feasible weights back to bounds, clean solver output and provide SLSQP fallback when cvxpy or its solvers are unavailable.
    """

    def __init__(self, mu: pd.Series, Sigma: pd.DataFrame, w_min: float = 0.0, w_max: float = 0.2):
        """
        Initialise the optimiser with expected excess returns, covariance and weight bounds.
        The inputs are aligned by asset name and the covariance matrix is symmetrised for numerical stability.
        """
        self.assets = mu.index.tolist()
        self.n_assets = len(self.assets)
        self.mu = mu.reindex(self.assets).values.astype(float)
        self.Sigma = Sigma.reindex(index=self.assets, columns=self.assets).values.astype(float)
        self.Sigma = 0.5 * (self.Sigma + self.Sigma.T)
        self.w_min = float(w_min)
        self.w_max = float(w_max)
        self.last_solver_info = {}

    def _common_constraints(self, w) -> list:
        """
        Create the common long-only, fully invested and capped-weight constraints for cvxpy solvers.
        These constraints are shared by mean-variance allocation methods.
        """
        return [
            cp.sum(w) == 1.0,
            w >= self.w_min,
            w <= self.w_max,
        ]

    def _project_to_bounds(self, weights: np.ndarray, tolerance: float = 1e-10) -> np.ndarray:
        """
        Project near-feasible numerical weights back to the capped simplex.
        This corrects tiny solver tolerances while preserving full investment and asset bounds.
        """
        weights = np.asarray(weights, dtype=float).flatten()
        weights = np.clip(weights, self.w_min, self.w_max)

        for _ in range(100):
            diff = 1.0 - weights.sum()
            if abs(diff) <= tolerance:
                break

            if diff > 0:
                free = weights < self.w_max - tolerance
            else:
                free = weights > self.w_min + tolerance

            if not np.any(free):
                break

            weights[free] += diff / free.sum()
            weights = np.clip(weights, self.w_min, self.w_max)

        if abs(weights.sum() - 1.0) > 1e-7:
            raise ValueError(
                "Unable to project weights to constraints. "
                f"min={weights.min():.6f}, max={weights.max():.6f}, sum={weights.sum():.6f}"
            )
        return weights / weights.sum()

    def _clean_weights(self, weights: np.ndarray, tolerance: float = 1e-4) -> np.ndarray:
        """
        Validate optimiser output and clean small numerical violations of the constraints.
        Material violations raise an error; small floating-point deviations are projected back to feasibility.
        """
        weights = np.asarray(weights, dtype=float).flatten()
        if weights.shape[0] != self.n_assets or not np.all(np.isfinite(weights)):
            raise ValueError("Optimiser returned invalid weights.")

        material_violation = (
            weights.min() < self.w_min - tolerance
            or weights.max() > self.w_max + tolerance
            or abs(weights.sum() - 1.0) > tolerance
        )
        if material_violation:
            raise ValueError(
                "Optimiser returned weights outside constraints. "
                f"min={weights.min():.6f}, max={weights.max():.6f}, sum={weights.sum():.6f}"
            )

        return self._project_to_bounds(weights)

    def _solve_slsqp_utility(self, risk_aversion: float) -> np.ndarray:
        """
        Solve the mean-variance utility problem with scipy SLSQP as a fallback.
        This keeps the notebook usable if cvxpy or its solvers are unavailable.
        """
        def negative_utility(weights):
            """
            Internal SLSQP objective equal to the negative of mean-variance utility.
            Minimising this function is equivalent to maximising expected excess return minus risk penalty.
            """
            return -(weights @ self.mu - (risk_aversion / 2.0) * (weights @ self.Sigma @ weights))

        constraints = [{"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}]
        bounds = [(self.w_min, self.w_max)] * self.n_assets
        x0 = np.ones(self.n_assets) / self.n_assets

        result = minimize(
            negative_utility,
            x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 1000, "ftol": 1e-12},
        )

        if not result.success:
            raise ValueError(f"Mean-variance fallback optimisation failed: {result.message}")
        self.last_solver_info = {
            "method": "SciPy SLSQP",
            "status": str(result.message),
            "success": bool(result.success),
            "iterations": int(result.nit),
            "multistarts": 1,
            "successful_starts": 1 if result.success else 0,
            "objective_dispersion": np.nan,
        }
        return result.x

    def mean_variance(self, risk_aversion: float) -> pd.Series:
        """
        Compute a constrained mean-variance portfolio for a selected risk-aversion level.
        It tries available cvxpy solvers first and falls back to SLSQP if needed.
        """
        if cp is not None:
            try:
                w = cp.Variable(self.n_assets)
                sigma_psd = cp.psd_wrap(self.Sigma) if hasattr(cp, "psd_wrap") else self.Sigma
                objective = cp.Maximize(self.mu.T @ w - (risk_aversion / 2.0) * cp.quad_form(w, sigma_psd))
                prob = cp.Problem(objective, self._common_constraints(w))

                for solver in ["CLARABEL", "ECOS", "OSQP", "SCS"]:
                    if solver not in cp.installed_solvers():
                        continue
                    try:
                        prob.solve(solver=solver, eps=1e-8) if solver == "SCS" else prob.solve(solver=solver)
                        if prob.status == "optimal" and w.value is not None:
                            self.last_solver_info = {
                                "method": f"CVXPY/{solver}",
                                "status": str(prob.status),
                                "success": True,
                                "iterations": getattr(prob.solver_stats, "num_iters", None),
                                "multistarts": 1,
                                "successful_starts": 1,
                                "objective_dispersion": np.nan,
                            }
                            return pd.Series(self._clean_weights(w.value), index=self.assets)
                    except Exception:
                        continue
            except Exception:
                pass

        # SLSQP fallback avoids hard failure when a cvxpy solver is unavailable
        # in the notebook environment.
        fallback_weights = self._solve_slsqp_utility(risk_aversion)
        return pd.Series(self._clean_weights(fallback_weights), index=self.assets)

    def max_sharpe(self) -> pd.Series:
        """
        Compute the constrained maximum-Sharpe portfolio using scipy SLSQP.
        Because inputs are excess returns, the objective maximises excess return per unit volatility.
        """
        def negative_sharpe(weights):
            """
            Internal SLSQP objective equal to the negative portfolio Sharpe ratio.
            Minimising it is equivalent to maximising the model-implied Sharpe ratio.
            """
            port_return = weights @ self.mu
            port_vol = np.sqrt(weights @ self.Sigma @ weights)
            if port_vol <= 0:
                return 1e6
            return -port_return / port_vol

        constraints = [{"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}]
        bounds = [(self.w_min, self.w_max)] * self.n_assets
        rng = np.random.default_rng(42)
        starts = [np.ones(self.n_assets) / self.n_assets]
        starts.extend(self._project_to_bounds(rng.dirichlet(np.ones(self.n_assets))) for _ in range(4))
        successful = []
        for start in starts:
            result = minimize(
                negative_sharpe,
                start,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000, "ftol": 1e-12},
            )
            if result.success:
                successful.append(result)
        if not successful:
            raise ValueError("Max Sharpe optimisation failed for all deterministic starts.")
        result = min(successful, key=lambda item: item.fun)
        objective_values = np.array([item.fun for item in successful], dtype=float)
        self.last_solver_info = {
            "method": "SciPy SLSQP",
            "status": str(result.message),
            "success": True,
            "iterations": int(result.nit),
            "multistarts": len(starts),
            "successful_starts": len(successful),
            "objective_dispersion": (
                float(objective_values.max() - objective_values.min())
                if len(successful) > 1 else np.nan
            ),
        }
        return pd.Series(self._clean_weights(result.x), index=self.assets)

    def equal_risk_contribution(self) -> pd.Series:
        """
        Compute an Equal Risk Contribution portfolio under the same long-only capped constraints.
        It is used as a risk-budgeting benchmark rather than the main BL alpha allocation.
        """
        def risk_contribution_error(weights):
            """
            Internal objective measuring deviations from equal total risk contributions.
            SLSQP minimises this error to obtain the ERC allocation.
            """
            weights = weights.reshape(-1, 1)
            port_vol = np.sqrt(weights.T @ self.Sigma @ weights)[0, 0]
            if port_vol <= 0:
                return 1e6
            marginal_rc = (self.Sigma @ weights) / port_vol
            total_rc = weights * marginal_rc
            target_rc = port_vol / self.n_assets
            return np.sum((total_rc - target_rc) ** 2)

        constraints = [{"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}]
        bounds = [(self.w_min, self.w_max)] * self.n_assets
        rng = np.random.default_rng(43)
        starts = [np.ones(self.n_assets) / self.n_assets]
        starts.extend(self._project_to_bounds(rng.dirichlet(np.ones(self.n_assets))) for _ in range(4))
        successful = []
        for start in starts:
            result = minimize(
                risk_contribution_error,
                start,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000, "ftol": 1e-12},
            )
            if result.success:
                successful.append(result)
        if not successful:
            raise ValueError("ERC optimisation failed for all deterministic starts.")
        result = min(successful, key=lambda item: item.fun)
        objective_values = np.array([item.fun for item in successful], dtype=float)
        self.last_solver_info = {
            "method": "SciPy SLSQP",
            "status": str(result.message),
            "success": True,
            "iterations": int(result.nit),
            "multistarts": len(starts),
            "successful_starts": len(successful),
            "objective_dispersion": (
                float(objective_values.max() - objective_values.min())
                if len(successful) > 1 else np.nan
            ),
        }
        return pd.Series(self._clean_weights(result.x), index=self.assets)

    def calc_portfolio_stats(self, weights: pd.Series, rf_daily: float = None) -> dict:
        """
        Calculate model-implied return, volatility, Sharpe ratio and risk contributions for a weight vector.
        The Sharpe calculation does not subtract the risk-free rate again because mu is already excess return.
        """
        weights = weights.reindex(self.assets).values.astype(float)
        port_return = weights @ self.mu
        port_vol = np.sqrt(weights @ self.Sigma @ weights)
        if port_vol <= 0:
            raise ValueError("Portfolio volatility must be positive.")

        sharpe = port_return / port_vol
        marginal_rc = self.Sigma @ weights
        total_rc = weights * marginal_rc
        risk_contribution = pd.Series(total_rc / total_rc.sum(), index=self.assets)

        return {
            "expected_return_daily": port_return,
            "volatility_daily": port_vol,
            "sharpe_daily": sharpe,
            "expected_return_annual": port_return * 252,
            "volatility_annual": port_vol * np.sqrt(252),
            "sharpe_annual": sharpe * np.sqrt(252),
            "risk_contribution": risk_contribution,
        }


## Performance calculations


In [ ]:

# ===================== 2.4 Backtest and factor-analysis helpers =====================
def calculate_performance_metrics_from_returns(return_series, rf_daily=0.0, periods_per_year=252):
    """
    Align daily portfolio returns and risk-free rates, then compute cumulative return, CAGR, excess return, volatility, Sharpe, Sortino, drawdown, Calmar and observation count.
    """
    returns = pd.Series(return_series, dtype=float).rename("return").dropna()
    if np.isscalar(rf_daily):
        risk_free = pd.Series(float(rf_daily), index=returns.index, name="rf")
    else:
        risk_free = pd.Series(rf_daily, dtype=float).rename("rf").reindex(returns.index).ffill()

    panel = pd.concat([returns, risk_free], axis=1).dropna()
    if len(panel) < 2:
        raise ValueError("At least two aligned return observations are required.")
    if (1.0 + panel["return"] <= 0.0).any():
        raise ValueError("Daily returns must be greater than -100%.")

    returns = panel["return"]
    excess = panel["return"] - panel["rf"]
    wealth = (1.0 + returns).cumprod()
    cumulative_return = wealth.iloc[-1] - 1.0
    cagr = wealth.iloc[-1] ** (periods_per_year / len(returns)) - 1.0
    annual_volatility = returns.std(ddof=1) * np.sqrt(periods_per_year)
    excess_volatility = excess.std(ddof=1)
    sharpe = excess.mean() / excess_volatility * np.sqrt(periods_per_year) if excess_volatility > 0 else np.nan

    downside = np.minimum(excess.to_numpy(), 0.0)
    downside_deviation = np.sqrt(np.mean(downside ** 2)) * np.sqrt(periods_per_year)
    annual_excess_return = excess.mean() * periods_per_year
    sortino = annual_excess_return / downside_deviation if downside_deviation > 0 else np.nan

    running_peak = np.maximum.accumulate(np.r_[1.0, wealth.to_numpy()])[1:]
    drawdown = wealth.to_numpy() / running_peak - 1.0
    maximum_drawdown = float(np.min(drawdown))
    calmar = cagr / abs(maximum_drawdown) if maximum_drawdown < 0 else np.nan

    return pd.Series({
        "Cumulative Return": cumulative_return,
        "CAGR": cagr,
        "Annualized Excess Return": annual_excess_return,
        "Annualized Volatility": annual_volatility,
        "Annualized Sharpe Ratio": sharpe,
        "Annualized Sortino Ratio": sortino,
        "Maximum Drawdown": maximum_drawdown,
        "Calmar Ratio": calmar,
        "Observations": int(len(returns)),
    })


class BacktestEngine:
    """
    Buy-and-hold backtest engine with daily weight drift after the initial allocation.
    """

    def __init__(self, initial_weights: pd.Series, asset_returns: pd.DataFrame):
        """
        Validate and normalize the initial asset weights and store the matching daily simple-return panel.
        """
        assets = asset_returns.columns
        weights = pd.Series(initial_weights, dtype=float).reindex(assets)
        if weights.isna().any() or weights.sum() <= 0:
            raise ValueError("Initial weights must cover every asset and have positive total weight.")
        self.initial_weights = weights / weights.sum()
        self.asset_returns = asset_returns.astype(float)

    def calc_portfolio_returns(self) -> pd.Series:
        """
        Compute buy-and-hold daily portfolio returns while allowing asset weights to drift after each close.
        """
        portfolio_returns = pd.Series(index=self.asset_returns.index, dtype=float)
        current_weights = self.initial_weights.copy()
        for date in self.asset_returns.index:
            daily_return = float(current_weights @ self.asset_returns.loc[date])
            portfolio_returns.loc[date] = daily_return
            current_weights = current_weights * (1.0 + self.asset_returns.loc[date])
            current_weights = current_weights / current_weights.sum()
        return portfolio_returns

    def calc_performance_metrics(self, port_returns: pd.Series, rf_daily=0.0) -> pd.Series:
        """
        Delegate to the shared performance routine to compute the standard 252-trading-day annualized metrics.
        """
        return calculate_performance_metrics_from_returns(port_returns, rf_daily)

    def plot_performance(self, port_returns: pd.Series, benchmark_returns=None):
        """
        Plot portfolio cumulative wealth and drawdown and optionally overlay date-aligned benchmark wealth series.
        """
        cumulative_portfolio = (1.0 + port_returns).cumprod()
        running_max = np.maximum.accumulate(np.r_[1.0, cumulative_portfolio.to_numpy()])[1:]
        drawdown = cumulative_portfolio.to_numpy() / running_max - 1.0
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
        )
        ax1.plot(cumulative_portfolio, label="Views-On BL", linewidth=1.6)
        if benchmark_returns is not None:
            benchmark_frame = benchmark_returns.to_frame() if isinstance(benchmark_returns, pd.Series) else benchmark_returns
            for column in benchmark_frame.columns:
                ax1.plot((1.0 + benchmark_frame[column]).cumprod(), label=str(column), linewidth=1.2, linestyle="--")
        sample_end = pd.Timestamp(port_returns.index.max()).date().isoformat()
        ax1.set_title(f"Retrospective Main-Sample Cumulative Wealth (through {sample_end})")
        ax1.legend(ncol=2)
        ax1.grid(alpha=0.3)
        ax2.fill_between(port_returns.index, drawdown, 0, color="#b23a48", alpha=0.35)
        ax2.set_title("Views-On BL Drawdown")
        ax2.set_xlim(port_returns.index.min(), port_returns.index.max())
        ax2.grid(alpha=0.3)
        plt.tight_layout()
        return fig


class FactorAnalyzer:
    """
    HAC-robust regression and rolling-beta diagnostics for aligned return series.
    """

    def __init__(self, portfolio_excess_returns: pd.Series, factor_returns: pd.DataFrame):
        """
        Store same-frequency portfolio excess returns and candidate factor returns for aligned regression analysis.
        """
        self.port_ret = pd.Series(portfolio_excess_returns, dtype=float)
        self.factor_ret = pd.DataFrame(factor_returns, dtype=float)

    def factor_regression(self, factor_names=None, hac_lags=5) -> pd.DataFrame:
        """
        Fit a date-aligned multifactor OLS regression with HAC standard errors and return coefficient estimates, confidence intervals and model diagnostics.
        """
        factor_names = factor_names or self.factor_ret.columns.tolist()
        panel = pd.concat([self.port_ret.rename("Portfolio"), self.factor_ret[factor_names]], axis=1).dropna()
        design = sm.add_constant(panel[factor_names], has_constant="add")
        model = sm.OLS(panel["Portfolio"], design).fit(
            cov_type="HAC", cov_kwds={"maxlags": int(hac_lags)}
        )
        confidence_interval = model.conf_int(alpha=0.05)
        result = pd.DataFrame({
            "Coefficient": model.params,
            "HAC Standard Error": model.bse,
            "HAC t-statistic": model.tvalues,
            "HAC p-value": model.pvalues,
            "HAC 95% CI Lower": confidence_interval[0],
            "HAC 95% CI Upper": confidence_interval[1],
        })
        result.attrs["model_diagnostics"] = pd.Series({
            "R-squared": model.rsquared,
            "Adjusted R-squared": model.rsquared_adj,
            "Standardized Factor Condition Number": float(np.linalg.cond(
                ((design[factor_names] - design[factor_names].mean()) / design[factor_names].std(ddof=0)).to_numpy()
            )),
            "HAC Lags": int(hac_lags),
            "Observations": int(model.nobs),
        })
        return result

    def rolling_beta(self, factor_name: str, window=252) -> pd.Series:
        """
        Estimate the portfolio's single-factor OLS beta in every complete trailing window and return the dated beta series.
        """
        panel = pd.concat([
            self.port_ret.rename("Portfolio"),
            self.factor_ret[factor_name].rename(factor_name),
        ], axis=1).dropna()
        rolling_betas = pd.Series(index=panel.index, dtype=float)
        for end in range(window, len(panel) + 1):
            sample = panel.iloc[end - window:end]
            model = sm.OLS(sample["Portfolio"], sm.add_constant(sample[[factor_name]], has_constant="add")).fit()
            rolling_betas.iloc[end - 1] = model.params[factor_name]
        return rolling_betas.dropna()


## Descriptive statistics


In [ ]:
# ===================== 3.1 Descriptive Statistics =====================
def descriptive_max_drawdown(return_series):
    """
    Compute the most negative peak-to-trough drawdown from non-empty simple returns, including initial wealth of one.
    """
    wealth = (1.0 + pd.Series(return_series, dtype=float).dropna()).cumprod().to_numpy()
    peaks = np.maximum.accumulate(np.r_[1.0, wealth])[1:]
    return float(np.min(wealth / peaks - 1.0))


asset_excess_for_stats = data["asset_excess"].reindex(data["asset_returns"].index)
desc_stats = pd.DataFrame({
    "Annualized Arithmetic Mean": data["asset_returns"].mean() * 252.0,
    "Annualized Volatility": data["asset_returns"].std(ddof=1) * np.sqrt(252.0),
    "Annualized Excess-Return Sharpe": (
        asset_excess_for_stats.mean() / asset_excess_for_stats.std(ddof=1)
    ) * np.sqrt(252.0),
    "Maximum Drawdown": data["asset_returns"].apply(descriptive_max_drawdown),
})
desc_stats.to_csv(DATA_OUTPUT_DIR / "asset_descriptive_statistics.csv", encoding=CSV_ENCODING)

desc_stats_display = desc_stats.copy().astype(object)
for column in [
    "Annualized Arithmetic Mean",
    "Annualized Volatility",
    "Maximum Drawdown",
]:
    desc_stats_display[column] = desc_stats[column].map(
        lambda value: format_display_value(value, style="percent", na_label="Not estimated")
    )
desc_stats_display["Annualized Excess-Return Sharpe"] = (
    desc_stats["Annualized Excess-Return Sharpe"].map(
        lambda value: format_display_value(value, style="ratio", na_label="Not estimated")
    )
)

print("=== Asset Descriptive Statistics ===")
display(desc_stats_display)


## Asset and factor diagnostics


In [ ]:
# ===================== 3.2 Asset universe diagnostics =====================
asset_corr = data["asset_returns"].corr()
asset_corr.to_csv(DATA_OUTPUT_DIR / "asset_correlation_matrix.csv", encoding=CSV_ENCODING)

plt.figure(figsize=(10, 8))
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message=r"The set_bad function will be deprecated.*",
        category=PendingDeprecationWarning,
    )
    sns.heatmap(asset_corr, cmap="vlag", center=0, annot=False, square=True, linewidths=0.2)
plt.title("Asset Return Correlation Matrix")
plt.tight_layout()
plt.savefig(FIGURE_OUTPUT_DIR / "asset_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

upper_corr = asset_corr.where(np.triu(np.ones(asset_corr.shape), k=1).astype(bool))
diversification_summary = pd.Series({
    "Average Pairwise Correlation": upper_corr.stack().mean(),
    "Average Absolute Pairwise Correlation": upper_corr.abs().stack().mean(),
    "Maximum Pairwise Correlation": upper_corr.stack().max(),
    "Minimum Pairwise Correlation": upper_corr.stack().min(),
    "Number of Assets": len(asset_corr)
})

ASSET_SCREEN_HAC_LAGS = 5
factor_proxy_panel = data["factor_excess"][["Market", "ESG", "Momentum"]].dropna()
orthogonal_proxy_panel = pd.DataFrame(index=factor_proxy_panel.index)
orthogonal_proxy_panel["Market"] = factor_proxy_panel["Market"]
proxy_market_rows = []

for proxy_name, residual_name, display_name in [
    ("ESG", "ESG Residual", "ESGU excess"),
    ("Momentum", "Momentum Residual", "MTUM excess"),
]:
    proxy_design = sm.add_constant(factor_proxy_panel[["Market"]], has_constant="add")
    proxy_model = sm.OLS(factor_proxy_panel[proxy_name], proxy_design).fit(
        cov_type="HAC", cov_kwds={"maxlags": ASSET_SCREEN_HAC_LAGS}
    )
    orthogonal_proxy_panel[residual_name] = proxy_model.resid
    proxy_market_rows.append({
        "Proxy": display_name,
        "Market Beta": proxy_model.params["Market"],
        "Market-only R-squared": proxy_model.rsquared,
        "Residual-Market Correlation": proxy_model.resid.corr(factor_proxy_panel["Market"]),
    })

proxy_market_summary = pd.DataFrame(proxy_market_rows).set_index("Proxy")
proxy_market_summary.to_csv(DATA_OUTPUT_DIR / "factor_proxy_market_only_diagnostics.csv", encoding=CSV_ENCODING)

asset_factor_rows = []
for asset in data["asset_excess"].columns:
    regression_panel = pd.concat([
        data["asset_excess"][asset].rename("Asset Excess"),
        orthogonal_proxy_panel,
    ], axis=1).dropna()

    market_design = sm.add_constant(regression_panel[["Market"]], has_constant="add")
    market_model = sm.OLS(regression_panel["Asset Excess"], market_design).fit(
        cov_type="HAC", cov_kwds={"maxlags": ASSET_SCREEN_HAC_LAGS}
    )

    orthogonal_design = sm.add_constant(
        regression_panel[["Market", "ESG Residual", "Momentum Residual"]],
        has_constant="add",
    )
    orthogonal_model = sm.OLS(regression_panel["Asset Excess"], orthogonal_design).fit(
        cov_type="HAC", cov_kwds={"maxlags": ASSET_SCREEN_HAC_LAGS}
    )
    confidence_interval = orthogonal_model.conf_int(alpha=0.05)

    asset_factor_rows.append({
        "Asset": asset,
        "Annualized Volatility": data["asset_returns"][asset].std(ddof=1) * np.sqrt(252.0),
        "Market-only Beta": market_model.params["Market"],
        "Market-only R-squared": market_model.rsquared,
        "ESG Residual HAC Beta": orthogonal_model.params["ESG Residual"],
        "ESG Residual HAC 95% CI Lower": confidence_interval.loc["ESG Residual", 0],
        "ESG Residual HAC 95% CI Upper": confidence_interval.loc["ESG Residual", 1],
        "ESG Residual HAC p-value": orthogonal_model.pvalues["ESG Residual"],
        "Momentum Residual HAC Beta": orthogonal_model.params["Momentum Residual"],
        "Momentum Residual HAC 95% CI Lower": confidence_interval.loc["Momentum Residual", 0],
        "Momentum Residual HAC 95% CI Upper": confidence_interval.loc["Momentum Residual", 1],
        "Momentum Residual HAC p-value": orthogonal_model.pvalues["Momentum Residual"],
        "Observations": int(orthogonal_model.nobs),
    })

asset_factor_beta_screen = pd.DataFrame(asset_factor_rows).set_index("Asset")
asset_factor_beta_screen.to_csv(
    DATA_OUTPUT_DIR / "asset_orthogonalized_factor_beta_screen.csv", encoding=CSV_ENCODING
)

def format_asset_factor_beta_screen(df):
    """
    Format asset and proxy diagnostics without altering their full-precision CSV exports.
    Numerically orthogonal residual correlations, missing estimates and infinite estimates are labelled explicitly.
    """
    formatted = df.copy().astype(object)
    for column in formatted.columns:
        column_label = str(column)
        if "p-value" in column_label:
            formatted[column] = df[column].map(
                lambda value: format_display_value(value, style="pvalue", na_label="Not estimated")
            )
        elif "Volatility" in column_label:
            formatted[column] = df[column].map(
                lambda value: format_display_value(value, style="percent", na_label="Not estimated")
            )
        elif "Observations" in column_label:
            formatted[column] = df[column].map(
                lambda value: format_display_value(value, style="integer", na_label="Not estimated")
            )
        elif "Residual-Market Correlation" in column_label:
            formatted[column] = df[column].map(
                lambda value: (
                    "0 (within numerical precision)"
                    if not pd.isna(value) and np.isfinite(float(value)) and abs(float(value)) < 1e-12
                    else format_display_value(value, style="decimal4", na_label="Not estimated")
                )
            )
        else:
            formatted[column] = df[column].map(
                lambda value: format_display_value(value, style="decimal4", na_label="Not estimated")
            )
    return formatted

asset_market_screen = asset_factor_beta_screen[[
    "Annualized Volatility",
    "Market-only Beta",
    "Market-only R-squared",
]]
asset_residual_screen = asset_factor_beta_screen[[
    "ESG Residual HAC Beta",
    "ESG Residual HAC 95% CI Lower",
    "ESG Residual HAC 95% CI Upper",
    "ESG Residual HAC p-value",
    "Momentum Residual HAC Beta",
    "Momentum Residual HAC 95% CI Lower",
    "Momentum Residual HAC 95% CI Upper",
    "Momentum Residual HAC p-value",
]]

ROLLING_PROXY_WINDOW = 252
rolling_proxy_diagnostics = pd.DataFrame(index=factor_proxy_panel.index)
rolling_market_variance = factor_proxy_panel["Market"].rolling(
    ROLLING_PROXY_WINDOW, min_periods=ROLLING_PROXY_WINDOW
).var()
for proxy_name, ticker_name in [("ESG", "ESGU"), ("Momentum", "MTUM")]:
    rolling_proxy_diagnostics[f"{ticker_name}-on-SPY Rolling Beta"] = (
        factor_proxy_panel[proxy_name].rolling(
            ROLLING_PROXY_WINDOW, min_periods=ROLLING_PROXY_WINDOW
        ).cov(factor_proxy_panel["Market"]) / rolling_market_variance
    )
    rolling_proxy_diagnostics[f"{ticker_name}-on-SPY Rolling R-squared"] = (
        factor_proxy_panel[proxy_name].rolling(
            ROLLING_PROXY_WINDOW, min_periods=ROLLING_PROXY_WINDOW
        ).corr(factor_proxy_panel["Market"]).pow(2)
    )

rolling_proxy_diagnostics = rolling_proxy_diagnostics.dropna()
rolling_proxy_diagnostics.to_csv(
    DATA_OUTPUT_DIR / "rolling_252d_proxy_market_beta_r2.csv", index_label="Date", encoding=CSV_ENCODING
)

print("=== Diversification Summary ===")
diversification_summary_display = diversification_summary.copy().astype(object)
for label in diversification_summary.index:
    style = "integer" if label == "Number of Assets" else "decimal4"
    diversification_summary_display.loc[label] = format_display_value(
        diversification_summary.loc[label], style=style, na_label="Not estimated"
    )
display(diversification_summary_display)

print("=== Proxy-on-market diagnostics before orthogonalization ===")
display(format_asset_factor_beta_screen(proxy_market_summary))

print("=== Asset market-only beta and R-squared ===")
display(format_asset_factor_beta_screen(asset_market_screen))

print("=== Asset exposure to market-orthogonal proxy residuals (HAC) ===")
display(format_asset_factor_beta_screen(asset_residual_screen))

if len(rolling_proxy_diagnostics) > 0:
    rolling_proxy_summary = pd.DataFrame({
        "Latest": rolling_proxy_diagnostics.iloc[-1],
        "Median": rolling_proxy_diagnostics.median(),
        "Minimum": rolling_proxy_diagnostics.min(),
        "Maximum": rolling_proxy_diagnostics.max(),
    })
    rolling_proxy_summary.to_csv(
        DATA_OUTPUT_DIR / "rolling_252d_proxy_market_beta_r2_summary.csv", encoding=CSV_ENCODING
    )
    print("=== Rolling 252-day ESGU/MTUM-on-SPY beta and R-squared ===")
    display(format_asset_factor_beta_screen(rolling_proxy_summary))

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    for ticker_name, color in [("ESGU", "#0072B2"), ("MTUM", "#D55E00")]:
        axes[0].plot(
            rolling_proxy_diagnostics.index,
            rolling_proxy_diagnostics[f"{ticker_name}-on-SPY Rolling Beta"],
            label=ticker_name,
            color=color,
            linewidth=1.3,
        )
        axes[1].plot(
            rolling_proxy_diagnostics.index,
            rolling_proxy_diagnostics[f"{ticker_name}-on-SPY Rolling R-squared"],
            label=ticker_name,
            color=color,
            linewidth=1.3,
        )
    axes[0].set_title("Rolling 252-Day Proxy Beta to SPY Excess Return")
    axes[0].set_ylabel("Beta")
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    axes[1].set_title("Rolling 252-Day Proxy R-squared to SPY Excess Return")
    axes[1].set_ylabel("R-squared")
    axes[1].set_ylim(0.0, 1.02)
    axes[1].legend()
    axes[1].grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIGURE_OUTPUT_DIR / "rolling_252d_proxy_market_beta_r2.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Rolling proxy diagnostics skipped: fewer than 252 aligned observations.")


## Sector scenario exploration


In [ ]:
# ===================== 3.3.1 Factor Value Assessment =====================
from statsmodels.stats.multitest import multipletests
from scipy import stats

FACTOR_LONG_ASSETS = ["XLK", "XLU", "XLV"]
FACTOR_SHORT_ASSETS = ["XLE", "XLB", "XLI"]
FACTOR_ONE_WAY_COST = 0.001

def build_normalized_leg_weights(assets, market_weights):
    """
    Build normalized weights for one side of the long-short factor from official AUM proxy weights.
    If an AUM value is unavailable, the function falls back to equal weights for transparent reproducibility.
    """
    weights = market_weights.reindex(assets).astype(float)
    if weights.isna().any() or weights.sum() <= 0:
        weights = pd.Series(1.0, index=assets, dtype=float)
    return weights / weights.sum()

def build_weighted_leg_return(asset_returns, assets, weights):
    """
    Calculate the return series of one weighted basket leg.
    It aligns assets and weights explicitly so the factor construction is auditable.
    """
    aligned_weights = weights.reindex(assets).astype(float)
    aligned_weights = aligned_weights / aligned_weights.sum()
    return asset_returns[assets].mul(aligned_weights, axis=1).sum(axis=1)

def calculate_factor_max_drawdown(return_series):
    """
    Calculate maximum drawdown for a factor or basket return series.
    The statistic is included because factor value should be judged by path risk, not only average return.
    """
    clean_returns = pd.Series(return_series, dtype=float).dropna()
    if clean_returns.empty:
        return np.nan
    wealth = np.r_[1.0, (1.0 + clean_returns).cumprod().to_numpy(dtype=float)]
    running_peak = np.maximum.accumulate(wealth)
    return float(np.min(wealth / running_peak - 1.0))

def calculate_factor_value_metrics(return_series, rf_daily=0.0):
    """
    Report CAGR and annualized arithmetic mean separately. Annualized Sharpe uses the
    daily excess-return mean divided by its daily standard deviation, multiplied by sqrt(252).
    Long-only baskets use the date-aligned daily FRED rate; zero-investment spreads use zero.
    """
    clean_returns = pd.Series(return_series, dtype=float).dropna()
    if len(clean_returns) == 0:
        return pd.Series(dtype=float)
    if np.isscalar(rf_daily):
        aligned_rf = pd.Series(float(rf_daily), index=clean_returns.index, dtype=float)
    else:
        aligned_rf = pd.Series(rf_daily, dtype=float).reindex(clean_returns.index)
        if aligned_rf.isna().any():
            missing_date = aligned_rf.index[aligned_rf.isna()][0]
            raise ValueError(f'Risk-free series is missing the factor-return date {missing_date.date()}.')

    cumulative_return = (1.0 + clean_returns).prod() - 1.0
    if 1.0 + cumulative_return > 0:
        cagr = (1.0 + cumulative_return) ** (252.0 / len(clean_returns)) - 1.0
    else:
        cagr = np.nan
    annualized_arithmetic_mean = clean_returns.mean() * 252.0
    daily_excess_returns = clean_returns - aligned_rf
    daily_excess_volatility = daily_excess_returns.std(ddof=1)
    annual_volatility = clean_returns.std(ddof=1) * np.sqrt(252.0)
    sharpe = (
        daily_excess_returns.mean() / daily_excess_volatility * np.sqrt(252.0)
        if daily_excess_volatility > 0 else np.nan
    )
    if len(clean_returns) > 1 and clean_returns.std() > 0:
        t_test = stats.ttest_1samp(clean_returns, popmean=0.0, nan_policy="omit")
        mean_t_stat = float(t_test.statistic)
        mean_p_value = float(t_test.pvalue)
    else:
        mean_t_stat = np.nan
        mean_p_value = np.nan

    return pd.Series({
        "Cumulative Return": cumulative_return,
        "CAGR": cagr,
        "Annualized Arithmetic Mean": annualized_arithmetic_mean,
        "Annualized Volatility": annual_volatility,
        "Sharpe Ratio": sharpe,
        "Maximum Drawdown": calculate_factor_max_drawdown(clean_returns),
        "Daily Hit Rate": (clean_returns > 0).mean(),
        "Daily Mean t-stat": mean_t_stat,
        "Daily Mean p-value": mean_p_value,
        "Skewness": clean_returns.skew(),
        "Excess Kurtosis": clean_returns.kurtosis(),
        "Observations": len(clean_returns)
    })

def run_factor_alpha_regression(target_returns, control_returns, label, hac_lags=5):
    """
    Regress the target factor on control factors and report annualized alpha with Newey-West/HAC standard errors.
    HAC inference is used because daily factor returns may be heteroskedastic and serially correlated.
    """
    regression_data = pd.concat(
        [target_returns.rename("Target"), control_returns],
        axis=1
    ).dropna()
    X = sm.add_constant(regression_data[control_returns.columns])
    model = sm.OLS(regression_data["Target"], X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    alpha_ci = model.conf_int(alpha=0.05).loc["const"] * 252.0
    result = {
        "Model": label,
        "Annualized Alpha": model.params["const"] * 252,
        "Annualized Alpha HAC 95% CI Lower": alpha_ci.iloc[0],
        "Annualized Alpha HAC 95% CI Upper": alpha_ci.iloc[1],
        "Alpha t-stat": model.tvalues["const"],
        "Alpha p-value": model.pvalues["const"],
        "Adjusted R-squared": model.rsquared_adj,
        "Inference": "Newey-West HAC",
        "HAC Lags": hac_lags,
        "Observations": int(model.nobs)
    }
    for factor_name in control_returns.columns:
        result[f"{factor_name} Beta"] = model.params.get(factor_name, np.nan)
        result[f"{factor_name} p-value"] = model.pvalues.get(factor_name, np.nan)
    return pd.Series(result)

def calculate_rolling_factor_diagnostics(spread_returns, control_returns, window=252):
    """
    Calculate rolling Sharpe ratio and rolling market-plus-momentum alpha for the low-carbon spread.
    Rolling alpha p-values use HAC standard errors; diagnostics show whether factor value is stable or concentrated in one subperiod.
    """
    aligned = pd.concat([spread_returns.rename("Spread"), control_returns], axis=1).dropna()
    rows = []
    for end_position in range(window, len(aligned) + 1):
        sample = aligned.iloc[end_position - window:end_position]
        spread_sample = sample["Spread"]
        rolling_vol = spread_sample.std()
        rolling_sharpe = (
            spread_sample.mean() / rolling_vol * np.sqrt(252)
            if rolling_vol > 0 else np.nan
        )
        X = sm.add_constant(sample[control_returns.columns])
        model = sm.OLS(sample["Spread"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        rows.append({
            "Date": sample.index[-1],
            "Rolling Sharpe": rolling_sharpe,
            "Rolling Annualized Alpha": model.params["const"] * 252,
            "Rolling Alpha p-value": model.pvalues["const"],
            "Rolling Adjusted R-squared": model.rsquared_adj
        })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index("Date")

def run_rebalanced_factor_strategy(asset_returns, target_weights, one_way_cost=0.001):
    """
    Simulate a monthly rebalanced long-short factor return stream with one-way proportional transaction costs.
    The routine uses target gross exposure, realised daily returns and drifted weights to estimate turnover.
    """
    aligned_returns = asset_returns.reindex(columns=target_weights.index).dropna()
    target_weights = target_weights.reindex(aligned_returns.columns).astype(float)
    current_weights = pd.Series(0.0, index=target_weights.index)
    rebalance_dates = set(aligned_returns.groupby(aligned_returns.index.to_period("M")).head(1).index)
    rows = []

    for date, row in aligned_returns.iterrows():
        turnover = 0.0
        if date in rebalance_dates:
            turnover = (target_weights - current_weights).abs().sum()
            current_weights = target_weights.copy()

        gross_return = float((current_weights * row).sum())
        net_return = gross_return - turnover * one_way_cost
        denominator = 1.0 + gross_return
        if denominator > 0:
            current_weights = current_weights * (1.0 + row) / denominator
        else:
            current_weights = target_weights.copy()

        rows.append({
            "Date": date,
            "Gross Return": gross_return,
            "Net Return": net_return,
            "Turnover": turnover,
            "Cost": turnover * one_way_cost
        })

    return pd.DataFrame(rows).set_index("Date")

def format_factor_value_table(df, percent_rows=None, ratio_rows=None, integer_rows=None, decimal4_rows=None):
    """
    Format factor value tables for notebook display while preserving numeric CSV exports.
    It converts values to object dtype before inserting percentage strings and N/A labels.
    """
    percent_rows = percent_rows or []
    ratio_rows = ratio_rows or []
    integer_rows = integer_rows or []
    decimal4_rows = decimal4_rows or []
    formatted = df.copy().astype(object)
    for row in formatted.index:
        row_label = str(row)
        if row in percent_rows:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="percent"))
        elif row in integer_rows or "Observations" in row_label or "Lags" in row_label:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="integer"))
        elif "p-value" in row_label:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="pvalue"))
        elif row in decimal4_rows:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="decimal4"))
        elif row in ratio_rows:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="ratio"))
        else:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="ratio") if pd.isna(value) or isinstance(value, (int, float, np.integer, np.floating)) else value)
    return formatted

def newey_west_mean_test(return_series, hac_lags=5, annualization=252):
    """
    Test whether a return or IC series has a non-zero mean using Newey-West/HAC standard errors.
    The function reports annualized means for returns when annualization=252 and raw means for IC when annualization=1.
    """
    clean_returns = return_series.dropna().astype(float)
    if len(clean_returns) < max(10, hac_lags + 2) or clean_returns.std() == 0:
        return pd.Series({
            'Mean': np.nan,
            'Annualized Mean': np.nan,
            'HAC t-stat': np.nan,
            'HAC p-value': np.nan,
            'HAC 95% CI Lower': np.nan,
            'HAC 95% CI Upper': np.nan,
            'Observations': len(clean_returns),
            'HAC Lags': hac_lags
        })
    X = pd.DataFrame({'const': 1.0}, index=clean_returns.index)
    model = sm.OLS(clean_returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': hac_lags})
    ci_lower, ci_upper = model.conf_int(alpha=0.05).loc['const']
    return pd.Series({
        'Mean': model.params['const'],
        'Annualized Mean': model.params['const'] * annualization,
        'HAC t-stat': model.tvalues['const'],
        'HAC p-value': model.pvalues['const'],
        'HAC 95% CI Lower': ci_lower * annualization,
        'HAC 95% CI Upper': ci_upper * annualization,
        'Observations': int(model.nobs),
        'HAC Lags': hac_lags
    })

def build_factor_candidate_panel(asset_returns, candidates):
    """
    Build fixed candidate long-short factors with equal weights inside each leg.
    The rule is fixed independently of current AUM so both retrospective temporal segments use identical, observable inputs.
    """
    factor_returns = {}
    construction_rows = []
    for candidate in candidates:
        factor_name = candidate['Factor']
        long_assets = candidate['Long Assets']
        short_assets = candidate['Short Assets']
        long_weights = pd.Series(1.0 / len(long_assets), index=long_assets, dtype=float)
        short_weights = pd.Series(1.0 / len(short_assets), index=short_assets, dtype=float)
        long_return = build_weighted_leg_return(asset_returns, long_assets, long_weights)
        short_return = build_weighted_leg_return(asset_returns, short_assets, short_weights)
        factor_returns[factor_name] = (long_return - short_return).dropna()
        construction_rows.append({
            'Factor': factor_name,
            'Weighting Rule': 'Equal weight fixed within each leg for this analysis',
            'Long Assets': ', '.join(long_assets),
            'Short Assets': ', '.join(short_assets),
            'Long Weights': '; '.join([
                f'{asset}:{format_display_value(weight, style="percent")}'
                for asset, weight in long_weights.items()
            ]),
            'Short Weights': '; '.join([
                f'{asset}:{format_display_value(weight, style="percent")}'
                for asset, weight in short_weights.items()
            ]),
            'Economic Rationale': candidate['Economic Rationale']
        })
    return pd.DataFrame(factor_returns).dropna(how='all'), pd.DataFrame(construction_rows).set_index('Factor')

def calculate_factor_discovery_summary(candidate_returns, validation_fraction=0.40, hac_lags=5):
    """
    Compare candidate factors with a discovery/validation split and HAC mean tests.
    The earlier segment ranks hypotheses; the later retrospective segment checks whether the direction persists.
    """
    aligned = candidate_returns.dropna()
    if len(aligned) < 80:
        raise ValueError('Candidate factor discovery requires at least 80 aligned daily observations.')
    split_position = int(len(aligned) * (1.0 - validation_fraction))
    split_position = min(max(split_position, 40), len(aligned) - 40)
    discovery = aligned.iloc[:split_position]
    validation = aligned.iloc[split_position:]
    rows = []
    for factor_name in aligned.columns:
        full_series = aligned[factor_name].dropna()
        discovery_series = discovery[factor_name].dropna()
        validation_series = validation[factor_name].dropna()
        discovery_test = newey_west_mean_test(discovery_series, hac_lags=hac_lags, annualization=252)
        validation_test = newey_west_mean_test(validation_series, hac_lags=hac_lags, annualization=252)
        full_test = newey_west_mean_test(full_series, hac_lags=hac_lags, annualization=252)
        full_metrics = calculate_factor_value_metrics(full_series, rf_daily=0.0)
        validation_metrics = calculate_factor_value_metrics(validation_series, rf_daily=0.0)
        rows.append({
            'Factor': factor_name,
            'Discovery Annualized Mean': discovery_test['Annualized Mean'],
            'Discovery HAC t-stat': discovery_test['HAC t-stat'],
            'Discovery HAC p-value': discovery_test['HAC p-value'],
            'Validation Annualized Mean': validation_test['Annualized Mean'],
            'Validation HAC t-stat': validation_test['HAC t-stat'],
            'Validation HAC p-value': validation_test['HAC p-value'],
            'Validation Sharpe': validation_metrics['Sharpe Ratio'],
            'Full Annualized Mean': full_test['Annualized Mean'],
            'Full HAC t-stat': full_test['HAC t-stat'],
            'Full HAC p-value': full_test['HAC p-value'],
            'Full Sharpe': full_metrics['Sharpe Ratio'],
            'Full Maximum Drawdown': full_metrics['Maximum Drawdown'],
            'Discovery Observations': int(discovery_test['Observations']),
            'Validation Observations': int(validation_test['Observations'])
        })
    summary = pd.DataFrame(rows).set_index('Factor')
    sample_labels = ('Discovery', 'Validation', 'Full')
    for sample_label in sample_labels:
        raw_column = f'{sample_label} HAC p-value'
        adjusted_column = f'{sample_label} Holm p-value'
        summary[adjusted_column] = multipletests(summary[raw_column].to_numpy(), method='holm')[1]
    summary['Discovery Rank by HAC t-stat'] = (
        summary['Discovery HAC t-stat'].rank(ascending=False, method='min').astype(int)
    )
    summary['Validation Direction'] = np.where(
        summary['Validation Annualized Mean'] > 0,
        'Positive',
        'Negative'
    )
    return summary.sort_values('Discovery Rank by HAC t-stat')

def calculate_static_signal_ic(asset_returns, signal_scores, label, as_of_date, hac_lags=3):
    """
    Calculate monthly IC and RankIC for a static cross-sectional signal against next-month ETF returns.
    This provides a formal factor-research check separate from long-short realised returns.
    Only calendar months completed strictly before the exclusive research cut-off enter the inference sample.
    """
    scores = pd.Series(signal_scores, dtype=float).dropna()
    scores = scores[scores != 0.0]
    signal_return_panel = asset_returns.reindex(columns=scores.index).sort_index()
    if signal_return_panel.empty or signal_return_panel.dropna(how='all').empty:
        raise ValueError('Static-signal IC requires a non-empty dated return panel.')
    if not isinstance(signal_return_panel.index, pd.DatetimeIndex):
        raise TypeError('Static-signal IC requires a DatetimeIndex.')
    research_cutoff = pd.Timestamp(as_of_date).normalize()
    if pd.isna(research_cutoff):
        raise ValueError('Static-signal IC requires a valid research cut-off date.')
    signal_return_panel = signal_return_panel.loc[signal_return_panel.index < research_cutoff]
    cutoff_period = research_cutoff.to_period('M')
    complete_daily_returns = signal_return_panel.loc[
        signal_return_panel.index.to_period('M') < cutoff_period
    ]
    monthly_returns = (1.0 + complete_daily_returns).resample('ME').prod(min_count=1) - 1.0
    if (monthly_returns.index.to_period('M') >= cutoff_period).any():
        raise AssertionError('Terminal partial month entered the monthly IC sample.')
    rows = []
    for position in range(len(monthly_returns) - 1):
        next_returns = monthly_returns.iloc[position + 1].dropna()
        aligned_scores = scores.reindex(next_returns.index).dropna()
        next_returns = next_returns.reindex(aligned_scores.index)
        if len(next_returns) < 4 or aligned_scores.nunique() < 2 or next_returns.std() == 0:
            continue
        rows.append({
            'Signal Date': monthly_returns.index[position],
            'Return Date': monthly_returns.index[position + 1],
            'IC': aligned_scores.corr(next_returns, method='pearson'),
            'RankIC': aligned_scores.rank().corr(next_returns.rank(), method='pearson')
        })
    ic_series = pd.DataFrame(rows).set_index('Return Date') if rows else pd.DataFrame()
    if ic_series.empty:
        summary = pd.Series({
            'Signal': label,
            'Mean IC': np.nan,
            'Mean RankIC': np.nan,
            'RankICIR': np.nan,
            'Positive RankIC Ratio': np.nan,
            'RankIC HAC t-stat': np.nan,
            'RankIC HAC p-value': np.nan,
            'RankIC HAC 95% CI Lower': np.nan,
            'RankIC HAC 95% CI Upper': np.nan,
            'Monthly Observations': 0
        })
        return ic_series, summary
    rankic_test = newey_west_mean_test(ic_series['RankIC'], hac_lags=hac_lags, annualization=1)
    rankic_std = ic_series['RankIC'].std()
    summary = pd.Series({
        'Signal': label,
        'Mean IC': ic_series['IC'].mean(),
        'Mean RankIC': ic_series['RankIC'].mean(),
        'RankICIR': ic_series['RankIC'].mean() / rankic_std if rankic_std > 0 else np.nan,
        'Positive RankIC Ratio': (ic_series['RankIC'] > 0).mean(),
        'RankIC HAC t-stat': rankic_test['HAC t-stat'],
        'RankIC HAC p-value': rankic_test['HAC p-value'],
        'RankIC HAC 95% CI Lower': rankic_test['HAC 95% CI Lower'],
        'RankIC HAC 95% CI Upper': rankic_test['HAC 95% CI Upper'],
        'Monthly Observations': len(ic_series)
    })
    return ic_series, summary

def format_factor_discovery_table(df):
    """
    Format factor discovery and formal-test tables by column type for notebook display.
    Numeric CSV files remain unformatted; this helper only affects visual presentation.
    """
    formatted = df.copy().astype(object)
    percent_keywords = ['Mean', 'Annualized', 'Drawdown', 'Ratio', 'CI Lower', 'CI Upper', 'CI']
    integer_keywords = ['Observations', 'Monthly Observations', 'HAC Lags', 'Rank by']
    pvalue_keywords = ['p-value']
    ic_keywords = ['Mean IC', 'Mean RankIC', 'RankIC HAC 95%']
    ratio_keywords = ['t-stat', 'Sharpe', 'RankICIR']
    for column in formatted.columns:
        if any(keyword in column for keyword in pvalue_keywords):
            formatted[column] = formatted[column].map(lambda value: format_display_value(value, style='pvalue'))
        elif any(keyword in column for keyword in integer_keywords):
            formatted[column] = formatted[column].map(lambda value: format_display_value(value, style='integer'))
        elif any(keyword in column for keyword in ic_keywords):
            formatted[column] = formatted[column].map(lambda value: format_display_value(value, style='decimal4'))
        elif any(keyword in column for keyword in percent_keywords):
            formatted[column] = formatted[column].map(lambda value: format_display_value(value, style='percent'))
        elif any(keyword in column for keyword in ratio_keywords):
            formatted[column] = formatted[column].map(lambda value: format_display_value(value, style='ratio'))
        else:
            formatted[column] = formatted[column].map(lambda value: format_display_value(value, style='ratio') if pd.isna(value) or isinstance(value, (int, float, np.integer, np.floating)) else value)
    return formatted

# Candidate-signal evidence uses ex-ante equal-weight legs; current AUM is reserved for the retrospective BL prior.
low_carbon_factor_weights = pd.Series(1.0 / len(FACTOR_LONG_ASSETS), index=FACTOR_LONG_ASSETS)
high_carbon_factor_weights = pd.Series(1.0 / len(FACTOR_SHORT_ASSETS), index=FACTOR_SHORT_ASSETS)
low_carbon_basket_returns = build_weighted_leg_return(data["asset_returns"], FACTOR_LONG_ASSETS, low_carbon_factor_weights)
high_carbon_basket_returns = build_weighted_leg_return(data["asset_returns"], FACTOR_SHORT_ASSETS, high_carbon_factor_weights)
low_minus_high_spread = (low_carbon_basket_returns - high_carbon_basket_returns).dropna()

low_carbon_factor_construction = pd.DataFrame({
    "Long Leg Weight": low_carbon_factor_weights,
    "Short Leg Weight": high_carbon_factor_weights,
}).fillna(0.0)
net_factor_weights = pd.concat([low_carbon_factor_weights, -high_carbon_factor_weights])
low_carbon_factor_construction["Net Factor Weight"] = net_factor_weights.reindex(low_carbon_factor_construction.index).fillna(0.0)
low_carbon_factor_construction.to_csv(DATA_OUTPUT_DIR / "low_carbon_factor_construction.csv", encoding=CSV_ENCODING)

factor_value_returns = pd.DataFrame({
    "Low-Carbon Basket": low_carbon_basket_returns,
    "High-Carbon Basket": high_carbon_basket_returns,
    "Low-minus-High Spread": low_minus_high_spread,
    "Market Proxy Excess": data["factor_excess"]["Market"],
    "Momentum Proxy Excess": data["factor_excess"]["Momentum"]
}).dropna()
factor_value_returns.to_csv(DATA_OUTPUT_DIR / "low_carbon_factor_returns.csv", index_label="Date", encoding=CSV_ENCODING)

factor_rf_daily = data["rf_daily"].reindex(factor_value_returns.index)
factor_value_metrics = pd.DataFrame({
    "Low-Carbon Basket": calculate_factor_value_metrics(
        factor_value_returns["Low-Carbon Basket"], rf_daily=factor_rf_daily
    ),
    "High-Carbon Basket": calculate_factor_value_metrics(
        factor_value_returns["High-Carbon Basket"], rf_daily=factor_rf_daily
    ),
    "Low-minus-High Spread": calculate_factor_value_metrics(
        factor_value_returns["Low-minus-High Spread"], rf_daily=0.0
    )
})
factor_value_metrics.to_csv(DATA_OUTPUT_DIR / "low_carbon_factor_value_metrics.csv", encoding=CSV_ENCODING)

factor_alpha_summary = pd.DataFrame([
    run_factor_alpha_regression(
        factor_value_returns["Low-minus-High Spread"],
        factor_value_returns[["Market Proxy Excess"]],
        "Market-adjusted"
    ),
    run_factor_alpha_regression(
        factor_value_returns["Low-minus-High Spread"],
        factor_value_returns[[
            "Market Proxy Excess",
            "Momentum Proxy Excess"
        ]],
        "Market + Momentum adjusted"
    )
]).set_index("Model")
factor_alpha_summary.to_csv(DATA_OUTPUT_DIR / "low_carbon_factor_alpha_regression.csv", encoding=CSV_ENCODING)

monthly_factor_strategy = run_rebalanced_factor_strategy(
    data["asset_returns"],
    net_factor_weights,
    one_way_cost=FACTOR_ONE_WAY_COST
)
monthly_factor_strategy.to_csv(DATA_OUTPUT_DIR / "low_carbon_factor_monthly_rebalanced_returns.csv", index_label="Date", encoding=CSV_ENCODING)

factor_cost_impact = pd.DataFrame({
    "Monthly Gross": calculate_factor_value_metrics(
        monthly_factor_strategy["Gross Return"], rf_daily=0.0
    ),
    "Monthly Net 10 bps": calculate_factor_value_metrics(
        monthly_factor_strategy["Net Return"], rf_daily=0.0
    )
})
factor_cost_impact.loc["Average Monthly Turnover"] = [
    monthly_factor_strategy.groupby(monthly_factor_strategy.index.to_period("M"))["Turnover"].sum().mean(),
    monthly_factor_strategy.groupby(monthly_factor_strategy.index.to_period("M"))["Turnover"].sum().mean()
]
factor_cost_impact.loc["Average Annual Cost (net column only)"] = [
    0.0,
    monthly_factor_strategy["Cost"].mean() * 252
]
factor_cost_impact.to_csv(DATA_OUTPUT_DIR / "low_carbon_factor_cost_impact.csv", encoding=CSV_ENCODING)

rolling_factor_diagnostics = calculate_rolling_factor_diagnostics(
    factor_value_returns["Low-minus-High Spread"],
    factor_value_returns[[
        "Market Proxy Excess",
        "Momentum Proxy Excess"
    ]],
    window=252
)
rolling_factor_diagnostics.to_csv(DATA_OUTPUT_DIR / "low_carbon_rolling_factor_diagnostics.csv", index_label="Date", encoding=CSV_ENCODING)

LOW_CARBON_DISCOVERY_NAME = 'Low-Carbon Sector Proxy Spread'
FACTOR_DISCOVERY_CANDIDATES = [
    {
        'Factor': LOW_CARBON_DISCOVERY_NAME,
        'Long Assets': ['XLK', 'XLU', 'XLV'],
        'Short Assets': ['XLE', 'XLB', 'XLI'],
        'Economic Rationale': 'Low-carbon and transition-beneficiary sectors versus high-carbon/cyclical sectors.'
    },
    {
        'Factor': 'Technology-Communication Growth Spread',
        'Long Assets': ['XLK', 'XLC'],
        'Short Assets': ['XLP', 'XLU'],
        'Economic Rationale': 'Growth-oriented sectors versus defensive bond-proxy sectors.'
    },
    {
        'Factor': 'Cyclical Recovery Spread',
        'Long Assets': ['XLI', 'XLB', 'XLY', 'XLF'],
        'Short Assets': ['XLP', 'XLU', 'XLV'],
        'Economic Rationale': 'Cyclical and demand-sensitive sectors versus defensive sectors.'
    },
    {
        'Factor': 'Inflation-Hedge Real Asset Spread',
        'Long Assets': ['XLE', 'XLB', 'GLD'],
        'Short Assets': ['TLT', 'XLU'],
        'Economic Rationale': 'Energy, materials and gold versus long-duration fixed income and utility exposure.'
    },
    {
        'Factor': 'Financials versus Duration Spread',
        'Long Assets': ['XLF'],
        'Short Assets': ['TLT'],
        'Economic Rationale': 'Financial-sector exposure versus long-duration Treasury exposure.'
    }
]

candidate_factor_returns, candidate_factor_construction = build_factor_candidate_panel(
    data['asset_returns'],
    FACTOR_DISCOVERY_CANDIDATES
)
candidate_factor_returns.to_csv(DATA_OUTPUT_DIR / 'candidate_factor_returns.csv', index_label='Date', encoding=CSV_ENCODING)
candidate_factor_construction.to_csv(DATA_OUTPUT_DIR / 'candidate_factor_construction.csv', encoding=CSV_ENCODING)

factor_discovery_summary = calculate_factor_discovery_summary(
    candidate_factor_returns,
    validation_fraction=0.40,
    hac_lags=5
)
factor_discovery_summary.to_csv(DATA_OUTPUT_DIR / 'factor_candidate_discovery_validation.csv', encoding=CSV_ENCODING)

robustness_candidate_factor_returns, robustness_candidate_factor_construction = build_factor_candidate_panel(
    robustness_data['asset_returns'],
    FACTOR_DISCOVERY_CANDIDATES
)
robustness_candidate_factor_returns.to_csv(DATA_OUTPUT_DIR / 'candidate_factor_returns_5y.csv', index_label='Date', encoding=CSV_ENCODING)
robustness_factor_discovery_summary = calculate_factor_discovery_summary(
    robustness_candidate_factor_returns,
    validation_fraction=0.40,
    hac_lags=5
)
robustness_factor_discovery_summary.to_csv(DATA_OUTPUT_DIR / 'factor_candidate_discovery_validation_5y.csv', encoding=CSV_ENCODING)

factor_mean_hac_tests = pd.DataFrame({
    'Low-Carbon Basket': newey_west_mean_test(factor_value_returns['Low-Carbon Basket'], hac_lags=5),
    'High-Carbon Basket': newey_west_mean_test(factor_value_returns['High-Carbon Basket'], hac_lags=5),
    'Low-minus-High Spread': newey_west_mean_test(factor_value_returns['Low-minus-High Spread'], hac_lags=5)
}).T
factor_mean_hac_tests.to_csv(DATA_OUTPUT_DIR / 'low_carbon_factor_newey_west_tests.csv', encoding=CSV_ENCODING)

low_carbon_signal_scores = pd.Series(0.0, index=data['asset_returns'].columns)
low_carbon_signal_scores.loc[FACTOR_LONG_ASSETS] = 1.0
low_carbon_signal_scores.loc[FACTOR_SHORT_ASSETS] = -1.0
low_carbon_ic_series, low_carbon_ic_summary = calculate_static_signal_ic(
    data['asset_returns'],
    low_carbon_signal_scores,
    LOW_CARBON_DISCOVERY_NAME,
    as_of_date=ANALYSIS_AS_OF_DATE,
    hac_lags=3
)
low_carbon_ic_series.to_csv(DATA_OUTPUT_DIR / 'low_carbon_monthly_ic_rankic.csv', index_label='Date', encoding=CSV_ENCODING)
pd.DataFrame([low_carbon_ic_summary]).set_index('Signal').to_csv(DATA_OUTPUT_DIR / 'low_carbon_ic_rankic_summary.csv', encoding=CSV_ENCODING)

print("=== Candidate factor discovery and retrospective temporal split ===")
display(format_factor_discovery_table(factor_discovery_summary))

print("=== Five-year candidate factor robustness ===")
display(format_factor_discovery_table(robustness_factor_discovery_summary))

print("=== Low-carbon factor Newey-West mean tests ===")
display(format_factor_discovery_table(factor_mean_hac_tests))

print("=== Low-carbon monthly IC / RankIC validation ===")
display(format_factor_discovery_table(pd.DataFrame([low_carbon_ic_summary]).set_index('Signal')))

print("=== Low-carbon factor construction ===")
low_carbon_factor_construction_display = low_carbon_factor_construction.copy().astype(object)
for column in ["Long Leg Weight", "Short Leg Weight"]:
    low_carbon_factor_construction_display[column] = low_carbon_factor_construction[column].map(
        lambda value: (
            "Not in leg" if abs(value) < 1e-12
            else format_display_value(value, style="percent")
        )
    )
low_carbon_factor_construction_display["Net Factor Weight"] = low_carbon_factor_construction[
    "Net Factor Weight"
].map(
    lambda value: (
        "Not in factor" if abs(value) < 1e-12
        else format_display_value(value, style="percent")
    )
)
display(low_carbon_factor_construction_display)

print("=== Low-carbon factor value metrics ===")
display(format_factor_value_table(
    factor_value_metrics,
    percent_rows=[
        "Cumulative Return",
        "CAGR",
        "Annualized Arithmetic Mean",
        "Annualized Volatility",
        "Maximum Drawdown",
        "Daily Hit Rate"
    ],
    ratio_rows=[
        "Sharpe Ratio",
        "Daily Mean t-stat",
        "Daily Mean p-value",
        "Skewness",
        "Excess Kurtosis"
    ]
))

print("=== Low-carbon spread HAC alpha regression ===")
display(format_factor_value_table(
    factor_alpha_summary.T,
    percent_rows=[
        "Annualized Alpha",
        "Annualized Alpha HAC 95% CI Lower",
        "Annualized Alpha HAC 95% CI Upper"
    ],
    decimal4_rows=["Adjusted R-squared"],
    ratio_rows=[
        "Alpha t-stat",
        "Alpha p-value",
        "Market Proxy Excess",
        "Momentum Proxy Excess"
    ]
))

print("=== Low-carbon factor cost impact ===")
display(format_factor_value_table(
    factor_cost_impact,
    percent_rows=[
        "Cumulative Return",
        "CAGR",
        "Annualized Arithmetic Mean",
        "Annualized Volatility",
        "Maximum Drawdown",
        "Daily Hit Rate",
        "Average Monthly Turnover",
        "Average Annual Cost (net column only)"
    ],
    ratio_rows=[
        "Sharpe Ratio",
        "Daily Mean t-stat",
        "Daily Mean p-value"
    ]
))

if len(rolling_factor_diagnostics) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    axes[0].plot(
        rolling_factor_diagnostics.index,
        rolling_factor_diagnostics["Rolling Sharpe"],
        color="#1f77b4",
        linewidth=1.3
    )
    axes[0].axhline(0, color="black", linewidth=0.8, alpha=0.5)
    axes[0].set_title("Low-Carbon Spread Rolling 1-Year Sharpe")
    axes[0].set_ylabel("Sharpe")
    axes[0].grid(alpha=0.3)

    axes[1].plot(
        rolling_factor_diagnostics.index,
        rolling_factor_diagnostics["Rolling Annualized Alpha"],
        color="#2ca02c",
        linewidth=1.3
    )
    axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.5)
    axes[1].set_title("Rolling Market + Momentum Adjusted Alpha")
    axes[1].set_ylabel("Annualized Alpha")
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURE_OUTPUT_DIR / "low_carbon_factor_value_diagnostics.png", dpi=300, bbox_inches="tight")
    plt.show()

else:
    print("Rolling factor diagnostics skipped: fewer than 252 aligned observations.")

three_year_factor_evidence = factor_discovery_summary.loc[LOW_CARBON_DISCOVERY_NAME]
five_year_factor_evidence = robustness_factor_discovery_summary.loc[LOW_CARBON_DISCOVERY_NAME]
alpha_model_name = "Market + Momentum adjusted"
factor_evidence_summary = pd.DataFrame([
    {
        "Test": "3Y equal-leg spread mean",
        "Estimate": format_display_value(
            three_year_factor_evidence["Full Annualized Mean"], style="percent"
        ),
        "Raw p-value": format_display_value(
            three_year_factor_evidence["Full HAC p-value"], style="pvalue"
        ),
        "Holm p-value": format_display_value(
            three_year_factor_evidence["Full Holm p-value"], style="pvalue"
        ),
        "Decision": "Not significant; scenario only",
    },
    {
        "Test": "3Y market + momentum HAC alpha",
        "Estimate": format_display_value(
            factor_alpha_summary.loc[alpha_model_name, "Annualized Alpha"], style="percent"
        ),
        "Raw p-value": format_display_value(
            factor_alpha_summary.loc[alpha_model_name, "Alpha p-value"], style="pvalue"
        ),
        "Holm p-value": "Outside candidate family",
        "Decision": "No standalone adjusted alpha",
    },
    {
        "Test": "Monthly cross-sectional RankIC",
        "Estimate": format_display_value(
            low_carbon_ic_summary["Mean RankIC"], style="decimal4"
        ),
        "Raw p-value": format_display_value(
            low_carbon_ic_summary["RankIC HAC p-value"], style="pvalue"
        ),
        "Holm p-value": "Outside candidate family",
        "Decision": "No predictive ranking evidence",
    },
    {
        "Test": "5Y equal-leg spread mean",
        "Estimate": format_display_value(
            five_year_factor_evidence["Full Annualized Mean"], style="percent"
        ),
        "Raw p-value": format_display_value(
            five_year_factor_evidence["Full HAC p-value"], style="pvalue"
        ),
        "Holm p-value": format_display_value(
            five_year_factor_evidence["Full Holm p-value"], style="pvalue"
        ),
        "Decision": "No longer-window support",
    },
])
factor_evidence_summary.to_csv(
    DATA_OUTPUT_DIR / "low_carbon_decision_evidence.csv", index=False, encoding=CSV_ENCODING
)
print("=== Decision-focused low-carbon evidence ===")
display(factor_evidence_summary)


## View calibration and posterior estimates


In [ ]:

# ===================== 3.4 BL view calibration and posterior =====================
VIEW_ABSOLUTE_ASSET = "XLK"
VIEW_LONG_ASSETS = ["XLK", "XLU", "XLV"]
VIEW_SHORT_ASSETS = ["XLE", "XLB", "XLI"]
VIEW_ABSOLUTE_RETURN_CAP = 0.07
VIEW_RELATIVE_RETURN_CAP = 0.04
VIEW_EVIDENCE_HAIRCUT = 0.50
VIEW_HAC_LAGS = 5


def view_leg_weights(assets, prior_weights, weight_scheme):
    """
    Construct equal or AUM-proxy within-leg weights that exactly match the corresponding Black-Litterman P-matrix row.
    """
    if weight_scheme == "equal":
        return pd.Series(1.0 / len(assets), index=assets)
    if weight_scheme == "aum_proxy":
        weights = pd.Series(prior_weights, dtype=float).reindex(assets)
        if weights.isna().any() or weights.sum() <= 0:
            raise ValueError("View basket weights are missing or non-positive.")
        return weights / weights.sum()
    raise ValueError("weight_scheme must be 'equal' or 'aum_proxy'.")


def view_basket_return(excess_returns, assets, prior_weights, weight_scheme):
    """
    Combine asset excess returns using the fixed basket weights used in the corresponding Black-Litterman P row.
    """
    weights = view_leg_weights(assets, prior_weights, weight_scheme)
    return excess_returns[assets].mul(weights, axis=1).sum(axis=1)


def estimate_signed_view(series, return_cap, max_confidence, label, haircut=0.50, hac_lags=5):
    """
    Estimate a signed annualized view and evidence-based confidence from the supplied sample using a discovery/validation split, HAC inference, a haircut and symmetric caps.
    """
    values = pd.Series(series, dtype=float).dropna()
    if len(values) < 60:
        raise ValueError(f"Insufficient observations to calibrate {label}: {len(values)}")
    split = int(np.floor(0.60 * len(values)))
    split = min(max(split, 30), len(values) - 30)
    discovery = values.iloc[:split]
    validation = values.iloc[split:]
    discovery_mean = discovery.mean() * 252.0
    validation_mean = validation.mean() * 252.0
    full_mean = values.mean() * 252.0
    compounded = (1.0 + values).prod() ** (252.0 / len(values)) - 1.0
    anchor = float(np.nanmedian([discovery_mean, validation_mean, compounded]))
    view_return = float(np.clip(haircut * anchor, -return_cap, return_cap))

    fit = sm.OLS(values.to_numpy(), np.ones((len(values), 1))).fit(
        cov_type="HAC", cov_kwds={"maxlags": int(hac_lags)}
    )
    p_value = float(fit.pvalues[0])
    t_statistic = float(fit.tvalues[0])
    sign_stable = bool(
        np.sign(discovery_mean) == np.sign(validation_mean)
        and abs(discovery_mean) > 1e-12
        and abs(validation_mean) > 1e-12
    )
    if not sign_stable or not np.isfinite(p_value):
        confidence = 0.10
    elif p_value <= 0.01:
        confidence = 0.70
    elif p_value <= 0.05:
        confidence = 0.50
    elif p_value <= 0.10:
        confidence = 0.30
    else:
        confidence = 0.10
    confidence = float(min(confidence, max_confidence))

    diagnostics = {
        "View": label,
        "Observations": int(len(values)),
        "Discovery Annual Mean": discovery_mean,
        "Validation Annual Mean": validation_mean,
        "Full Annual Mean": full_mean,
        "Compounded Annual Return": compounded,
        "Scenario Anchor": anchor,
        "Evidence Haircut": haircut,
        "Symmetric Cap": return_cap,
        "BL Input": view_return,
        "HAC t-statistic": t_statistic,
        "HAC p-value": p_value,
        "Sign Stable": sign_stable,
        "Confidence": confidence,
    }
    return view_return, confidence, diagnostics


def calibrate_bl_views(excess_returns, prior_weights, weight_scheme):
    """
    Calibrate the absolute XLK and relative low-minus-high-carbon view magnitudes and confidences using only the supplied estimation window.
    """
    absolute_series = excess_returns[VIEW_ABSOLUTE_ASSET]
    relative_series = (
        view_basket_return(excess_returns, VIEW_LONG_ASSETS, prior_weights, weight_scheme)
        - view_basket_return(excess_returns, VIEW_SHORT_ASSETS, prior_weights, weight_scheme)
    )
    absolute_return, absolute_confidence, absolute_diag = estimate_signed_view(
        absolute_series,
        VIEW_ABSOLUTE_RETURN_CAP,
        max_confidence=0.70,
        label="Absolute XLK scenario",
        haircut=VIEW_EVIDENCE_HAIRCUT,
        hac_lags=VIEW_HAC_LAGS,
    )
    relative_return, relative_confidence, relative_diag = estimate_signed_view(
        relative_series,
        VIEW_RELATIVE_RETURN_CAP,
        max_confidence=0.60,
        label="Low-minus-high carbon sector scenario",
        haircut=VIEW_EVIDENCE_HAIRCUT,
        hac_lags=VIEW_HAC_LAGS,
    )
    views = [
        {
            "type": "absolute",
            "asset": VIEW_ABSOLUTE_ASSET,
            "return": absolute_return,
            "confidence": absolute_confidence,
        },
        {
            "type": "relative",
            "assets_long": VIEW_LONG_ASSETS,
            "assets_short": VIEW_SHORT_ASSETS,
            "return": relative_return,
            "confidence": relative_confidence,
        },
    ]
    diagnostics = pd.DataFrame([absolute_diag, relative_diag])
    diagnostics["Basket Weight Scheme"] = (
        "ETF-AUM proxy" if weight_scheme == "aum_proxy" else "Equal-weight legs"
    )
    diagnostics["Research Role"] = "Conditional scenario, not validated alpha"
    return views, diagnostics


def build_view_scenarios(calibrated_views):
    """
    Build the declared views-off, component, combined, zero-relative and adverse Black-Litterman scenario sets from calibrated views.
    """
    absolute_view = dict(calibrated_views[0])
    relative_view = dict(calibrated_views[1])
    zero_relative = dict(relative_view)
    zero_relative["return"] = 0.0
    adverse_relative = dict(relative_view)
    adverse_relative["return"] = -VIEW_RELATIVE_RETURN_CAP
    adverse_absolute = dict(absolute_view)
    adverse_absolute["return"] = -VIEW_ABSOLUTE_RETURN_CAP
    return {
        "Views-Off Neutral MVO": [],
        "Absolute Only": [absolute_view],
        "Relative Only": [relative_view],
        "Both Views": [absolute_view, relative_view],
        "Zero-Return Relative View": [absolute_view, zero_relative],
        "Adverse Relative": [absolute_view, adverse_relative],
        "Adverse Absolute": [adverse_absolute, relative_view],
        "Joint Adverse": [adverse_absolute, adverse_relative],
    }


def assert_relative_view_alignment(model, views, prior_weights, weight_scheme):
    """
    Assert that the long and short basket weights used for relative-view calibration exactly match the relative Black-Litterman P row.
    """
    relative_indices = [i for i, view in enumerate(views) if view["type"] == "relative"]
    if not relative_indices:
        return
    P, _, _ = model._build_view_matrices(views, weight_scheme=weight_scheme)
    expected = pd.Series(0.0, index=model.assets)
    expected.loc[VIEW_LONG_ASSETS] = view_leg_weights(VIEW_LONG_ASSETS, prior_weights, weight_scheme)
    expected.loc[VIEW_SHORT_ASSETS] = -view_leg_weights(VIEW_SHORT_ASSETS, prior_weights, weight_scheme)
    for row_index in relative_indices:
        if not np.allclose(P[row_index], expected.reindex(model.assets).to_numpy(), atol=1e-12):
            raise AssertionError("Relative-view P weights do not match the calibration basket.")


# Retrospective main-sample calibration: current AUM is used consistently in Q and P.
BL_VIEWS, view_rationale = calibrate_bl_views(
    data["asset_excess"], data["market_weights"], weight_scheme="aum_proxy"
)
bl_model = BlackLittermanModel(
    excess_returns=data["asset_excess"],
    market_weights=data["market_weights"],
    tau=0.025,
    risk_aversion_market=3.0,
)
assert_relative_view_alignment(bl_model, BL_VIEWS, data["market_weights"], "aum_proxy")
bl_result = bl_model.posterior(BL_VIEWS, weight_scheme="aum_proxy")

view_rationale.to_csv(DATA_OUTPUT_DIR / "bl_view_rationale.csv", index=False, encoding=CSV_ENCODING)
view_rationale_display = view_rationale.copy().astype(object)
percent_columns = [
    "Discovery Annual Mean",
    "Validation Annual Mean",
    "Full Annual Mean",
    "Compounded Annual Return",
    "Scenario Anchor",
    "Evidence Haircut",
    "Symmetric Cap",
    "BL Input",
    "Confidence",
]
for column in percent_columns:
    view_rationale_display[column] = view_rationale[column].map(
        lambda value: format_display_value(value, style="percent")
    )
view_rationale_display["HAC p-value"] = view_rationale["HAC p-value"].map(
    lambda value: format_display_value(value, style="pvalue")
)
view_rationale_display["HAC t-statistic"] = view_rationale["HAC t-statistic"].map(
    lambda value: format_display_value(value, style="ratio")
)

print("=== Retrospective BL scenario calibration ===")
display(view_rationale_display)
print("Signed calibration uses a 50% haircut and symmetric caps; negative evidence is not floored at zero.")

return_compare = pd.DataFrame({
    "Prior Equilibrium Return (Annualized)": bl_result["pi_prior"] * 252,
    "Posterior Expected Return (Annualized)": bl_result["mu_post"] * 252,
    "Annualized Posterior-minus-Prior Change": (bl_result["mu_post"] - bl_result["pi_prior"]) * 252,
})
return_compare.to_csv(DATA_OUTPUT_DIR / "bl_prior_posterior.csv", encoding=CSV_ENCODING)
print("=== Prior vs Posterior Comparison (Annualized) ===")
return_compare_display = return_compare.copy().astype(object)
for column in [
    "Prior Equilibrium Return (Annualized)",
    "Posterior Expected Return (Annualized)",
]:
    return_compare_display[column] = return_compare[column].map(
        lambda value: format_display_value(value, style="percent")
    )
return_compare_display["Annualized Posterior-minus-Prior Change"] = return_compare[
    "Annualized Posterior-minus-Prior Change"
].map(lambda value: format_display_value(value, style="basis_points"))
display(return_compare_display)


## View comparison


In [ ]:
# Plot return levels and the economically small update separately
x = np.arange(len(bl_model.assets))
width = 0.35
annual_prior = bl_result["pi_prior"] * 252.0
annual_posterior = bl_result["mu_post"] * 252.0
posterior_change_bp = (annual_posterior - annual_prior) * 10000.0
fig, (ax_returns, ax_change) = plt.subplots(
    2, 1, figsize=(11, 8), sharex=True,
    gridspec_kw={"height_ratios": [2.0, 1.0]}, constrained_layout=True,
)
ax_returns.bar(x - width / 2, annual_prior, width, label="Prior Equilibrium", color="#2878b5")
ax_returns.bar(x + width / 2, annual_posterior, width, label="Posterior BL", color="#c44e52")
ax_returns.set_ylabel("Annual Excess Return")
ax_returns.set_title("Prior and Posterior Expected Returns")
ax_returns.legend()
ax_returns.grid(axis="y", alpha=0.3)
change_colors = np.where(posterior_change_bp >= 0.0, "#2a7f62", "#b23a48")
ax_change.bar(x, posterior_change_bp, color=change_colors, width=0.62)
ax_change.axhline(0.0, color="#333333", linewidth=0.8)
for asset_position, change_value in enumerate(posterior_change_bp):
    ax_change.annotate(
        f"{change_value:+.2f}", xy=(asset_position, change_value),
        xytext=(0, 4 if change_value >= 0.0 else -12), textcoords="offset points",
        ha="center", va="bottom" if change_value >= 0.0 else "top", fontsize=8,
    )
ax_change.set_ylabel("Posterior - Prior (bp)")
ax_change.set_title("Black-Litterman Posterior Update")
ax_change.set_xticks(x, bl_model.assets, rotation=45)
ax_change.grid(axis="y", alpha=0.3)
fig.savefig(FIGURE_OUTPUT_DIR / "prior_vs_posterior_returns.png", dpi=300, bbox_inches="tight")
plt.show()

## Portfolio allocations


In [ ]:

# ===================== 3.5 Optimal Portfolio Allocation =====================
TRUSTEE_LABEL = "Trustee / Conservative (λ=5)"
MARKET_LABEL = "Market / Neutral (λ=3)"
KELLY_STYLE_LABEL = "Kelly-style / Aggressive (λ=1)"

optimizer = PortfolioOptimizer(mu=bl_result["mu_post"], Sigma=bl_result["Sigma_post"])
rf_mean = float(data["rf_daily"].mean())


def format_metric_table(df, percent_rows=None, ratio_rows=None, integer_rows=None, decimal4_rows=None, scientific_rows=None, basis_point_rows=None):
    """
    Format a metric table by declared row type without changing the exported numeric data.
    """
    percent_rows = percent_rows or []
    ratio_rows = ratio_rows or []
    integer_rows = integer_rows or []
    decimal4_rows = decimal4_rows or []
    scientific_rows = scientific_rows or []
    basis_point_rows = basis_point_rows or []
    formatted = df.copy().astype(object)
    for row in formatted.index:
        if row in percent_rows:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="percent"))
        elif row in ratio_rows:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="ratio"))
        elif row in integer_rows:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="integer"))
        elif row in decimal4_rows:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="decimal4"))
        elif row in scientific_rows:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="scientific"))
        elif row in basis_point_rows:
            formatted.loc[row] = formatted.loc[row].map(lambda value: format_display_value(value, style="basis_points"))
        else:
            formatted.loc[row] = formatted.loc[row].map(
                lambda value: format_display_value(value, style="decimal4")
                if pd.isna(value) or isinstance(value, (int, float, np.integer, np.floating)) else value
            )
    return formatted


risk_aversions = {
    TRUSTEE_LABEL: 5.0,
    MARKET_LABEL: 3.0,
    KELLY_STYLE_LABEL: 1.0,
}
mv_weights = {}
mv_stats = {}
main_solver_records = {}
for name, risk_aversion in risk_aversions.items():
    weights = optimizer.mean_variance(risk_aversion=risk_aversion)
    mv_weights[name] = weights
    mv_stats[name] = optimizer.calc_portfolio_stats(weights)
    main_solver_records[name] = dict(optimizer.last_solver_info)

maxsharpe_w = optimizer.max_sharpe()
maxsharpe_stats = optimizer.calc_portfolio_stats(maxsharpe_w)
main_solver_records["Max Sharpe"] = dict(optimizer.last_solver_info)
erc_w = optimizer.equal_risk_contribution()
erc_stats = optimizer.calc_portfolio_stats(erc_w)
main_solver_records["ERC"] = dict(optimizer.last_solver_info)

allocation_stats = {
    "Trustee / Conservative MV": mv_stats[TRUSTEE_LABEL],
    "Market / Neutral MV": mv_stats[MARKET_LABEL],
    "Kelly-style / Aggressive MV": mv_stats[KELLY_STYLE_LABEL],
    "Max Sharpe": maxsharpe_stats,
    "ERC": erc_stats,
}
allocation_weights = {
    "Trustee / Conservative MV": mv_weights[TRUSTEE_LABEL],
    "Market / Neutral MV": mv_weights[MARKET_LABEL],
    "Kelly-style / Aggressive MV": mv_weights[KELLY_STYLE_LABEL],
    "Max Sharpe": maxsharpe_w,
    "ERC": erc_w,
}
CAP_BINDING_TOLERANCE = 1e-5
perf_summary = pd.DataFrame({
    name: [
        stats["expected_return_annual"],
        stats["volatility_annual"],
        stats["sharpe_annual"],
        allocation_weights[name].max(),
        1.0 / np.square(allocation_weights[name].to_numpy(dtype=float)).sum(),
        int(np.isclose(
            allocation_weights[name].to_numpy(dtype=float), optimizer.w_max,
            atol=CAP_BINDING_TOLERANCE, rtol=0.0,
        ).sum()),
    ]
    for name, stats in allocation_stats.items()
}, index=[
    "Annualized Expected Excess Return",
    "Annualized Predicted Volatility",
    "Model-implied Sharpe Ratio",
    "Maximum Single-Asset Weight",
    "Effective Number of Assets",
    "Binding 20% Upper Caps",
])

portfolio_weights = pd.DataFrame({
    "Trustee / Conservative MV": mv_weights[TRUSTEE_LABEL],
    "Market / Neutral MV": mv_weights[MARKET_LABEL],
    "Kelly-style / Aggressive MV": mv_weights[KELLY_STYLE_LABEL],
    "Max Sharpe": maxsharpe_w,
    "ERC": erc_w,
    "Official AUM Prior": data["market_weights"],
})
solver_name_map = {
    TRUSTEE_LABEL: "Trustee / Conservative MV",
    MARKET_LABEL: "Market / Neutral MV",
    KELLY_STYLE_LABEL: "Kelly-style / Aggressive MV",
}
solver_diagnostics = pd.DataFrame.from_dict(main_solver_records, orient="index").rename(index=solver_name_map)
solver_diagnostics = solver_diagnostics.reindex(list(allocation_weights))
single_start_mask = solver_diagnostics["multistarts"].fillna(1).le(1)
solver_diagnostics.loc[single_start_mask, "objective_dispersion"] = np.nan
solver_diagnostics.loc[single_start_mask, "successful_starts"] = solver_diagnostics.loc[single_start_mask, "success"].astype(int)
solver_diagnostics["weight_sum_error"] = [
    abs(float(weights.sum()) - 1.0) for weights in allocation_weights.values()
]
solver_diagnostics["max_bound_violation"] = [
    max(0.0, -float(weights.min()), float(weights.max()) - optimizer.w_max)
    for weights in allocation_weights.values()
]
portfolio_weights.to_csv(DATA_OUTPUT_DIR / "portfolio_weights.csv", encoding=CSV_ENCODING)
perf_summary.to_csv(DATA_OUTPUT_DIR / "model_allocation_metrics.csv", encoding=CSV_ENCODING)
solver_diagnostics.to_csv(DATA_OUTPUT_DIR / "solver_status.csv", encoding=CSV_ENCODING)

print("=== Model-Implied Allocation Diagnostics ===")
display(format_metric_table(
    perf_summary,
    percent_rows=[
        "Annualized Expected Excess Return",
        "Annualized Predicted Volatility",
        "Maximum Single-Asset Weight",
    ],
    ratio_rows=[
        "Model-implied Sharpe Ratio",
        "Effective Number of Assets",
    ],
    integer_rows=["Binding 20% Upper Caps"],
))
print("=== Exact portfolio weights ===")
portfolio_weights_display = portfolio_weights.copy().astype(object)
for column in portfolio_weights.columns:
    portfolio_weights_display[column] = portfolio_weights[column].map(
        lambda value: format_display_value(value, style="percent4")
    )
display(portfolio_weights_display)
print("=== Solver and feasibility diagnostics ===")
solver_diagnostics_display = solver_diagnostics.copy().astype(object)
for column in ["iterations", "multistarts", "successful_starts"]:
    solver_diagnostics_display[column] = solver_diagnostics[column].map(
        lambda value: format_display_value(value, style="integer")
    )
solver_diagnostics_display["objective_dispersion"] = solver_diagnostics["objective_dispersion"].map(
    lambda value: format_display_value(
        value,
        style="scientific",
        na_label="N/A (single start)",
    )
)
for column in ["weight_sum_error", "max_bound_violation"]:
    solver_diagnostics_display[column] = solver_diagnostics[column].map(
        lambda value: format_display_value(value, style="scientific", na_label="Not reported")
    )
solver_diagnostics_display = solver_diagnostics_display.rename(columns={
    "method": "Method",
    "status": "Status",
    "success": "Success",
    "iterations": "Iterations",
    "multistarts": "Starts",
    "successful_starts": "Successful Starts",
    "objective_dispersion": "Within-Strategy Objective Range",
    "weight_sum_error": "Weight-Sum Error",
    "max_bound_violation": "Maximum Bound Violation",
})
display(solver_diagnostics_display.T)


## Allocation display


In [ ]:
# Draw the exact exported strategy weights
weights_df = portfolio_weights.drop(columns=["Official AUM Prior"]).copy()

plt.figure(figsize=(12, 6))
weights_df.T.plot(kind="bar", stacked=True, cmap="tab20", ax=plt.gca())
plt.title("Portfolio Weights Comparison Across Strategies")
plt.ylabel("Weight")
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_OUTPUT_DIR / "portfolio_weights_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## Matched view ablations


In [ ]:

# ===================== 3.5.1 View ablation and covariance comparison =====================
MAIN_VIEW_SCENARIOS = build_view_scenarios(BL_VIEWS)
view_ablation_weights = {}
view_ablation_metrics_rows = []
view_ablation_returns = {}

for scenario_name, scenario_views in MAIN_VIEW_SCENARIOS.items():
    scenario_result = bl_model.posterior(scenario_views, weight_scheme="aum_proxy")
    scenario_optimizer = PortfolioOptimizer(scenario_result["mu_post"], scenario_result["Sigma_post"])
    scenario_weights = scenario_optimizer.mean_variance(risk_aversion=3.0)
    scenario_returns = BacktestEngine(scenario_weights, data["asset_returns"]).calc_portfolio_returns()
    scenario_metrics = calculate_performance_metrics_from_returns(scenario_returns, data["rf_daily"])
    scenario_stats = scenario_optimizer.calc_portfolio_stats(scenario_weights)
    view_ablation_weights[scenario_name] = scenario_weights
    view_ablation_returns[scenario_name] = scenario_returns
    view_ablation_metrics_rows.append({
        "Scenario": scenario_name,
        "Expected Excess Return": scenario_stats["expected_return_annual"],
        "Predicted Volatility": scenario_stats["volatility_annual"],
        "Model Sharpe": scenario_stats["sharpe_annual"],
        "Maximum Weight": scenario_weights.max(),
        "Effective Assets": 1.0 / np.square(scenario_weights).sum(),
        "Retrospective CAGR": scenario_metrics["CAGR"],
        "Retrospective Sharpe": scenario_metrics["Annualized Sharpe Ratio"],
        "Retrospective Max Drawdown": scenario_metrics["Maximum Drawdown"],
    })

view_ablation_weights = pd.DataFrame(view_ablation_weights)
view_ablation_returns = pd.DataFrame(view_ablation_returns)
view_ablation_metrics = pd.DataFrame(view_ablation_metrics_rows).set_index("Scenario")
views_off_weights = view_ablation_weights["Views-Off Neutral MVO"]
view_ablation_metrics["Active Share vs Views-Off MVO"] = {
    column: 0.5 * np.abs(view_ablation_weights[column] - views_off_weights).sum()
    for column in view_ablation_weights.columns
}
assert np.allclose(view_ablation_weights["Both Views"], mv_weights[MARKET_LABEL], atol=1e-5)

view_ablation_weights.to_csv(DATA_OUTPUT_DIR / "view_ablation_weights.csv", encoding=CSV_ENCODING)
view_ablation_metrics.to_csv(DATA_OUTPUT_DIR / "view_ablation_metrics.csv", encoding=CSV_ENCODING)
view_ablation_returns.to_csv(DATA_OUTPUT_DIR / "view_ablation_retrospective_returns.csv", index_label="Date", encoding=CSV_ENCODING)

print("=== View ablation and adverse scenario ===")
display(format_metric_table(
    view_ablation_metrics.T,
    percent_rows=[
        "Expected Excess Return",
        "Predicted Volatility",
        "Maximum Weight",
        "Retrospective CAGR",
        "Retrospective Max Drawdown",
        "Active Share vs Views-Off MVO",
    ],
    ratio_rows=[
        "Model Sharpe",
        "Effective Assets",
        "Retrospective Sharpe",
    ],
))

# Same posterior mean, different covariance convention.
covariance_variant_inputs = {
    "Historical Sigma": bl_result["Sigma_historical"],
    "Predictive Sigma + M": bl_result["Sigma_post"],
}
covariance_variant_weights = {}
covariance_variant_rows = []
for covariance_name, covariance_matrix in covariance_variant_inputs.items():
    covariance_optimizer = PortfolioOptimizer(bl_result["mu_post"], covariance_matrix)
    covariance_weights = covariance_optimizer.mean_variance(risk_aversion=3.0)
    covariance_returns = BacktestEngine(covariance_weights, data["asset_returns"]).calc_portfolio_returns()
    covariance_metrics = calculate_performance_metrics_from_returns(covariance_returns, data["rf_daily"])
    covariance_variant_weights[covariance_name] = covariance_weights
    covariance_variant_rows.append({
        "Covariance": covariance_name,
        "Expected Excess Return": float(covariance_weights @ bl_result["mu_post"]) * 252.0,
        "Optimisation Volatility": float(np.sqrt(covariance_weights @ covariance_matrix @ covariance_weights)) * np.sqrt(252.0),
        "Maximum Weight": covariance_weights.max(),
        "Effective Assets": 1.0 / np.square(covariance_weights).sum(),
        "Retrospective CAGR": covariance_metrics["CAGR"],
        "Retrospective Sharpe": covariance_metrics["Annualized Sharpe Ratio"],
    })

covariance_variant_weights = pd.DataFrame(covariance_variant_weights)
covariance_variant_metrics = pd.DataFrame(covariance_variant_rows).set_index("Covariance")
historical_weights = covariance_variant_weights["Historical Sigma"]
covariance_variant_metrics["Active Share vs Historical Sigma"] = {
    column: 0.5 * np.abs(covariance_variant_weights[column] - historical_weights).sum()
    for column in covariance_variant_weights.columns
}
covariance_variant_weights.to_csv(DATA_OUTPUT_DIR / "predictive_covariance_weight_comparison.csv", encoding=CSV_ENCODING)
covariance_variant_metrics.to_csv(DATA_OUTPUT_DIR / "predictive_covariance_metrics.csv", encoding=CSV_ENCODING)

print("=== Historical Sigma vs Predictive Sigma + M ===")
display(format_metric_table(
    covariance_variant_metrics.T,
    percent_rows=[
        "Expected Excess Return",
        "Optimisation Volatility",
        "Maximum Weight",
        "Retrospective CAGR",
        "Active Share vs Historical Sigma",
    ],
    ratio_rows=["Effective Assets", "Retrospective Sharpe"],
))


## Retrospective performance context


In [ ]:

# ===================== 3.6 Retrospective main-sample backtest =====================
main_equal_weights = pd.Series(1.0 / data["asset_returns"].shape[1], index=data["asset_returns"].columns)
main_allocations = {
    "Views-On BL": mv_weights[MARKET_LABEL],
    "Views-Off Neutral MVO": view_ablation_weights["Views-Off Neutral MVO"],
    "Equal-Weight": main_equal_weights,
    "AUM-Weighted Prior": data["market_weights"],
}
main_return_series = {
    name: BacktestEngine(weights, data["asset_returns"]).calc_portfolio_returns()
    for name, weights in main_allocations.items()
}
main_return_series["SPY External Reference"] = data["factor_returns"]["Market"].reindex(data["asset_returns"].index)
main_returns = pd.DataFrame(main_return_series).dropna()
main_rf = data["rf_daily"].reindex(main_returns.index)
main_metrics = pd.DataFrame({
    column: calculate_performance_metrics_from_returns(main_returns[column], main_rf)
    for column in main_returns.columns
})

port_returns = main_returns["Views-On BL"].rename("Views-On BL")
benchmark_returns = main_returns["Equal-Weight"].rename("Equal-Weight")
port_metrics = main_metrics["Views-On BL"]
bench_metrics = main_metrics["Equal-Weight"]

main_returns.to_csv(DATA_OUTPUT_DIR / "main_in_sample_returns.csv", index_label="Date", encoding=CSV_ENCODING)
main_metrics.to_csv(DATA_OUTPUT_DIR / "main_in_sample_metrics.csv", encoding=CSV_ENCODING)

main_view_active = main_returns["Views-On BL"] - main_returns["Views-Off Neutral MVO"]
main_view_tracking_error = main_view_active.std(ddof=1) * np.sqrt(252.0)
incremental_view_effect = pd.DataFrame({
    "Value": {
        "CAGR Difference": (
            main_metrics.loc["CAGR", "Views-On BL"]
            - main_metrics.loc["CAGR", "Views-Off Neutral MVO"]
        ),
        "Sharpe Difference": (
            main_metrics.loc["Annualized Sharpe Ratio", "Views-On BL"]
            - main_metrics.loc["Annualized Sharpe Ratio", "Views-Off Neutral MVO"]
        ),
        "Maximum Drawdown Difference": (
            main_metrics.loc["Maximum Drawdown", "Views-On BL"]
            - main_metrics.loc["Maximum Drawdown", "Views-Off Neutral MVO"]
        ),
        "Annualized Mean Active Return": main_view_active.mean() * 252.0,
        "Realised Tracking Error": main_view_tracking_error,
        "Information Ratio": (
            main_view_active.mean() * 252.0 / main_view_tracking_error
            if main_view_tracking_error > 0 else np.nan
        ),
        "Active Share": view_ablation_metrics.loc[
            "Both Views", "Active Share vs Views-Off MVO"
        ],
    }
})
incremental_view_effect.to_csv(DATA_OUTPUT_DIR / "incremental_view_effect.csv", encoding=CSV_ENCODING)

print("=== Retrospective main-sample performance ===")
display(format_metric_table(
    main_metrics,
    percent_rows=[
        "Cumulative Return",
        "CAGR",
        "Annualized Excess Return",
        "Annualized Volatility",
        "Maximum Drawdown",
    ],
    ratio_rows=[
        "Annualized Sharpe Ratio",
        "Annualized Sortino Ratio",
        "Calmar Ratio",
    ],
    integer_rows=["Observations"],
))
print("=== Incremental view effect versus matched views-off ===")
display(format_metric_table(
    incremental_view_effect,
    percent_rows=[
        "Realised Tracking Error", "Active Share",
    ],
    ratio_rows=["Sharpe Difference", "Information Ratio"],
    basis_point_rows=[
        "CAGR Difference", "Maximum Drawdown Difference",
        "Annualized Mean Active Return",
    ],
))

backtest = BacktestEngine(main_allocations["Views-On BL"], data["asset_returns"].reindex(main_returns.index))
fig = backtest.plot_performance(
    port_returns,
    main_returns[["Views-Off Neutral MVO", "Equal-Weight", "AUM-Weighted Prior", "SPY External Reference"]],
)
fig.savefig(FIGURE_OUTPUT_DIR / "backtest_performance.png", dpi=300, bbox_inches="tight")
plt.show()


## Factor regressions


In [ ]:

# ===================== 3.7 HAC factor exposure and collinearity diagnosis =====================
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests


def format_regression_display_table(frame, term_column=None):
    """
    Format regression output by statistical role, with annualized alpha shown as a percentage.
    """
    formatted = frame.copy().astype(object)
    for row_index in formatted.index:
        term = str(frame.loc[row_index, term_column]) if term_column else str(row_index)
        alpha_row = "Alpha (annualized)" in term
        for column in formatted.columns:
            label = str(column)
            value = frame.loc[row_index, column]
            if "p-value" in label:
                na_label = "Not adjusted" if alpha_row and "Holm" in label else "N/A"
                formatted.loc[row_index, column] = format_display_value(
                    value, style="pvalue", na_label=na_label
                )
            elif any(keyword in label for keyword in [
                "Coefficient", "Beta", "Standard Error",
                "CI Lower", "CI Upper",
            ]):
                formatted.loc[row_index, column] = format_display_value(
                    value, style="percent" if alpha_row else "decimal4"
                )
            elif "t-stat" in label or "R-squared" in label:
                formatted.loc[row_index, column] = format_display_value(value, style="decimal4")
    return formatted


regression_panel = pd.concat([
    port_returns.rename("Portfolio Return"),
    data["rf_daily"].rename("RF"),
    data["factor_excess"][["Market", "ESG", "Momentum"]],
], axis=1).dropna()
portfolio_excess = regression_panel["Portfolio Return"] - regression_panel["RF"]
raw_factor_proxies = regression_panel[["Market", "ESG", "Momentum"]]

raw_factor_correlation = raw_factor_proxies.corr()
raw_design = sm.add_constant(raw_factor_proxies, has_constant="add")
raw_factor_vif = pd.DataFrame({
    "Variable": raw_design.columns,
    "VIF": [variance_inflation_factor(raw_design.to_numpy(), i) for i in range(raw_design.shape[1])],
}).set_index("Variable")

proxy_market_rows = []
orthogonal_factors = {"Market": raw_factor_proxies["Market"]}
for proxy_name in ["ESG", "Momentum"]:
    market_fit = sm.OLS(
        raw_factor_proxies[proxy_name],
        sm.add_constant(raw_factor_proxies[["Market"]], has_constant="add"),
    ).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    orthogonal_name = f"{proxy_name} Orthogonalized"
    orthogonal_factors[orthogonal_name] = market_fit.resid
    market_beta_ci = market_fit.conf_int(alpha=0.05).loc["Market"]
    proxy_market_rows.append({
        "Proxy": proxy_name,
        "Market Beta": market_fit.params["Market"],
        "HAC 95% CI Lower": market_beta_ci.iloc[0],
        "HAC 95% CI Upper": market_beta_ci.iloc[1],
        "HAC t-statistic": market_fit.tvalues["Market"],
        "HAC p-value": market_fit.pvalues["Market"],
        "R-squared": market_fit.rsquared,
    })

orthogonal_factor_frame = pd.DataFrame(orthogonal_factors).dropna()
orthogonal_design = sm.add_constant(orthogonal_factor_frame, has_constant="add")
orthogonal_factor_vif = pd.DataFrame({
    "Variable": orthogonal_design.columns,
    "VIF": [variance_inflation_factor(orthogonal_design.to_numpy(), i) for i in range(orthogonal_design.shape[1])],
}).set_index("Variable")

factor_analyzer = FactorAnalyzer(portfolio_excess, orthogonal_factor_frame)
PORTFOLIO_FACTOR_NAMES = ["Market", "ESG Orthogonalized", "Momentum Orthogonalized"]
PORTFOLIO_HAC_LAG_GRID = (1, 5, 10, 21)
reg_result = factor_analyzer.factor_regression(
    PORTFOLIO_FACTOR_NAMES, hac_lags=5
)
slope_mask = reg_result.index != "const"
reg_result["Holm-adjusted p-value"] = np.nan
reg_result.loc[slope_mask, "Holm-adjusted p-value"] = multipletests(
    reg_result.loc[slope_mask, "HAC p-value"].to_numpy(), method="holm"
)[1]
portfolio_hac_rows = []
for hac_lags in PORTFOLIO_HAC_LAG_GRID:
    lag_result = factor_analyzer.factor_regression(PORTFOLIO_FACTOR_NAMES, hac_lags=hac_lags)
    lag_adjusted = pd.Series(np.nan, index=lag_result.index, dtype=float)
    lag_slope_mask = lag_result.index != "const"
    lag_adjusted.loc[lag_slope_mask] = multipletests(
        lag_result.loc[lag_slope_mask, "HAC p-value"].to_numpy(), method="holm"
    )[1]
    for term, row in lag_result.iterrows():
        scale = 252.0 if term == "const" else 1.0
        portfolio_hac_rows.append({
            "HAC Lags": hac_lags,
            "Term": "Alpha (annualized)" if term == "const" else term,
            "Coefficient": row["Coefficient"] * scale,
            "HAC Standard Error": row["HAC Standard Error"] * scale,
            "HAC t-statistic": row["HAC t-statistic"],
            "HAC p-value": row["HAC p-value"],
            "Holm-adjusted p-value": lag_adjusted.loc[term],
            "HAC 95% CI Lower": row["HAC 95% CI Lower"] * scale,
            "HAC 95% CI Upper": row["HAC 95% CI Upper"] * scale,
            "Reporting Scale": "Annualized alpha" if term == "const" else "Factor beta",
        })
portfolio_regression_hac_lag_sensitivity = pd.DataFrame(portfolio_hac_rows).set_index(
    ["HAC Lags", "Term"]
)
regression_report = reg_result.copy()
annualized_columns = [
    "Coefficient", "HAC Standard Error",
    "HAC 95% CI Lower", "HAC 95% CI Upper",
]
regression_report.loc["const", annualized_columns] = (
    regression_report.loc["const", annualized_columns] * 252.0
)
regression_report["Reporting Scale"] = "Factor beta"
regression_report.loc["const", "Reporting Scale"] = "Annualized alpha"
regression_report = regression_report.rename(index={"const": "Alpha (annualized)"})
reg_diagnostics = pd.DataFrame(reg_result.attrs["model_diagnostics"], columns=["Value"])
factor_market_summary = pd.DataFrame(proxy_market_rows).set_index("Proxy")

raw_factor_correlation.to_csv(DATA_OUTPUT_DIR / "factor_proxy_correlation.csv", encoding=CSV_ENCODING)
raw_factor_vif.to_csv(DATA_OUTPUT_DIR / "factor_proxy_raw_vif.csv", encoding=CSV_ENCODING)
orthogonal_factor_vif.to_csv(DATA_OUTPUT_DIR / "factor_proxy_orthogonalized_vif.csv", encoding=CSV_ENCODING)
factor_market_summary.to_csv(DATA_OUTPUT_DIR / "factor_proxy_market_regressions.csv", encoding=CSV_ENCODING)
reg_result.to_csv(DATA_OUTPUT_DIR / "portfolio_orthogonalized_factor_regression_raw_daily.csv", encoding=CSV_ENCODING)
regression_report.to_csv(DATA_OUTPUT_DIR / "portfolio_orthogonalized_factor_regression.csv", encoding=CSV_ENCODING)
reg_diagnostics.to_csv(DATA_OUTPUT_DIR / "portfolio_orthogonalized_factor_diagnostics.csv", encoding=CSV_ENCODING)
portfolio_regression_hac_lag_sensitivity.to_csv(
    DATA_OUTPUT_DIR / "portfolio_orthogonalized_factor_hac_lag_sensitivity.csv", encoding=CSV_ENCODING
)

print("=== Raw factor-proxy correlations ===")
display(format_numeric_frame(raw_factor_correlation, style="decimal4", na_label="Not estimated"))
print("=== Raw-proxy VIF diagnostics ===")
display(format_numeric_frame(raw_factor_vif, style="decimal4", na_label="Not estimated"))
print("=== ESGU and MTUM regressed on SPY excess returns ===")
display(format_regression_display_table(factor_market_summary))
print("=== Orthogonalized-factor VIF ===")
display(format_numeric_frame(orthogonal_factor_vif, style="decimal4", na_label="Not estimated"))
print("=== Portfolio excess-return regression with HAC inference ===")
display(format_regression_display_table(regression_report))
display(format_metric_table(
    reg_diagnostics,
    ratio_rows=["R-squared", "Adjusted R-squared", "Condition Number"],
    integer_rows=["HAC Lags", "Observations"],
))
print("=== Portfolio regression HAC-lag sensitivity ===")
hac_sensitivity_display = portfolio_regression_hac_lag_sensitivity.reset_index()[[
    "HAC Lags", "Term", "Coefficient", "HAC Standard Error",
    "HAC p-value", "Holm-adjusted p-value",
]]
display(format_regression_display_table(hac_sensitivity_display, term_column="Term"))

def rolling_multifactor_diagnostics(target, factors, coefficient, window=252, hac_lags=5):
    """
    Estimate a dated rolling multifactor coefficient with HAC standard error and 95% confidence interval in every complete trailing window.
    """
    panel = pd.concat([target.rename("Portfolio Excess"), factors], axis=1).dropna()
    records = []
    for end_position in range(window - 1, len(panel)):
        rolling_panel = panel.iloc[end_position - window + 1:end_position + 1]
        design = sm.add_constant(rolling_panel[PORTFOLIO_FACTOR_NAMES], has_constant="add")
        fitted = sm.OLS(rolling_panel["Portfolio Excess"], design).fit(
            cov_type="HAC", cov_kwds={"maxlags": int(hac_lags)}
        )
        confidence_interval = fitted.conf_int(alpha=0.05).loc[coefficient]
        records.append({
            "Date": rolling_panel.index[-1],
            "Coefficient": float(fitted.params[coefficient]),
            "HAC Standard Error": float(fitted.bse[coefficient]),
            "HAC 95% CI Lower": float(confidence_interval.iloc[0]),
            "HAC 95% CI Upper": float(confidence_interval.iloc[1]),
            "Window Observations": int(window),
            "HAC Lags": int(hac_lags),
        })
    return pd.DataFrame.from_records(records).set_index("Date")

rolling_esg_diagnostics = rolling_multifactor_diagnostics(
    portfolio_excess, orthogonal_factor_frame, "ESG Orthogonalized", window=252, hac_lags=5
)
rolling_esg_diagnostics.to_csv(
    DATA_OUTPUT_DIR / "rolling_252d_portfolio_esg_residual_coefficient.csv",
    index_label="Date", encoding=CSV_ENCODING,
)
rolling_esg = rolling_esg_diagnostics["Coefficient"]
rolling_esg_sign_crossings = int((rolling_esg * rolling_esg.shift(1) < 0.0).sum())
rolling_esg_summary = pd.DataFrame({"Value": {
    "Rolling Windows": int(len(rolling_esg)),
    "Window Observations": 252,
    "HAC Lags": 5,
    "Minimum Coefficient": float(rolling_esg.min()),
    "Maximum Coefficient": float(rolling_esg.max()),
    "Negative Windows": int((rolling_esg < 0.0).sum()),
    "Negative Window Share": float((rolling_esg < 0.0).mean()),
    "Zero Crossings": rolling_esg_sign_crossings,
    "Terminal Coefficient": float(rolling_esg.iloc[-1]),
}})
rolling_esg_summary.to_csv(
    DATA_OUTPUT_DIR / "rolling_252d_portfolio_esg_residual_coefficient_summary.csv",
    encoding=CSV_ENCODING,
)
print("=== Rolling ESG-residual coefficient stability ===")
display(format_metric_table(
    rolling_esg_summary,
    percent_rows=["Negative Window Share"],
    decimal4_rows=[
        "Minimum Coefficient", "Maximum Coefficient",
        "Terminal Coefficient",
    ],
    integer_rows=[
        "Rolling Windows", "Window Observations",
        "HAC Lags", "Negative Windows",
        "Zero Crossings",
    ],
))
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(rolling_esg.index, rolling_esg, color="#2a7f62", linewidth=1.3, label="Rolling coefficient")
ax.fill_between(
    rolling_esg.index,
    rolling_esg_diagnostics["HAC 95% CI Lower"],
    rolling_esg_diagnostics["HAC 95% CI Upper"],
    color="#2a7f62", alpha=0.16, label="HAC 95% interval",
)
ax.axhline(0.0, color="#333333", linewidth=0.9)
ax.set_title("Retrospective Rolling ESGU-Residual Coefficient (252 Days)")
ax.set_ylabel("Residual ESGU Coefficient")
ax.set_xlim(rolling_esg.index.min(), rolling_esg.index.max())
ax.legend(loc="best")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_OUTPUT_DIR / "rolling_esg_beta.png", dpi=300, bbox_inches="tight")
plt.show()


## Numerical and specification checks


In [ ]:

# ===================== 3.8.1 Retrospective robustness checks =====================
def calculate_max_drawdown(return_series):
    """
    Compute the most negative peak-to-trough drawdown from non-empty simple returns, including initial wealth of one.
    """
    returns = pd.Series(return_series, dtype=float).dropna()
    wealth = (1.0 + returns).cumprod().to_numpy()
    peaks = np.maximum.accumulate(np.r_[1.0, wealth])[1:]
    return float(np.min(wealth / peaks - 1.0))


def run_neutral_mv_for_tau(tau_value):
    """
    Rebuild the AUM-prior Black-Litterman neutral mean-variance portfolio for one tau value and return its weights, realized returns and performance metrics.
    """
    model = BlackLittermanModel(
        data["asset_excess"], data["market_weights"],
        tau=tau_value, risk_aversion_market=3.0,
    )
    result = model.posterior(BL_VIEWS, weight_scheme="aum_proxy")
    weights = PortfolioOptimizer(result["mu_post"], result["Sigma_post"]).mean_variance(risk_aversion=3.0)
    returns = BacktestEngine(weights, data["asset_returns"]).calc_portfolio_returns()
    metrics = calculate_performance_metrics_from_returns(returns, data["rf_daily"])
    return weights, returns, metrics


tau_rows = {}
base_weights = mv_weights[MARKET_LABEL]
for tau_value in [0.01, 0.025, 0.05]:
    weights, returns, metrics = run_neutral_mv_for_tau(tau_value)
    tau_rows[f"tau={tau_value}"] = {
        "CAGR": metrics["CAGR"],
        "Annualized Volatility": metrics["Annualized Volatility"],
        "Annualized Sharpe Ratio": metrics["Annualized Sharpe Ratio"],
        "Maximum Drawdown": metrics["Maximum Drawdown"],
        "Maximum Weight": weights.max(),
        "Active Share vs Base": 0.5 * np.abs(weights - base_weights).sum(),
    }
tau_sensitivity = pd.DataFrame(tau_rows)
tau_sensitivity.to_csv(DATA_OUTPUT_DIR / "tau_sensitivity.csv", encoding=CSV_ENCODING)
print("=== Tau sensitivity (retrospective) ===")
display(format_metric_table(
    tau_sensitivity,
    percent_rows=[
        "CAGR", "Annualized Volatility",
        "Maximum Drawdown", "Maximum Weight",
        "Active Share vs Base",
    ],
    ratio_rows=["Annualized Sharpe Ratio"],
))


def build_ledoit_wolf_covariance(excess_returns):
    """
    Estimate the daily return covariance with the required scikit-learn Ledoit-Wolf estimator.
    """
    clean_returns = excess_returns.dropna().astype(float)
    if clean_returns.shape[0] < 2:
        raise ValueError("Covariance shrinkage requires at least two complete observations.")
    estimator = LedoitWolf().fit(clean_returns.to_numpy())
    covariance = pd.DataFrame(
        estimator.covariance_, index=clean_returns.columns, columns=clean_returns.columns
    )
    return covariance, "Ledoit-Wolf shrinkage (scikit-learn)", float(estimator.shrinkage_)


shrink_covariance, shrinkage_method, shrinkage_intensity = build_ledoit_wolf_covariance(data["asset_excess"])
shrink_model = BlackLittermanModel(
    data["asset_excess"], data["market_weights"], tau=0.025,
    risk_aversion_market=3.0, covariance_matrix=shrink_covariance,
)
shrink_result = shrink_model.posterior(BL_VIEWS, weight_scheme="aum_proxy")
shrink_weights = PortfolioOptimizer(shrink_result["mu_post"], shrink_result["Sigma_post"]).mean_variance(risk_aversion=3.0)
shrink_returns = BacktestEngine(shrink_weights, data["asset_returns"]).calc_portfolio_returns()
shrink_metrics = calculate_performance_metrics_from_returns(shrink_returns, data["rf_daily"])
covariance_robustness = pd.DataFrame({
    "Sample covariance": {
        "CAGR": port_metrics["CAGR"],
        "Annualized Volatility": port_metrics["Annualized Volatility"],
        "Annualized Sharpe Ratio": port_metrics["Annualized Sharpe Ratio"],
        "Maximum Drawdown": port_metrics["Maximum Drawdown"],
        "Maximum Weight": base_weights.max(),
        "Active Share vs Sample": 0.0,
        "Shrinkage Intensity": 0.0,
        "Covariance Condition Number": np.linalg.cond(bl_result["Sigma_historical"]),
        "Covariance Minimum Eigenvalue": np.linalg.eigvalsh(bl_result["Sigma_historical"]).min(),
    },
    shrinkage_method: {
        "CAGR": shrink_metrics["CAGR"],
        "Annualized Volatility": shrink_metrics["Annualized Volatility"],
        "Annualized Sharpe Ratio": shrink_metrics["Annualized Sharpe Ratio"],
        "Maximum Drawdown": shrink_metrics["Maximum Drawdown"],
        "Maximum Weight": shrink_weights.max(),
        "Active Share vs Sample": 0.5 * np.abs(shrink_weights - base_weights).sum(),
        "Shrinkage Intensity": shrinkage_intensity,
        "Covariance Condition Number": np.linalg.cond(shrink_covariance),
        "Covariance Minimum Eigenvalue": np.linalg.eigvalsh(shrink_covariance).min(),
    },
})
covariance_robustness.to_csv(DATA_OUTPUT_DIR / "covariance_shrinkage_robustness.csv", encoding=CSV_ENCODING)
print(
    "=== Covariance shrinkage robustness: {} ({}) ===".format(
        shrinkage_method, format_display_value(shrinkage_intensity, style="percent")
    )
)
display(format_metric_table(
    covariance_robustness,
    percent_rows=[
        "CAGR", "Annualized Volatility",
        "Maximum Drawdown", "Maximum Weight",
        "Active Share vs Sample", "Shrinkage Intensity",
    ],
    ratio_rows=[
        "Annualized Sharpe Ratio",
        "Covariance Condition Number",
    ],
    scientific_rows=["Covariance Minimum Eigenvalue"],
))


def calculate_active_diagnostics(strategy_weights, benchmark_weights, strategy_returns, comparison_returns, benchmark_name):
    """
    Compute active share, model-implied initial active risk, realized active return, tracking error, information ratio and relative-wealth drawdown versus a benchmark.
    """
    assets = data["asset_returns"].columns
    strategy_weights = strategy_weights.reindex(assets).astype(float)
    benchmark_weights = benchmark_weights.reindex(assets).astype(float)
    benchmark_weights = benchmark_weights / benchmark_weights.sum()
    active_weights = strategy_weights - benchmark_weights
    covariance = data["asset_returns"].cov().reindex(index=assets, columns=assets).to_numpy()
    active_series = pd.concat([
        strategy_returns.rename("strategy"), comparison_returns.rename("benchmark")
    ], axis=1).dropna()
    active_returns = active_series["strategy"] - active_series["benchmark"]
    tracking_error = active_returns.std(ddof=1) * np.sqrt(252.0)
    active_return = active_returns.mean() * 252.0
    strategy_wealth = (1.0 + active_series["strategy"]).cumprod()
    benchmark_wealth = (1.0 + active_series["benchmark"]).cumprod()
    relative_wealth = strategy_wealth / benchmark_wealth
    relative_wealth_peaks = np.maximum.accumulate(
        np.r_[1.0, relative_wealth.to_numpy(dtype=float)]
    )[1:]
    maximum_relative_wealth_drawdown = float(
        np.min(relative_wealth.to_numpy(dtype=float) / relative_wealth_peaks - 1.0)
    )
    return {
        "Benchmark": benchmark_name,
        "Active Share": 0.5 * np.abs(active_weights).sum(),
        "Retrospective Model-Implied Initial Active Risk": float(np.sqrt(active_weights.to_numpy() @ covariance @ active_weights.to_numpy()) * np.sqrt(252.0)),
        "Realised Active Return": active_return,
        "Realised Tracking Error": tracking_error,
        "Information Ratio": active_return / tracking_error if tracking_error > 0 else np.nan,
        "Maximum Relative-Wealth Drawdown": maximum_relative_wealth_drawdown,
    }


equal_weights = main_equal_weights
aum_weighted_returns = main_returns["AUM-Weighted Prior"]
equal_weight_buy_hold_returns = main_returns["Equal-Weight"]
views_off_returns = main_returns["Views-Off Neutral MVO"]
active_risk_diagnostics = pd.DataFrame([
    calculate_active_diagnostics(base_weights, data["market_weights"], port_returns, aum_weighted_returns, "AUM-weighted prior"),
    calculate_active_diagnostics(base_weights, equal_weights, port_returns, equal_weight_buy_hold_returns, "Equal-weight buy-and-hold"),
    calculate_active_diagnostics(base_weights, view_ablation_weights["Views-Off Neutral MVO"], port_returns, views_off_returns, "Views-off neutral MVO"),
]).set_index("Benchmark")
active_risk_diagnostics.to_csv(DATA_OUTPUT_DIR / "active_risk_diagnostics.csv", encoding=CSV_ENCODING)
print("=== Active risk diagnostics (retrospective) ===")
display(format_metric_table(
    active_risk_diagnostics.T,
    percent_rows=[
        "Active Share", "Retrospective Model-Implied Initial Active Risk",
        "Realised Active Return", "Realised Tracking Error",
        "Maximum Relative-Wealth Drawdown",
    ],
    ratio_rows=["Information Ratio"],
))


def calculate_risk_contribution_table(weight_dict, covariance_matrix):
    """
    Align each portfolio to the common asset order and return normalized asset-level variance risk contributions under the supplied covariance matrix.
    """
    covariance_matrix = covariance_matrix.reindex(index=data["asset_returns"].columns, columns=data["asset_returns"].columns)
    rows = {}
    for name, weights in weight_dict.items():
        weights = weights.reindex(covariance_matrix.index).astype(float)
        marginal = covariance_matrix.to_numpy() @ weights.to_numpy()
        total = weights.to_numpy() * marginal
        rows[name] = pd.Series(total / total.sum(), index=covariance_matrix.index)
    return pd.DataFrame(rows)


risk_contribution_table = calculate_risk_contribution_table({
    "Trustee / Conservative MV": mv_weights[TRUSTEE_LABEL],
    "Market / Neutral MV": mv_weights[MARKET_LABEL],
    "Kelly-style / Aggressive MV": mv_weights[KELLY_STYLE_LABEL],
    "Max Sharpe": maxsharpe_w,
    "ERC": erc_w,
}, bl_result["Sigma_post"])
risk_contribution_table.to_csv(DATA_OUTPUT_DIR / "risk_contributions.csv", encoding=CSV_ENCODING)
print("=== Model-implied risk contributions ===")
risk_contribution_display = risk_contribution_table.apply(
    lambda column: column.map(lambda value: format_display_value(value, style="percent"))
)
display(risk_contribution_display)

constrained_weight_table = portfolio_weights.drop(columns=["Official AUM Prior"])
maximum_weight_sum_error = float((constrained_weight_table.sum(axis=0) - 1.0).abs().max())
maximum_bound_violation = float(max(
    0.0,
    -constrained_weight_table.min().min(),
    constrained_weight_table.max().max() - 0.20,
))
no_view_numerical_check = bl_model.posterior([], weight_scheme="aum_proxy")
no_view_mean_error = float((
    no_view_numerical_check["mu_post"] - no_view_numerical_check["pi_prior"]
).abs().max())
if no_view_mean_error >= 1e-12:
    raise AssertionError("No-view Black-Litterman posterior does not reproduce the prior.")
erc_rc_deviation = float((risk_contribution_table["ERC"] - 1.0 / len(risk_contribution_table)).abs().max())
numerical_qa = pd.DataFrame([
    ["Historical covariance min eigenvalue", np.linalg.eigvalsh(bl_result["Sigma_historical"]).min(), "> 0"],
    ["Predictive covariance min eigenvalue", np.linalg.eigvalsh(bl_result["Sigma_post"]).min(), "> 0"],
    ["Historical covariance condition number", np.linalg.cond(bl_result["Sigma_historical"]), "reported"],
    ["BL linear-solve relative residual", bl_result["linear_solve_relative_residual"], "< 1e-10"],
    ["No-view max absolute posterior-prior error", no_view_mean_error, "< 1e-12"],
    ["Maximum constrained weight-sum error", maximum_weight_sum_error, "< 1e-7"],
    ["Maximum constrained bound violation", maximum_bound_violation, "< 1e-7"],
    ["ERC max normalised-RC deviation from 1/N", erc_rc_deviation, "reported"],
], columns=["Numerical QA check", "Value", "Acceptance and interpretation"])
if not (
    numerical_qa.loc[numerical_qa["Numerical QA check"].str.contains("eigenvalue"), "Value"] > 0
).all():
    raise AssertionError("A covariance matrix failed the positive-eigenvalue QA check.")
if bl_result["linear_solve_relative_residual"] >= 1e-10:
    raise AssertionError("Black-Litterman linear-system residual exceeds tolerance.")
if maximum_weight_sum_error >= 1e-7 or maximum_bound_violation >= 1e-7:
    raise AssertionError("Optimised weights failed feasibility QA.")

numerical_qa.to_csv(DATA_OUTPUT_DIR / "numerical_qa.csv", index=False, encoding=CSV_ENCODING)
print("=== Numerical QA ===")
numerical_qa_display = numerical_qa.copy().astype(object)
condition_number_mask = numerical_qa_display["Numerical QA check"].str.contains("condition number")
numerical_qa_display.loc[condition_number_mask, "Value"] = numerical_qa.loc[
    condition_number_mask, "Value"
].map(lambda value: format_display_value(value, style="ratio"))
numerical_qa_display.loc[~condition_number_mask, "Value"] = numerical_qa.loc[
    ~condition_number_mask, "Value"
].map(lambda value: format_display_value(value, style="scientific"))
display(numerical_qa_display)

plt.figure(figsize=(12, 6))
risk_contribution_table.T.plot(kind="bar", stacked=True, cmap="tab20", ax=plt.gca())
plt.title("Model-Implied Risk Contributions Across Allocations")
plt.ylabel("Risk Contribution")
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_OUTPUT_DIR / "risk_contributions_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


def apply_initial_transaction_cost(returns, one_way_cost, initial_turnover=1.0):
    """
    Apply a proportional one-way implementation cost to the first return observation only, using the specified initial turnover.
    """
    adjusted = pd.Series(returns, dtype=float).copy()
    adjusted.iloc[0] = (1.0 + adjusted.iloc[0]) * (1.0 - one_way_cost * initial_turnover) - 1.0
    return adjusted


cost_rows = {}
for one_way_cost in [0.001, 0.003, 0.005]:
    net_returns = apply_initial_transaction_cost(port_returns, one_way_cost)
    net_metrics = calculate_performance_metrics_from_returns(net_returns, data["rf_daily"])
    cost_rows[f"{one_way_cost:.1%} one-way"] = {
        "Net Cumulative Return": net_metrics["Cumulative Return"],
        "Net CAGR": net_metrics["CAGR"],
        "Net Sharpe Ratio": net_metrics["Annualized Sharpe Ratio"],
        "CAGR Erosion": port_metrics["CAGR"] - net_metrics["CAGR"],
    }
transaction_cost_impact = pd.DataFrame(cost_rows)
transaction_cost_impact.to_csv(DATA_OUTPUT_DIR / "transaction_cost_impact.csv", encoding=CSV_ENCODING)
print("=== Initial implementation-cost sensitivity ===")
display(format_metric_table(
    transaction_cost_impact,
    percent_rows=["Net Cumulative Return", "Net CAGR", "CAGR Erosion"],
    ratio_rows=["Net Sharpe Ratio"],
))


# Five-year results remain retrospective: views and current AUM are calibrated over the full five-year sample.
robustness_5y_views, robustness_5y_view_diagnostics = calibrate_bl_views(
    robustness_data["asset_excess"], robustness_data["market_weights"], "aum_proxy"
)
robustness_5y_model = BlackLittermanModel(
    robustness_data["asset_excess"], robustness_data["market_weights"], tau=0.025, risk_aversion_market=3.0
)
assert_relative_view_alignment(robustness_5y_model, robustness_5y_views, robustness_data["market_weights"], "aum_proxy")
robustness_5y_bl = robustness_5y_model.posterior(robustness_5y_views, weight_scheme="aum_proxy")
robustness_5y_views_off = robustness_5y_model.posterior([], weight_scheme="aum_proxy")
robustness_5y_weights = PortfolioOptimizer(
    robustness_5y_bl["mu_post"], robustness_5y_bl["Sigma_post"]
).mean_variance(risk_aversion=3.0)
robustness_5y_views_off_weights = PortfolioOptimizer(
    robustness_5y_views_off["mu_post"], robustness_5y_views_off["Sigma_post"]
).mean_variance(risk_aversion=3.0)
robustness_equal_weights = pd.Series(
    1.0 / robustness_data["asset_returns"].shape[1], index=robustness_data["asset_returns"].columns
)
robustness_5y_returns = pd.DataFrame({
    "BL Views-On": BacktestEngine(robustness_5y_weights, robustness_data["asset_returns"]).calc_portfolio_returns(),
    "Views-Off Neutral MVO": BacktestEngine(
        robustness_5y_views_off_weights, robustness_data["asset_returns"]
    ).calc_portfolio_returns(),
    "Equal-Weight": BacktestEngine(robustness_equal_weights, robustness_data["asset_returns"]).calc_portfolio_returns(),
    "AUM-Weighted Prior": BacktestEngine(robustness_data["market_weights"], robustness_data["asset_returns"]).calc_portfolio_returns(),
    "SPY External Reference": robustness_data["factor_returns"]["Market"],
}).dropna()
robustness_5y_metrics = pd.DataFrame({
    column: calculate_performance_metrics_from_returns(
        robustness_5y_returns[column], robustness_data["rf_daily"].reindex(robustness_5y_returns.index)
    )
    for column in robustness_5y_returns.columns
})
robustness_5y_weight_table = pd.DataFrame({
    "BL Views-On Weight": robustness_5y_weights,
    "Views-Off Neutral MVO Weight": robustness_5y_views_off_weights,
    "Official AUM Weight": robustness_data["market_weights"],
    "Equal Weight": robustness_equal_weights,
})
robustness_5y_metrics.to_csv(DATA_OUTPUT_DIR / "robustness_5y_retrospective_metrics.csv", encoding=CSV_ENCODING)
robustness_5y_returns.to_csv(DATA_OUTPUT_DIR / "robustness_5y_retrospective_returns.csv", index_label="Date", encoding=CSV_ENCODING)
robustness_5y_weight_table.to_csv(DATA_OUTPUT_DIR / "robustness_5y_weights.csv", encoding=CSV_ENCODING)
robustness_5y_view_diagnostics.to_csv(DATA_OUTPUT_DIR / "robustness_5y_view_calibration.csv", index=False, encoding=CSV_ENCODING)
print("=== Five-year retrospective robustness ===")
display(format_metric_table(
    robustness_5y_metrics,
    percent_rows=[
        "Cumulative Return", "CAGR",
        "Annualized Excess Return", "Annualized Volatility",
        "Maximum Drawdown",
    ],
    ratio_rows=[
        "Annualized Sharpe Ratio", "Annualized Sortino Ratio",
        "Calmar Ratio",
    ],
    integer_rows=["Observations"],
))


## Retrospective walk-forward evaluation (pseudo-OOS)


In [ ]:

# ===================== 3.8.2 Nested point-in-time walk-forward =====================
OOS_TRAIN_WINDOW = 756
OOS_ONE_WAY_COST = 0.001


def run_oos_walk_forward_bl(data_bundle, train_window=756, one_way_cost=0.001):
    """
    Run monthly point-in-time Black-Litterman walk-forward ablations and bounded stress scenarios using trailing-window estimates, a one-close implementation lag and audited transaction costs.
    Return gross and net returns plus turnover, calibration, target-weight and solver-audit tables.
    """
    asset_returns = data_bundle["asset_returns"].copy().sort_index()
    asset_excess = data_bundle["asset_excess"].reindex(asset_returns.index).copy()
    risk_free = data_bundle["rf_daily"].reindex(asset_returns.index).copy()
    month_ends = asset_returns.groupby(asset_returns.index.to_period("M")).tail(1).index
    equal_prior = pd.Series(1.0 / asset_returns.shape[1], index=asset_returns.columns)

    strategy_names = [
        "Views-Off Neutral MVO", "Absolute Only", "Relative Only", "Both Views",
        "Zero-Return Relative View", "Adverse Relative", "Adverse Absolute",
        "Joint Adverse", "Monthly Equal",
    ]
    histories = {name: {"Gross": [], "Net": []} for name in strategy_names}
    previous_end_weights = {name: None for name in strategy_names}
    return_dates = []
    turnover_rows = []
    calibration_rows = []
    target_weight_rows = []
    solver_rows = []

    for month_index, rebalance_date in enumerate(month_ends[:-1]):
        position = asset_returns.index.get_loc(rebalance_date)
        if position + 1 < train_window:
            continue
        next_rebalance = month_ends[month_index + 1]
        holding_dates = asset_returns.loc[
            (asset_returns.index > rebalance_date) & (asset_returns.index <= next_rebalance)
        ].index
        if len(holding_dates) == 0:
            continue
        if len(holding_dates) < 2:
            raise AssertionError("A monthly OOS holding window is too short for the one-day implementation lag.")

        train_excess = asset_excess.iloc[position - train_window + 1:position + 1]
        if len(train_excess) != train_window:
            raise AssertionError(f"OOS training window is not exactly {train_window} observations.")
        if not (train_excess.index.max() == rebalance_date < holding_dates.min()):
            raise AssertionError("Training and holding windows overlap or are misordered.")

        calibrated_views, calibration_diagnostics = calibrate_bl_views(
            train_excess, equal_prior, weight_scheme="equal"
        )
        model = BlackLittermanModel(
            train_excess, equal_prior, tau=0.025, risk_aversion_market=3.0
        )
        assert_relative_view_alignment(model, calibrated_views, equal_prior, "equal")
        scenario_views = build_view_scenarios(calibrated_views)

        targets = {}
        for scenario_name, views in scenario_views.items():
            result = model.posterior(views, weight_scheme="equal")
            optimizer = PortfolioOptimizer(result["mu_post"], result["Sigma_post"])
            target = optimizer.mean_variance(risk_aversion=3.0)
            targets[scenario_name] = target.reindex(asset_returns.columns)
            solver_info = dict(optimizer.last_solver_info)
            weight_sum_error = abs(float(target.sum()) - 1.0)
            bound_violation = max(0.0, optimizer.w_min - float(target.min()), float(target.max()) - optimizer.w_max)
            minimum_eigenvalue = float(np.linalg.eigvalsh(result["Sigma_post"]).min())
            condition_number = float(np.linalg.cond(result["Sigma_post"]))
            linear_residual = float(result["linear_solve_relative_residual"])
            no_view_error = (
                float((result["mu_post"] - result["pi_prior"]).abs().max())
                if scenario_name == "Views-Off Neutral MVO" else np.nan
            )
            if not solver_info.get("success", False):
                raise AssertionError(f"OOS optimizer did not report success for {scenario_name} at {rebalance_date}.")
            if (weight_sum_error >= 1e-7 or bound_violation >= 1e-7 or minimum_eigenvalue <= 0.0
                    or not np.isfinite(condition_number) or linear_residual >= 1e-10
                    or (scenario_name == "Views-Off Neutral MVO" and no_view_error >= 1e-12)):
                raise AssertionError(f"OOS numerical QA failed for {scenario_name} at {rebalance_date}.")
            solver_rows.append({
                "Rebalance Date": rebalance_date, "Strategy": scenario_name,
                "Method": solver_info.get("method"), "Status": solver_info.get("status"),
                "Success": bool(solver_info.get("success")),
                "Weight Sum Error": weight_sum_error, "Bound Violation": bound_violation,
                "Predictive Covariance Min Eigenvalue": minimum_eigenvalue,
                "Predictive Covariance Condition Number": condition_number,
                "BL Linear Solve Relative Residual": linear_residual,
                "No-View Posterior-Prior Error": no_view_error,
            })
        targets["Monthly Equal"] = equal_prior.copy()
        solver_rows.append({
            "Rebalance Date": rebalance_date, "Strategy": "Monthly Equal",
            "Method": "Deterministic rule", "Status": "not applicable", "Success": True,
            "Weight Sum Error": abs(float(equal_prior.sum()) - 1.0), "Bound Violation": 0.0,
            "Predictive Covariance Min Eigenvalue": np.nan,
            "Predictive Covariance Condition Number": np.nan,
            "BL Linear Solve Relative Residual": np.nan,
            "No-View Posterior-Prior Error": np.nan,
        })

        for name, target in targets.items():
            if target.isna().any() or not np.isclose(target.sum(), 1.0, atol=1e-7):
                raise AssertionError(f"Invalid target weights for {name} at {rebalance_date}")

        current_weights = {
            name: (
                None if previous_end_weights[name] is None
                else previous_end_weights[name].copy()
            )
            for name in strategy_names
        }
        calibration_indexed = calibration_diagnostics.set_index("View")
        absolute_label = "Absolute XLK scenario"
        relative_label = "Low-minus-high carbon sector scenario"
        calibration_rows.append({
            "Signal Formation Date": rebalance_date,
            "Implementation Date": holding_dates[0],
            "Rebalance Date": rebalance_date,
            "Training Start": train_excess.index.min(),
            "Training End": train_excess.index.max(),
            "Holding Start": holding_dates.min(),
            "Holding End": holding_dates.max(),
            "Target Risky Return Accrual Start": holding_dates[1],
            "Implementation Lag Trading Days": 1,
            "Training Observations": int(len(train_excess)),
            "Absolute Q Annual": float(calibrated_views[0]["return"]),
            "Absolute Confidence": float(calibrated_views[0]["confidence"]),
            "Absolute HAC p-value": float(calibration_indexed.loc[absolute_label, "HAC p-value"]),
            "Relative Q Annual": float(calibrated_views[1]["return"]),
            "Relative Confidence": float(calibrated_views[1]["confidence"]),
            "Relative HAC p-value": float(calibration_indexed.loc[relative_label, "HAC p-value"]),
            "Relative Sign Stable": bool(calibration_indexed.loc[relative_label, "Sign Stable"]),
            "Prior": "Equal-weight weights fixed across dates; rolling implied returns",
            "P-Q Weight Match": True,
        })

        for holding_date_index, date in enumerate(holding_dates):
            return_dates.append(date)
            daily_asset_returns = asset_returns.loc[date]
            for name in strategy_names:
                if holding_date_index == 0:
                    if current_weights[name] is None:
                        gross_return = float(risk_free.loc[date])
                        pre_trade_weights = pd.Series(0.0, index=asset_returns.columns)
                        turnover = 1.0
                        traded_notional = 1.0
                    else:
                        gross_return = float(current_weights[name] @ daily_asset_returns)
                        drifted_weights = current_weights[name] * (1.0 + daily_asset_returns)
                        drifted_weights = drifted_weights / drifted_weights.sum()
                        pre_trade_weights = drifted_weights
                        turnover = float(0.5 * np.abs(targets[name] - drifted_weights).sum())
                        traded_notional = 2.0 * turnover
                    cost_rate = one_way_cost * traded_notional
                    signed_trade = targets[name] - pre_trade_weights
                    for ticker in asset_returns.columns:
                        target_weight_rows.append({
                            "Signal Formation Date": rebalance_date,
                            "Implementation Date": date,
                            "Target Effective Date": holding_dates[1],
                            "Rebalance Date": rebalance_date,
                            "Strategy": name,
                            "Ticker": ticker,
                            "Pre-Trade Weight": float(pre_trade_weights[ticker]),
                            "Target Weight": float(targets[name][ticker]),
                            "Signed Trade Weight": float(signed_trade[ticker]),
                            "Absolute Trade Weight": float(abs(signed_trade[ticker])),
                        })
                    current_weights[name] = targets[name].copy()
                    turnover_rows.append({
                        "Signal Formation Date": rebalance_date,
                        "Rebalance Date": rebalance_date,
                        "Implementation Date": date,
                        "Strategy": name,
                        "Training Start": train_excess.index.min(),
                        "Training End": train_excess.index.max(),
                        "Holding Start": holding_dates.min(),
                        "Holding End": holding_dates.max(),
                        "Target Risky Return Accrual Start": holding_dates[1],
                        "Implementation Lag Trading Days": 1,
                        "Training Observations": int(len(train_excess)),
                        "Turnover": float(turnover),
                        "Traded Notional": float(traded_notional),
                        "One-way Cost": float(one_way_cost),
                        "Cost Rate Charged": float(cost_rate),
                        "Prior": "Equal-weight weights fixed across dates; rolling implied returns",
                        "View Basket Rule": "Equal-weight legs matching P",
                    })
                else:
                    gross_return = float(current_weights[name] @ daily_asset_returns)
                    cost_rate = 0.0
                    current_weights[name] = current_weights[name] * (1.0 + daily_asset_returns)
                    current_weights[name] = current_weights[name] / current_weights[name].sum()
                net_return = (1.0 + gross_return) * (1.0 - cost_rate) - 1.0
                histories[name]["Gross"].append(gross_return)
                histories[name]["Net"].append(net_return)

        for name in strategy_names:
            previous_end_weights[name] = current_weights[name].copy()

    index = pd.DatetimeIndex(return_dates, name="Date")
    returns_wide = pd.DataFrame({
        f"{name} {kind}": histories[name][kind]
        for name in strategy_names
        for kind in ["Gross", "Net"]
    }, index=index)
    turnover_table = pd.DataFrame(turnover_rows)
    calibration_table = pd.DataFrame(calibration_rows)
    target_weights_table = pd.DataFrame(target_weight_rows)
    solver_audit_table = pd.DataFrame(solver_rows)

    if len(returns_wide) > 0:
        if not returns_wide.index.is_monotonic_increasing or not returns_wide.index.is_unique:
            raise AssertionError("OOS return index must be increasing and unique.")
        if returns_wide.isna().any().any():
            raise AssertionError("OOS return streams contain missing values.")
        signal_dates = pd.to_datetime(turnover_table["Signal Formation Date"])
        implementation_dates = pd.to_datetime(turnover_table["Implementation Date"])
        target_effective_dates = pd.to_datetime(
            turnover_table["Target Risky Return Accrual Start"]
        )
        holding_end_dates = pd.to_datetime(turnover_table["Holding End"])
        if not (
            (signal_dates < implementation_dates)
            & (implementation_dates < target_effective_dates)
            & (target_effective_dates <= holding_end_dates)
        ).all():
            raise AssertionError("OOS signal, implementation and target-effective dates are misordered.")
        if not (
            implementation_dates
            == pd.to_datetime(turnover_table["Holding Start"])
        ).all():
            raise AssertionError("OOS implementation dates must equal holding-window starts.")
        rebalance_count = calibration_table["Rebalance Date"].nunique()
        if len(turnover_table) != rebalance_count * len(strategy_names):
            raise AssertionError("OOS turnover audit must contain one row per strategy and signal date.")
        for name, strategy_turnover in turnover_table.groupby("Strategy", sort=False):
            strategy_turnover = strategy_turnover.sort_values("Implementation Date")
            first_trade = strategy_turnover.iloc[0]
            if not (
                np.isclose(first_trade["Turnover"], 1.0, atol=1e-12)
                and np.isclose(first_trade["Traded Notional"], 1.0, atol=1e-12)
            ):
                raise AssertionError(f"Initial cash-to-risky trade is not fully charged for {name}.")
            ongoing = strategy_turnover.iloc[1:]
            if len(ongoing) and not np.allclose(
                ongoing["Traded Notional"],
                2.0 * ongoing["Turnover"],
                atol=1e-12,
            ):
                raise AssertionError(f"Traded notional does not equal L1 turnover for {name}.")
        weight_sums = target_weights_table.groupby([
            "Rebalance Date", "Strategy"
        ])["Target Weight"].sum()
        if not np.allclose(weight_sums.to_numpy(), 1.0, atol=1e-7):
            raise AssertionError("One or more OOS target portfolios are not fully invested.")
        absolute_trades = target_weights_table.groupby([
            "Rebalance Date", "Strategy"
        ])["Absolute Trade Weight"].sum().sort_index()
        reported_notionals = turnover_table.set_index([
            "Rebalance Date", "Strategy"
        ])["Traded Notional"].sort_index()
        if not absolute_trades.index.equals(reported_notionals.index):
            raise AssertionError("Trade-weight and turnover audit keys do not align.")
        if not np.allclose(absolute_trades, reported_notionals, atol=1e-12):
            raise AssertionError("Absolute trade weights do not reconcile to charged notional.")
        if solver_audit_table.empty or not solver_audit_table["Success"].all():
            raise AssertionError("OOS solver audit is missing or contains a failure.")
    return returns_wide, turnover_table, calibration_table, target_weights_table, solver_audit_table


oos_returns_wide, oos_turnover, oos_view_calibration, oos_target_weights, oos_solver_audit = run_oos_walk_forward_bl(
    robustness_data, train_window=OOS_TRAIN_WINDOW, one_way_cost=OOS_ONE_WAY_COST
)

OOS_BASE_HAC_LAGS = 21
OOS_BASE_BLOCK_LENGTH = 21
OOS_BOOTSTRAP_REPETITIONS = 2000
OOS_BOOTSTRAP_BASE_SEED = 42

if len(oos_returns_wide) > 0:
    spy_gross = robustness_data["factor_returns"]["Market"].reindex(oos_returns_wide.index)
    common_index = oos_returns_wide.index.intersection(spy_gross.dropna().index)
    if not common_index.equals(oos_returns_wide.index):
        raise AssertionError("SPY has missing OOS dates; refusing to silently change the evaluated path.")
    oos_returns_wide = oos_returns_wide.reindex(common_index)
    spy_gross = spy_gross.reindex(common_index).rename("SPY Buy-and-Hold Gross")
    spy_net = apply_initial_transaction_cost(spy_gross, OOS_ONE_WAY_COST).rename("SPY Buy-and-Hold Net")
    oos_returns_wide[spy_gross.name] = spy_gross
    oos_returns_wide[spy_net.name] = spy_net
    oos_rf = robustness_data["rf_daily"].reindex(common_index)

    oos_metrics = pd.DataFrame({
        column: calculate_performance_metrics_from_returns(oos_returns_wide[column], oos_rf)
        for column in oos_returns_wide.columns
        if column.endswith("Net") or column == "Both Views Gross"
    })

    cost_grid_strategies = ["Both Views", "Views-Off Neutral MVO", "Monthly Equal"]
    cost_sensitivity_rows = []
    oos_cost_paths = {}
    for strategy_name in cost_grid_strategies:
        gross_path = oos_returns_wide[f"{strategy_name} Gross"]
        strategy_trades = oos_turnover.loc[
            oos_turnover["Strategy"] == strategy_name
        ].copy()
        for cost_bps in [0, 10, 25, 50]:
            one_way_rate = cost_bps / 10000.0
            charged_rate = pd.Series(0.0, index=gross_path.index)
            for _, trade in strategy_trades.iterrows():
                implementation_date = pd.Timestamp(trade["Implementation Date"])
                holding_start = pd.Timestamp(trade["Holding Start"])
                if implementation_date != holding_start:
                    raise AssertionError("Implementation date must equal the first holding-window date.")
                if implementation_date not in charged_rate.index:
                    raise AssertionError("Implementation date is absent from the OOS return path.")
                charged_rate.loc[implementation_date] = (
                    one_way_rate * float(trade["Traded Notional"])
                )
            cost_adjusted = ((1.0 + gross_path) * (1.0 - charged_rate) - 1.0).rename(
                f"{strategy_name} Net {cost_bps}bps"
            )
            oos_cost_paths[(strategy_name, cost_bps)] = cost_adjusted
            metrics = calculate_performance_metrics_from_returns(cost_adjusted, oos_rf)
            cost_sensitivity_rows.append({
                "Strategy": strategy_name, "One-way Cost (bps)": cost_bps,
                "Cumulative Return": metrics["Cumulative Return"],
                "CAGR": metrics["CAGR"],
                "Sharpe": metrics["Annualized Sharpe Ratio"],
                "Maximum Drawdown": metrics["Maximum Drawdown"],
            })
            if cost_bps == 10 and not np.allclose(
                cost_adjusted.to_numpy(), oos_returns_wide[f"{strategy_name} Net"].to_numpy(), atol=1e-12
            ):
                raise AssertionError(f"Cost-grid reconstruction does not match {strategy_name} at 10 bps.")
    oos_cost_sensitivity = pd.DataFrame(cost_sensitivity_rows).set_index(
        ["Strategy", "One-way Cost (bps)"]
    )
    oos_cost_report = pd.DataFrame({
        cost_bps: {
            "Both Views CAGR": oos_cost_sensitivity.loc[("Both Views", cost_bps), "CAGR"],
            "Views-Off CAGR": oos_cost_sensitivity.loc[("Views-Off Neutral MVO", cost_bps), "CAGR"],
            "Both minus Views-Off CAGR": (
                oos_cost_sensitivity.loc[("Both Views", cost_bps), "CAGR"]
                - oos_cost_sensitivity.loc[("Views-Off Neutral MVO", cost_bps), "CAGR"]
            ),
            "Monthly Equal CAGR": oos_cost_sensitivity.loc[("Monthly Equal", cost_bps), "CAGR"],
            "Both Views Sharpe": oos_cost_sensitivity.loc[("Both Views", cost_bps), "Sharpe"],
            "Views-Off Sharpe": oos_cost_sensitivity.loc[("Views-Off Neutral MVO", cost_bps), "Sharpe"],
            "Monthly Equal Sharpe": oos_cost_sensitivity.loc[("Monthly Equal", cost_bps), "Sharpe"],
        } for cost_bps in [0, 10, 25, 50]
    })
    oos_cost_sensitivity.to_csv(DATA_OUTPUT_DIR / "oos_cost_sensitivity.csv", encoding=CSV_ENCODING)
    oos_cost_report.to_csv(DATA_OUTPUT_DIR / "oos_cost_sensitivity_report.csv", encoding=CSV_ENCODING)

    oos_returns_wide.to_csv(DATA_OUTPUT_DIR / "oos_walk_forward_returns.csv", index_label="Date", encoding=CSV_ENCODING)
    oos_metrics.to_csv(DATA_OUTPUT_DIR / "oos_walk_forward_metrics.csv", encoding=CSV_ENCODING)
    oos_turnover.to_csv(DATA_OUTPUT_DIR / "oos_walk_forward_turnover.csv", index=False, encoding=CSV_ENCODING)
    oos_view_calibration.to_csv(DATA_OUTPUT_DIR / "oos_view_calibration_audit.csv", index=False, encoding=CSV_ENCODING)
    oos_target_weights.to_csv(DATA_OUTPUT_DIR / "oos_walk_forward_weights.csv", index=False, encoding=CSV_ENCODING)
    oos_solver_audit.to_csv(DATA_OUTPUT_DIR / "oos_solver_audit.csv", index=False, encoding=CSV_ENCODING)

    print("=== Point-in-time walk-forward performance ===")
    selected_metric_columns = [
        "Both Views Gross", "Both Views Net", "Views-Off Neutral MVO Net", "Absolute Only Net",
        "Relative Only Net", "Zero-Return Relative View Net", "Adverse Relative Net",
        "Adverse Absolute Net", "Joint Adverse Net", "Monthly Equal Net", "SPY Buy-and-Hold Net",
    ]
    display(format_metric_table(
        oos_metrics[selected_metric_columns],
        percent_rows=[
            "Cumulative Return", "CAGR",
            "Annualized Excess Return", "Annualized Volatility",
            "Maximum Drawdown",
        ],
        ratio_rows=[
            "Annualized Sharpe Ratio", "Annualized Sortino Ratio",
            "Calmar Ratio",
        ],
        integer_rows=["Observations"],
    ))
    both_views_turnover = oos_turnover.loc[
        oos_turnover["Strategy"] == "Both Views", "Turnover"
    ].reset_index(drop=True)
    print("Initial implementation turnover, Both Views: {}".format(
        format_display_value(both_views_turnover.iloc[0], style="percent")
    ))
    print("Average ongoing monthly turnover, Both Views: {}".format(
        format_display_value(both_views_turnover.iloc[1:].mean(), style="percent")
    ))
    print("OOS relative-Q range: [{}, {}]".format(
        format_display_value(
            oos_view_calibration["Relative Q Annual"].min(), style="percent"
        ),
        format_display_value(
            oos_view_calibration["Relative Q Annual"].max(), style="percent"
        ),
    ))
    print("=== OOS one-way transaction-cost sensitivity ===")
    oos_cost_report_display = oos_cost_report.copy()
    oos_cost_report_display.columns = [f"{int(value)} bps" for value in oos_cost_report_display.columns]
    display(format_metric_table(
        oos_cost_report_display,
        percent_rows=[
            "Both Views CAGR", "Views-Off CAGR", "Both minus Views-Off CAGR",
            "Monthly Equal CAGR",
        ],
        ratio_rows=["Both Views Sharpe", "Views-Off Sharpe", "Monthly Equal Sharpe"],
    ))

    oos_design_summary = pd.DataFrame({
        "Value": {
            "Evaluation start": oos_returns_wide.index.min().date().isoformat(),
            "Evaluation end": oos_returns_wide.index.max().date().isoformat(),
            "Daily observations": int(len(oos_returns_wide)),
            "Monthly rebalances": int(oos_view_calibration["Rebalance Date"].nunique()),
            "Training observations per rebalance": int(OOS_TRAIN_WINDOW),
            "Implementation lag (trading days)": 1,
            "Terminal holding window": "2026-07-01 to 2026-07-23 (cut-off truncated)",
            "Primary matched comparison": "Both Views versus Views-Off Neutral MVO",
            "Matched view-ablation Holm family size": 3,
            "Exploratory adverse-stress Holm family size": 3,
            "Performance-context comparisons": 2,
            "Zero-return relative-view role": "Descriptive boundary scenario; no standalone hypothesis",
            "Execution convention": "Incumbent drifts through implementation day; target effective next close-to-close return",
            "Initial inception convention": "Cash through first implementation close",
            "SPY context timing": "Buy-and-hold from evaluation start; no matched signal lag",
            "Base one-way cost": format_display_value(OOS_ONE_WAY_COST, style="percent"),
            "Both Views average ongoing turnover": format_display_value(
                both_views_turnover.iloc[1:].mean(), style="percent"
            ),
            "Audited solver rows": int(len(oos_solver_audit)),
            "Maximum solver weight-sum error": float(oos_solver_audit["Weight Sum Error"].max()),
            "Maximum solver bound violation": float(oos_solver_audit["Bound Violation"].max()),
            "Maximum BL linear-solve residual": float(oos_solver_audit["BL Linear Solve Relative Residual"].max()),
        }
    })
    report_strategy_columns = [
        "Both Views Net", "Views-Off Neutral MVO Net", "Monthly Equal Net",
        "SPY Buy-and-Hold Net", "Joint Adverse Net",
    ]
    report_metric_rows = [
        "CAGR", "Annualized Volatility",
        "Annualized Sharpe Ratio", "Maximum Drawdown",
    ]
    oos_report_performance = oos_metrics.loc[report_metric_rows, report_strategy_columns]
    oos_design_summary.to_csv(DATA_OUTPUT_DIR / "oos_report_design_summary.csv", encoding=CSV_ENCODING)
    oos_report_performance.to_csv(DATA_OUTPUT_DIR / "oos_report_performance.csv", encoding=CSV_ENCODING)
    print("=== OOS report design summary ===")
    oos_design_summary_display = oos_design_summary.copy().astype(object)
    for qa_label in [
        "Maximum solver weight-sum error", "Maximum solver bound violation",
        "Maximum BL linear-solve residual",
    ]:
        oos_design_summary_display.loc[qa_label, "Value"] = format_display_value(
            oos_design_summary.loc[qa_label, "Value"], style="scientific"
        )
    with pd.option_context("display.max_colwidth", None):
        display(oos_design_summary_display)
    print("=== OOS report performance summary ===")
    display(format_metric_table(
        oos_report_performance,
        percent_rows=[
            "CAGR", "Annualized Volatility",
            "Maximum Drawdown",
        ],
        ratio_rows=["Annualized Sharpe Ratio"],
    ))

    plot_columns = ["Both Views Net", "Views-Off Neutral MVO Net", "Monthly Equal Net", "SPY Buy-and-Hold Net"]
    oos_ax = (1.0 + oos_returns_wide[plot_columns]).cumprod().plot(figsize=(11, 5), linewidth=1.4)
    oos_end = pd.Timestamp(oos_returns_wide.index.max()).date().isoformat()
    oos_ax.set_title(f"Point-in-Time Net Comparators and External SPY Reference (through {oos_end})")
    oos_ax.set_ylabel("Cumulative Wealth")
    oos_ax.set_xlim(oos_returns_wide.index.min(), oos_returns_wide.index.max())
    oos_ax.grid(alpha=0.3)
    oos_ax.figure.tight_layout()
    oos_ax.figure.savefig(FIGURE_OUTPUT_DIR / "oos_walk_forward_performance.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    raise RuntimeError("Point-in-time walk-forward produced no observations after the 756-observation training window.")


## Paired inference and local audit outputs


In [ ]:

# ===================== 3.8.3 Paired HAC and block-bootstrap inference =====================
def annualized_daily_sharpe(returns, risk_free, periods_per_year=252):
    """
    Align daily simple returns with risk-free rates and compute the square-root-of-time annualized Sharpe ratio from daily excess returns.
    """
    panel = pd.concat([
        pd.Series(returns, dtype=float).rename("return"),
        pd.Series(risk_free, dtype=float).rename("rf"),
    ], axis=1).dropna()
    excess = panel["return"] - panel["rf"]
    volatility = excess.std(ddof=1)
    return excess.mean() / volatility * np.sqrt(periods_per_year) if volatility > 0 else np.nan


def paired_performance_test(strategy, benchmark, risk_free, benchmark_name,
                            hac_lags=21, block_length=21, bootstrap_reps=2000, seed=42):
    """
    Compare paired strategy and benchmark returns using HAC inference and a circular block bootstrap.
    Return annualized active-return and Sharpe-difference estimates, confidence intervals and p-values under the stated weak-stationarity assumption.
    """
    panel = pd.concat([
        pd.Series(strategy, dtype=float).rename("strategy"),
        pd.Series(benchmark, dtype=float).rename("benchmark"),
        pd.Series(risk_free, dtype=float).rename("rf"),
    ], axis=1).dropna()
    if len(panel) < max(60, 2 * block_length):
        raise ValueError("Insufficient paired observations for HAC/block-bootstrap inference.")

    active = panel["strategy"] - panel["benchmark"]
    hac_model = sm.OLS(active.to_numpy(), np.ones((len(active), 1))).fit(
        cov_type="HAC", cov_kwds={"maxlags": int(hac_lags)}
    )
    mean_active_annual = float(active.mean() * 252.0)
    hac_se_annual = float(hac_model.bse[0] * 252.0)
    hac_ci = (
        mean_active_annual - 1.96 * hac_se_annual,
        mean_active_annual + 1.96 * hac_se_annual,
    )
    observed_sharpe_difference = (
        annualized_daily_sharpe(panel["strategy"], panel["rf"])
        - annualized_daily_sharpe(panel["benchmark"], panel["rf"])
    )

    values = panel.to_numpy()
    centered_active = active.to_numpy() - active.mean()
    sample_size = len(panel)
    number_of_blocks = int(np.ceil(sample_size / block_length))
    offsets = np.arange(block_length)
    rng = np.random.default_rng(seed)
    active_bootstrap = np.empty(bootstrap_reps)
    null_active_bootstrap = np.empty(bootstrap_reps)
    sharpe_difference_bootstrap = np.empty(bootstrap_reps)

    for repetition in range(bootstrap_reps):
        starts = rng.integers(0, sample_size, size=number_of_blocks)
        indices = ((starts[:, None] + offsets) % sample_size).ravel()[:sample_size]
        sample = values[indices]
        active_bootstrap[repetition] = np.mean(sample[:, 0] - sample[:, 1]) * 252.0
        null_active_bootstrap[repetition] = np.mean(centered_active[indices]) * 252.0
        sample_strategy = pd.Series(sample[:, 0])
        sample_benchmark = pd.Series(sample[:, 1])
        sample_rf = pd.Series(sample[:, 2])
        sharpe_difference_bootstrap[repetition] = (
            annualized_daily_sharpe(sample_strategy, sample_rf)
            - annualized_daily_sharpe(sample_benchmark, sample_rf)
        )

    active_ci = np.quantile(active_bootstrap, [0.025, 0.975])
    sharpe_ci = np.quantile(sharpe_difference_bootstrap, [0.025, 0.975])
    bootstrap_p_value = (
        1.0 + np.sum(np.abs(null_active_bootstrap) >= abs(mean_active_annual))
    ) / (bootstrap_reps + 1.0)
    return pd.Series({
        "Benchmark": benchmark_name,
        "Observations": int(sample_size),
        "Annualized Mean Active Return": mean_active_annual,
        "HAC t-statistic": float(hac_model.tvalues[0]),
        "HAC p-value": float(hac_model.pvalues[0]),
        "HAC CI Lower": hac_ci[0],
        "HAC CI Upper": hac_ci[1],
        "Bootstrap Active CI Lower": float(active_ci[0]),
        "Bootstrap Active CI Upper": float(active_ci[1]),
        "Bootstrap Mean-Active p-value": float(bootstrap_p_value),
        "Sharpe Difference": float(observed_sharpe_difference),
        "Bootstrap Sharpe CI Lower": float(sharpe_ci[0]),
        "Bootstrap Sharpe CI Upper": float(sharpe_ci[1]),
        "HAC Lags": int(hac_lags),
        "Block Length": int(block_length),
        "Bootstrap Repetitions": int(bootstrap_reps),
        "Random Seed": int(seed),
    })


if len(oos_returns_wide) > 0:
    oos_comparisons = [
        ("Both Views Net", "Views-Off Neutral MVO Net"),
        ("Absolute Only Net", "Views-Off Neutral MVO Net"),
        ("Relative Only Net", "Views-Off Neutral MVO Net"),
        ("Both Views Net", "Monthly Equal Net"),
        ("Both Views Net", "SPY Buy-and-Hold Net"),
        ("Both Views Net", "Adverse Relative Net"),
        ("Both Views Net", "Adverse Absolute Net"),
        ("Both Views Net", "Joint Adverse Net"),
    ]
    paired_test_rows = []
    primary_comparison = ("Both Views Net", "Views-Off Neutral MVO Net")
    matched_ablation_family = {
        primary_comparison,
        ("Absolute Only Net", "Views-Off Neutral MVO Net"),
        ("Relative Only Net", "Views-Off Neutral MVO Net"),
    }
    adverse_stress_family = {
        ("Both Views Net", "Adverse Relative Net"),
        ("Both Views Net", "Adverse Absolute Net"),
        ("Both Views Net", "Joint Adverse Net"),
    }
    performance_context_comparisons = {
        ("Both Views Net", "Monthly Equal Net"),
        ("Both Views Net", "SPY Buy-and-Hold Net"),
    }
    for comparison_index, (strategy_name, benchmark_name) in enumerate(oos_comparisons):
        row = paired_performance_test(
            oos_returns_wide[strategy_name],
            oos_returns_wide[benchmark_name],
            oos_rf,
            benchmark_name=benchmark_name,
            hac_lags=OOS_BASE_HAC_LAGS,
            block_length=OOS_BASE_BLOCK_LENGTH,
            bootstrap_reps=OOS_BOOTSTRAP_REPETITIONS,
            seed=OOS_BOOTSTRAP_BASE_SEED + comparison_index,
        )
        row["Strategy"] = strategy_name
        pair = (strategy_name, benchmark_name)
        if pair == primary_comparison:
            row["Inference Role"] = "Primary matched view-effect comparison"
        elif pair in matched_ablation_family:
            row["Inference Role"] = "Matched component ablation"
        elif pair in adverse_stress_family:
            row["Inference Role"] = "Exploratory adverse stress"
        elif pair in performance_context_comparisons:
            row["Inference Role"] = "Performance context"
        else:
            raise AssertionError(f"Undeclared OOS comparison role: {pair}")
        row["Multiplicity Family"] = (
            "Matched view-ablation Holm family" if pair in matched_ablation_family
            else ("Adverse-stress Holm family" if pair in adverse_stress_family else "Outside multiplicity families")
        )
        row["Family Size"] = (
            len(matched_ablation_family) if pair in matched_ablation_family
            else (len(adverse_stress_family) if pair in adverse_stress_family else np.nan)
        )
        paired_test_rows.append(row)

    oos_paired_tests = pd.DataFrame(paired_test_rows).set_index(["Strategy", "Benchmark"])
    primary_mask = pd.Series(
        [pair == primary_comparison for pair in oos_paired_tests.index],
        index=oos_paired_tests.index, dtype=bool,
    )
    matched_ablation_mask = oos_paired_tests["Multiplicity Family"].eq("Matched view-ablation Holm family")
    adverse_stress_mask = oos_paired_tests["Multiplicity Family"].eq("Adverse-stress Holm family")
    performance_context_mask = oos_paired_tests["Inference Role"].eq("Performance context")
    if int(primary_mask.sum()) != 1 or int(matched_ablation_mask.sum()) != 3:
        raise AssertionError("OOS primary/matched-ablation family declaration is incomplete.")
    if int(adverse_stress_mask.sum()) != 3 or int(performance_context_mask.sum()) != 2:
        raise AssertionError("OOS stress/context family declaration is incomplete.")
    adjusted_hac_column = "Within-family Holm-adjusted HAC p-value"
    adjusted_bootstrap_column = (
        "Within-family Holm-adjusted Bootstrap Mean-Active p-value"
    )
    oos_paired_tests[adjusted_hac_column] = np.nan
    oos_paired_tests[adjusted_bootstrap_column] = np.nan
    for family_mask in [matched_ablation_mask, adverse_stress_mask]:
        oos_paired_tests.loc[family_mask, adjusted_hac_column] = multipletests(
            oos_paired_tests.loc[family_mask, "HAC p-value"].to_numpy(), method="holm"
        )[1]
        oos_paired_tests.loc[family_mask, adjusted_bootstrap_column] = multipletests(
            oos_paired_tests.loc[family_mask, "Bootstrap Mean-Active p-value"].to_numpy(),
            method="holm",
        )[1]
        if (
            (oos_paired_tests.loc[family_mask, adjusted_hac_column] + 1e-15
             < oos_paired_tests.loc[family_mask, "HAC p-value"]).any()
            or (oos_paired_tests.loc[family_mask, adjusted_bootstrap_column] + 1e-15
                < oos_paired_tests.loc[family_mask, "Bootstrap Mean-Active p-value"]).any()
        ):
            raise AssertionError("Holm-adjusted p-values cannot be below their raw p-values.")
    oos_paired_tests.to_csv(DATA_OUTPUT_DIR / "oos_paired_active_tests.csv", encoding=CSV_ENCODING)

    print("=== Paired OOS active-return and Sharpe-difference inference ===")
    inference_display = pd.DataFrame(index=oos_paired_tests.index)
    inference_display["Role"] = oos_paired_tests["Inference Role"]
    inference_display["Annualized Active Return"] = oos_paired_tests[
        "Annualized Mean Active Return"
    ].map(lambda value: format_display_value(value, style="percent"))
    inference_display["HAC 95% CI"] = oos_paired_tests.apply(
        lambda row: "[{}, {}]".format(
            format_display_value(row["HAC CI Lower"], style="percent"),
            format_display_value(row["HAC CI Upper"], style="percent"),
        ), axis=1
    )
    inference_display["HAC p-value"] = oos_paired_tests["HAC p-value"].map(
        lambda value: format_display_value(value, style="pvalue")
    )
    inference_display["Bootstrap Active 95% CI"] = oos_paired_tests.apply(
        lambda row: "[{}, {}]".format(
            format_display_value(row["Bootstrap Active CI Lower"], style="percent"),
            format_display_value(row["Bootstrap Active CI Upper"], style="percent"),
        ), axis=1
    )
    inference_display["Bootstrap p-value"] = oos_paired_tests[
        "Bootstrap Mean-Active p-value"
    ].map(lambda value: format_display_value(value, style="pvalue"))
    inference_display["Sharpe Difference"] = oos_paired_tests["Sharpe Difference"].map(
        lambda value: format_display_value(value, style="ratio")
    )
    inference_display["Bootstrap Sharpe 95% CI"] = oos_paired_tests.apply(
        lambda row: "[{}, {}]".format(
            format_display_value(row["Bootstrap Sharpe CI Lower"], style="ratio"),
            format_display_value(row["Bootstrap Sharpe CI Upper"], style="ratio"),
        ), axis=1
    )
    inference_display["Multiplicity Family"] = oos_paired_tests["Multiplicity Family"]
    for source_column, display_column in [
        (adjusted_hac_column, "Within-Family Holm HAC p-value"),
        (adjusted_bootstrap_column, "Within-Family Holm Bootstrap p-value"),
    ]:
        inference_display[display_column] = oos_paired_tests[source_column].map(
            lambda value: format_display_value(
                value, style="pvalue", na_label="Outside family"
            )
        )
    display(inference_display)

    sensitivity_rows = []
    sensitivity_comparisons = [
        ("Both Views Net", "Views-Off Neutral MVO Net"),
        ("Absolute Only Net", "Views-Off Neutral MVO Net"),
        ("Relative Only Net", "Views-Off Neutral MVO Net"),
    ]
    for hac_lags, block_length in [(10, 10), (21, 21), (42, 42)]:
        specification_rows = []
        for comparison_index, (strategy_name, benchmark_name) in enumerate(sensitivity_comparisons):
            comparison_pair = (strategy_name, benchmark_name)
            if (hac_lags, block_length) == (OOS_BASE_HAC_LAGS, OOS_BASE_BLOCK_LENGTH):
                sensitivity = oos_paired_tests.loc[comparison_pair].copy()
            else:
                sensitivity = paired_performance_test(
                    oos_returns_wide[strategy_name],
                    oos_returns_wide[benchmark_name],
                    oos_rf,
                    benchmark_name=benchmark_name,
                    hac_lags=hac_lags,
                    block_length=block_length,
                    bootstrap_reps=OOS_BOOTSTRAP_REPETITIONS,
                    seed=OOS_BOOTSTRAP_BASE_SEED + comparison_index,
                )
            specification_rows.append({
                "Strategy": strategy_name,
                "Benchmark": benchmark_name,
                "HAC Lags": hac_lags,
                "Block Length": block_length,
                "Bootstrap Repetitions": int(sensitivity["Bootstrap Repetitions"]),
                "Random Seed": int(sensitivity["Random Seed"]),
                "Annualized Mean Active Return": sensitivity["Annualized Mean Active Return"],
                "Raw HAC p-value": sensitivity["HAC p-value"],
                "Raw Bootstrap p-value": sensitivity["Bootstrap Mean-Active p-value"],
                "Bootstrap CI Lower": sensitivity["Bootstrap Active CI Lower"],
                "Bootstrap CI Upper": sensitivity["Bootstrap Active CI Upper"],
            })
        specification_hac_adjusted = multipletests(
            [row["Raw HAC p-value"] for row in specification_rows], method="holm"
        )[1]
        specification_bootstrap_adjusted = multipletests(
            [row["Raw Bootstrap p-value"] for row in specification_rows], method="holm"
        )[1]
        for row, hac_adjusted, bootstrap_adjusted in zip(
            specification_rows, specification_hac_adjusted, specification_bootstrap_adjusted
        ):
            row["Within-specification Holm HAC p-value"] = float(hac_adjusted)
            row["Within-specification Holm Bootstrap p-value"] = float(bootstrap_adjusted)
            sensitivity_rows.append(row)
    oos_inference_sensitivity = pd.DataFrame(sensitivity_rows)
    for comparison_pair in sensitivity_comparisons:
        baseline_row = oos_inference_sensitivity.loc[
            (oos_inference_sensitivity["Strategy"] == comparison_pair[0])
            & (oos_inference_sensitivity["Benchmark"] == comparison_pair[1])
            & (oos_inference_sensitivity["HAC Lags"] == OOS_BASE_HAC_LAGS)
            & (oos_inference_sensitivity["Block Length"] == OOS_BASE_BLOCK_LENGTH)
        ].iloc[0]
        main_row = oos_paired_tests.loc[comparison_pair]
        np.testing.assert_allclose(
            [
                baseline_row["Annualized Mean Active Return"],
                baseline_row["Raw HAC p-value"],
                baseline_row["Raw Bootstrap p-value"],
                baseline_row["Bootstrap CI Lower"],
                baseline_row["Bootstrap CI Upper"],
            ],
            [
                main_row["Annualized Mean Active Return"],
                main_row["HAC p-value"],
                main_row["Bootstrap Mean-Active p-value"],
                main_row["Bootstrap Active CI Lower"],
                main_row["Bootstrap Active CI Upper"],
            ],
            rtol=0.0, atol=0.0,
        )
        if (
            baseline_row["Bootstrap Repetitions"] != main_row["Bootstrap Repetitions"]
            or baseline_row["Random Seed"] != main_row["Random Seed"]
        ):
            raise AssertionError('Baseline sensitivity bootstrap metadata differs from the main test.')
    oos_inference_sensitivity.to_csv(
        DATA_OUTPUT_DIR / "oos_inference_sensitivity.csv", index=False, encoding=CSV_ENCODING
    )
    print("=== OOS HAC lag and bootstrap block sensitivity ===")
    oos_inference_sensitivity_display = oos_inference_sensitivity.copy().astype(object)
    for column in ["Annualized Mean Active Return", "Bootstrap CI Lower", "Bootstrap CI Upper"]:
        oos_inference_sensitivity_display[column] = oos_inference_sensitivity[column].map(
            lambda value: format_display_value(value, style="percent")
        )
    for column in ["HAC Lags", "Block Length", "Bootstrap Repetitions", "Random Seed"]:
        oos_inference_sensitivity_display[column] = oos_inference_sensitivity[column].map(
            lambda value: format_display_value(value, style="integer")
        )
    for column in [
        "Raw HAC p-value", "Raw Bootstrap p-value",
        "Within-specification Holm HAC p-value",
        "Within-specification Holm Bootstrap p-value",
    ]:
        oos_inference_sensitivity_display[column] = oos_inference_sensitivity[column].map(
            lambda value: format_display_value(value, style="pvalue")
        )
    display(oos_inference_sensitivity_display)

    cost_inference_rows = []
    primary_result = oos_paired_tests.loc[primary_comparison].copy()
    for cost_bps in [0, 10, 25, 50]:
        cost_strategy = pd.Series(oos_cost_paths[("Both Views", cost_bps)], dtype=float)
        cost_benchmark = pd.Series(oos_cost_paths[("Views-Off Neutral MVO", cost_bps)], dtype=float)
        if cost_bps == 10:
            if not cost_strategy.index.equals(oos_returns_wide.index):
                raise AssertionError('The 10 bp cost path index differs from the primary OOS index.')
            np.testing.assert_allclose(
                cost_strategy.to_numpy(), oos_returns_wide["Both Views Net"].to_numpy(),
                rtol=0.0, atol=1e-15,
            )
            np.testing.assert_allclose(
                cost_benchmark.to_numpy(), oos_returns_wide["Views-Off Neutral MVO Net"].to_numpy(),
                rtol=0.0, atol=1e-15,
            )
            result = primary_result
        else:
            result = paired_performance_test(
                cost_strategy,
                cost_benchmark,
                oos_rf, benchmark_name=f"Views-Off Neutral MVO at {cost_bps} bps",
                hac_lags=OOS_BASE_HAC_LAGS, block_length=OOS_BASE_BLOCK_LENGTH,
                bootstrap_reps=OOS_BOOTSTRAP_REPETITIONS, seed=OOS_BOOTSTRAP_BASE_SEED,
            )
        cost_inference_rows.append({
            "One-way Cost (bps)": cost_bps,
            "Benchmark": "Views-Off Neutral MVO",
            "Inference Role": "Primary matched comparison under alternative cost assumption",
            "Annualized Mean Active Return": result["Annualized Mean Active Return"],
            "HAC p-value": result["HAC p-value"],
            "HAC CI Lower": result["HAC CI Lower"],
            "HAC CI Upper": result["HAC CI Upper"],
            "Bootstrap p-value": result["Bootstrap Mean-Active p-value"],
            "HAC Lags": int(result["HAC Lags"]),
            "Block Length": int(result["Block Length"]),
            "Bootstrap Repetitions": int(result["Bootstrap Repetitions"]),
            "Random Seed": int(result["Random Seed"]),
        })
    oos_cost_grid_inference = pd.DataFrame(cost_inference_rows)
    baseline_sensitivity_p = oos_inference_sensitivity.loc[
        (oos_inference_sensitivity["Strategy"] == primary_comparison[0])
        & (oos_inference_sensitivity["Benchmark"] == primary_comparison[1])
        & (oos_inference_sensitivity["HAC Lags"] == OOS_BASE_HAC_LAGS)
        & (oos_inference_sensitivity["Block Length"] == OOS_BASE_BLOCK_LENGTH),
        "Raw Bootstrap p-value",
    ].iloc[0]
    baseline_cost_p = oos_cost_grid_inference.loc[
        oos_cost_grid_inference["One-way Cost (bps)"] == 10, "Bootstrap p-value"
    ].iloc[0]
    primary_bootstrap_p = primary_result["Bootstrap Mean-Active p-value"]
    if not (baseline_sensitivity_p == primary_bootstrap_p == baseline_cost_p):
        raise AssertionError('Identical baseline bootstrap specifications produced inconsistent p-values.')
    oos_cost_grid_inference.to_csv(
        DATA_OUTPUT_DIR / "oos_cost_grid_inference.csv", index=False, encoding=CSV_ENCODING
    )
else:
    oos_paired_tests = pd.DataFrame()
    oos_inference_sensitivity = pd.DataFrame()

numerical_techniques = pd.DataFrame([
    ["BL posterior", "Custom", "NumPy solve", "linear residual; no-view identity", "diagonal Omega"],
    ["MVO", "Custom + library", "CVXPY / SLSQP", "status; feasibility residual", "mean sensitivity"],
    ["Max-Sharpe BL", "Custom + library", "SciPy SLSQP, 5 starts", "status; objective dispersion", "non-convex ratio"],
    ["ERC comparator", "Custom + library", "SciPy SLSQP, 5 starts", "status; RC dispersion", "ignores expected return"],
    ["Covariance-shrinkage sensitivity", "Library", shrinkage_method, "intensity; eigenvalue; condition number", "not base covariance"],
    ["Walk-forward and costs", "Custom", "pandas / NumPy", "756 points; one-day lag; matched 0/10/25/50 bps", "research-selection bias"],
    ["HAC inference", "Library", "statsmodels", "95% CI; 10/21/42 lags", "finite-sample approximation"],
    ["Circular bootstrap", "Custom", "NumPy RNG", "paired paths; 10/21/42 blocks; 2,000 reps; shared seeds; baseline reuse", "weak stationarity"],
    ["Holm correction", "Library", "statsmodels", "three matched ablations; three adverse stresses; within-specification sensitivity", "reduced power"],
], columns=["Method", "Implementation", "Solver or package", "Validation", "Main limitation"])
numerical_techniques.to_csv(DATA_OUTPUT_DIR / "numerical_techniques_used.csv", index=False, encoding=CSV_ENCODING)
print("=== Numerical techniques and validation ===")
display(numerical_techniques)

def build_oos_inference_report(test_rows):
    """
    Format selected paired OOS tests with raw and within-family Holm p-values for report display and export.
    """
    rows = test_rows.reset_index()
    return pd.DataFrame({
        "Strategy": rows["Strategy"],
        "Benchmark": rows["Benchmark"],
        "Inference Role": rows["Inference Role"],
        "Multiplicity Family": rows["Multiplicity Family"],
        "Annualized Active Return": rows["Annualized Mean Active Return"].map(
            lambda value: format_display_value(value, style="percent")
        ),
        "HAC 95% CI": rows.apply(
            lambda row: "[{}, {}]".format(
                format_display_value(row["HAC CI Lower"], style="percent"),
                format_display_value(row["HAC CI Upper"], style="percent"),
            ), axis=1,
        ),
        "Raw HAC p-value": rows["HAC p-value"].map(
            lambda value: format_display_value(value, style="pvalue")
        ),
        "Within-Family Holm HAC p-value": rows[adjusted_hac_column].map(
            lambda value: format_display_value(value, style="pvalue")
        ),
        "Bootstrap 95% CI": rows.apply(
            lambda row: "[{}, {}]".format(
                format_display_value(row["Bootstrap Active CI Lower"], style="percent"),
                format_display_value(row["Bootstrap Active CI Upper"], style="percent"),
            ), axis=1,
        ),
        "Raw Bootstrap p-value": rows["Bootstrap Mean-Active p-value"].map(
            lambda value: format_display_value(value, style="pvalue")
        ),
        "Within-Family Holm Bootstrap p-value": rows[adjusted_bootstrap_column].map(
            lambda value: format_display_value(value, style="pvalue")
        ),
        "Sharpe Difference": rows["Sharpe Difference"].map(
            lambda value: format_display_value(value, style="ratio")
        ),
        "Sharpe-Difference Bootstrap 95% CI": rows.apply(
            lambda row: "[{}, {}]".format(
                format_display_value(row["Bootstrap Sharpe CI Lower"], style="ratio"),
                format_display_value(row["Bootstrap Sharpe CI Upper"], style="ratio"),
            ), axis=1,
        ),
    })


if len(oos_paired_tests) > 0:
    primary_index = [primary_comparison]
    matched_ablation_index = [
        primary_comparison,
        ("Absolute Only Net", "Views-Off Neutral MVO Net"),
        ("Relative Only Net", "Views-Off Neutral MVO Net"),
    ]
    adverse_stress_index = [
        ("Both Views Net", "Adverse Relative Net"),
        ("Both Views Net", "Adverse Absolute Net"),
        ("Both Views Net", "Joint Adverse Net"),
    ]
    primary_inference_report = build_oos_inference_report(oos_paired_tests.loc[primary_index])
    matched_ablation_inference_report = build_oos_inference_report(
        oos_paired_tests.loc[matched_ablation_index]
    )
    adverse_stress_inference_report = build_oos_inference_report(
        oos_paired_tests.loc[adverse_stress_index]
    )

    sensitivity_report_rows = []
    for (strategy_name, benchmark_name), group in oos_inference_sensitivity.groupby(
        ["Strategy", "Benchmark"], sort=False
    ):
        sensitivity_report_rows.append({
            "Strategy": strategy_name,
            "Benchmark": benchmark_name,
            "Raw HAC p-value range": "[{}, {}]".format(
                format_display_value(group["Raw HAC p-value"].min(), style="pvalue"),
                format_display_value(group["Raw HAC p-value"].max(), style="pvalue"),
            ),
            "Within-Specification Holm HAC p-value range": "[{}, {}]".format(
                format_display_value(group["Within-specification Holm HAC p-value"].min(), style="pvalue"),
                format_display_value(group["Within-specification Holm HAC p-value"].max(), style="pvalue"),
            ),
            "Raw Bootstrap p-value range": "[{}, {}]".format(
                format_display_value(group["Raw Bootstrap p-value"].min(), style="pvalue"),
                format_display_value(group["Raw Bootstrap p-value"].max(), style="pvalue"),
            ),
            "Within-Specification Holm Bootstrap p-value range": "[{}, {}]".format(
                format_display_value(
                    group["Within-specification Holm Bootstrap p-value"].min(), style="pvalue"
                ),
                format_display_value(
                    group["Within-specification Holm Bootstrap p-value"].max(), style="pvalue"
                ),
            ),
            "All bootstrap CIs include zero": bool(
                ((group["Bootstrap CI Lower"] <= 0.0) & (group["Bootstrap CI Upper"] >= 0.0)).all()
            ),
        })
    oos_inference_sensitivity_report = pd.DataFrame(sensitivity_report_rows)
    primary_inference_report.to_csv(
        DATA_OUTPUT_DIR / "oos_primary_inference_report.csv", index=False, encoding=CSV_ENCODING
    )
    matched_ablation_inference_report.to_csv(
        DATA_OUTPUT_DIR / "oos_matched_ablation_inference_report.csv", index=False, encoding=CSV_ENCODING
    )
    adverse_stress_inference_report.to_csv(
        DATA_OUTPUT_DIR / "oos_adverse_stress_inference_report.csv", index=False, encoding=CSV_ENCODING
    )
    oos_inference_sensitivity_report.to_csv(
        DATA_OUTPUT_DIR / "oos_inference_sensitivity_report.csv", index=False, encoding=CSV_ENCODING
    )
    print("=== Primary matched OOS comparison ===")
    display(primary_inference_report)
    print("=== Matched view-ablation Holm family ===")
    display(matched_ablation_inference_report)
    print("=== Exploratory adverse-stress Holm family ===")
    display(adverse_stress_inference_report)
    print("=== Matched-ablation inference sensitivity ===")
    display(oos_inference_sensitivity_report)


def sha256_file(path: Path) -> str:
    """
    Read an unchanged file in chunks and return its SHA-256 hexadecimal digest.
    """
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


artifact_paths = sorted(
    [path for path in RUN_OUTPUT_DIR.rglob("*") if path.is_file() and path != RUN_STATUS_PATH],
    key=lambda path: path.as_posix(),
)
package_names = [
    "numpy", "pandas", "scipy", "statsmodels", "cvxpy", "scikit-learn",
    "matplotlib", "seaborn", "yfinance",
]
package_versions = {}
for package_name in package_names:
    try:
        package_versions[package_name] = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        package_versions[package_name] = None

run_manifest = {
    "run_id": RUN_ID,
    "status": "analysis_complete_not_independently_validated",
    "started_utc": RUN_STARTED_UTC.isoformat(),
    "completed_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "analysis_cutoff": END_DATE_STR,
    "input_mode": "audited_archived_snapshot" if USE_FROZEN_SNAPSHOT else "live_with_archived_fallback",
    "risk_free": data["rf_metadata"],
    "actual_main_return_start": str(data["asset_returns"].index.min().date()),
    "actual_main_return_end": str(data["asset_returns"].index.max().date()),
    "actual_robustness_return_start": str(robustness_data["asset_returns"].index.min().date()),
    "actual_robustness_return_end": str(robustness_data["asset_returns"].index.max().date()),
    "snapshot_manifest_at_load": {
        "path": data["snapshot_manifest_path"],
        "sha256": data["snapshot_manifest_sha256"],
    },
    "oos_validation": {
        "training_observations": int(OOS_TRAIN_WINDOW),
        "daily_observations": int(len(oos_returns_wide)),
        "monthly_rebalances": int(oos_view_calibration["Rebalance Date"].nunique()),
        "implementation_lag_trading_days": 1,
        "solver_audit_rows": int(len(oos_solver_audit)),
        "primary_hypotheses": int(primary_mask.sum()),
        "matched_ablation_hypotheses": int(matched_ablation_mask.sum()),
        "adverse_stress_hypotheses": int(adverse_stress_mask.sum()),
        "performance_context_comparisons": int(performance_context_mask.sum()),
        "cost_grid_levels": int(oos_cost_grid_inference["One-way Cost (bps)"].nunique()),
    },
    "python": platform.python_version(),
    "python_executable": Path(sys.executable).name,
    "python_architecture": platform.architecture()[0],
    "packages": package_versions,
    "artifacts": [
        {
            "path": path.relative_to(PROJECT_ROOT).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in artifact_paths
    ],
}
RUN_STATUS_PATH.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
print("=== Analytical cells complete; inspect local audit outputs before interpreting results ===")
print(f"Run manifest: {RUN_STATUS_PATH}")
print(f"Hashed artifacts: {len(artifact_paths)}")
